# 🛰️ AstroShield — Centro de control

Algoritmo genético que diseña una constelación de **8 satélites** para observar **6 regiones prioritarias**,
respetando **zonas de desechos espaciales** y **posiciones ocupadas**. Ejecución de referencia: fitness
**0.9174** (99.95% de la cota de referencia), 6/6 regiones,
0 violaciones; 8/10 semillas aceptadas.

**Cómo usarlo en Colab:** `Entorno de ejecución → Ejecutar todo`. Si Colab avisa que el notebook no es de Google,
elige **Ejecutar de todos modos** (no usa tu Drive ni pide permisos).

1. **Instalación** (celdas ocultas): crea el paquete `astroshield`, las pruebas y los datos de referencia.
2. **Configuración:** elige el modo de ejecución.
3. **Centro de control:** aparece **al instante** con la ejecución de referencia. Globo 3D o mapa 2D (interruptor
   arriba a la derecha), línea de tiempo de la evolución y una pestaña por cada prueba (E1–E10).
4. **Ejecutar:** corre el algoritmo de verdad (≈ 4 min en modo completo) y **actualiza el mismo centro de control**
   con los datos nuevos.
5. **Laboratorio** (opcional): cambia las condiciones y observa el efecto.
6. **Descargar** los resultados y el centro de control como página HTML.

## 1. Instalación del paquete `astroshield`

Cada celda escribe un archivo del repositorio. Están ocultas para no estorbar; ábrelas para revisar el código.

In [1]:
import os
for carpeta in ('astroshield', 'pruebas', 'outputs'):
    os.makedirs(carpeta, exist_ok=True)

In [2]:
%%writefile astroshield/__init__.py
"""AstroShield: diseño evolutivo de una constelación de observación terrestre."""
__version__ = "2.0.0"

Writing astroshield/__init__.py


In [3]:
%%writefile astroshield/geometria.py
"""Funciones de geometría esférica.

Todos los ángulos están en grados, salvo que el nombre diga lo contrario. La
Tierra es una esfera perfecta de radio RADIO_TIERRA_KM (modelo educativo).
"""
from __future__ import annotations

import numpy as np

RADIO_TIERRA_KM = 6371.0


def normalizar_lon(lon):
    """Lleva la(s) longitud(es) al intervalo [-180, 180)."""
    return (np.asarray(lon) + 180.0) % 360.0 - 180.0


def km_a_grados(km):
    """Distancia sobre la superficie (km) -> ángulo central (grados)."""
    return np.degrees(np.asarray(km) / RADIO_TIERRA_KM)


def grados_a_km(grados):
    """Ángulo central (grados) -> distancia sobre la superficie (km)."""
    return np.radians(np.asarray(grados)) * RADIO_TIERRA_KM


def a_unitario(lat, lon):
    """Lat/lon (grados) -> vectores unitarios con forma (..., 3)."""
    lat_r = np.radians(lat)
    lon_r = np.radians(lon)
    cos_lat = np.cos(lat_r)
    return np.stack(
        [cos_lat * np.cos(lon_r), cos_lat * np.sin(lon_r), np.sin(lat_r)], axis=-1
    )


def desde_unitario(v):
    """Vectores unitarios (o cualquiera no nulo) (..., 3) -> (lat, lon) en grados."""
    v = np.asarray(v, dtype=float)
    v = v / np.linalg.norm(v, axis=-1, keepdims=True)
    lat = np.degrees(np.arcsin(np.clip(v[..., 2], -1.0, 1.0)))
    lon = np.degrees(np.arctan2(v[..., 1], v[..., 0]))
    return lat, lon


def distancia_angular(lat1, lon1, lat2, lon2):
    """Ángulo central de gran círculo (grados) con la fórmula de haversine."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = p2 - p1
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return np.degrees(2 * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0))))


def rumbo_inicial(lat1, lon1, lat2, lon2):
    """Rumbo inicial (grados, en sentido horario desde el norte) del punto 1 al punto 2."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dl = np.radians(np.asarray(lon2) - np.asarray(lon1))
    x = np.sin(dl) * np.cos(p2)
    y = np.cos(p1) * np.sin(p2) - np.sin(p1) * np.cos(p2) * np.cos(dl)
    return np.degrees(np.arctan2(x, y)) % 360.0


def destino(lat, lon, rumbo, distancia):
    """Punto al que se llega recorriendo `distancia` (grados) desde (lat, lon) con `rumbo`."""
    p1 = np.radians(lat)
    l1 = np.radians(lon)
    b = np.radians(rumbo)
    d = np.radians(distancia)
    p2 = np.arcsin(np.sin(p1) * np.cos(d) + np.cos(p1) * np.sin(d) * np.cos(b))
    l2 = l1 + np.arctan2(
        np.sin(b) * np.sin(d) * np.cos(p1), np.cos(d) - np.sin(p1) * np.sin(p2)
    )
    return np.degrees(p2), normalizar_lon(np.degrees(l2))


def circulo(lat, lon, radio, n=72):
    """Borde de un casquete esférico como (lats, lons), antihorario y cerrado."""
    rumbos = np.linspace(360.0, 0.0, n + 1)  # rumbo decreciente => antihorario en un mapa
    return destino(lat, lon, rumbos, radio)


def muestras_disco(lat, lon, radio, anillos=3):
    """Puntos de muestra (N, 2) que cubren un disco esférico: centro + anillos concéntricos.

    El anillo k (1..anillos) está a radio*k/anillos y tiene 8*k puntos, así que el
    anillo exterior queda exactamente sobre el borde de la región.
    """
    puntos = [(lat, lon)]
    for k in range(1, anillos + 1):
        n = 8 * k
        desfase = 0.0 if k % 2 else 180.0 / n  # alterna los anillos
        rumbos = desfase + np.arange(n) * 360.0 / n
        la, lo = destino(lat, lon, rumbos, radio * k / anillos)
        puntos.extend(zip(la, lo))
    return np.array(puntos, dtype=float)


def puntos_aleatorios(rng, n):
    """`n` puntos distribuidos uniformemente sobre la esfera -> arreglos (lat, lon)."""
    lat = np.degrees(np.arcsin(rng.uniform(-1.0, 1.0, n)))
    lon = rng.uniform(-180.0, 180.0, n)
    return lat, lon


def paso_gaussiano(rng, lat, lon, sigma):
    """Mueve cada punto un paso gaussiano 2-D (sigma en grados) sobre la esfera."""
    lat = np.asarray(lat, dtype=float)
    dx = rng.normal(0.0, sigma, lat.shape)
    dy = rng.normal(0.0, sigma, lat.shape)
    dist = np.hypot(dx, dy)
    rumbo = np.degrees(np.arctan2(dx, dy))
    return destino(lat, lon, rumbo, dist)

Writing astroshield/geometria.py


In [4]:
%%writefile astroshield/escenario.py
"""Datos del problema: regiones a observar, zonas prohibidas y parámetros del modelo.

Un Escenario es inmutable. Cada "cambio de condiciones" (sección 3.3 del
enunciado) produce un Escenario nuevo con los métodos `con_*`, para poder
revalidar las configuraciones anteriores contra el nuevo.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field, replace
from typing import Tuple

# Cómo se traducen las prioridades cualitativas del enunciado a pesos numéricos.
# Una región Alta cuenta el triple que una Normal; una Media, el doble.
PESOS_PRIORIDAD = {"alta": 3.0, "media": 2.0, "normal": 1.0}


@dataclass(frozen=True)
class Region:
    nombre: str
    lat: float
    lon: float
    prioridad: str  # "alta" | "media" | "normal"
    radio_km: float = 300.0

    @property
    def peso(self) -> float:
        return PESOS_PRIORIDAD[self.prioridad]


@dataclass(frozen=True)
class Zona:
    """Casquete prohibido sobre la esfera. Un satélite cuyo punto subsatelital
    cae dentro del casquete (a cualquier altitud) lo viola."""

    nombre: str
    lat: float
    lon: float
    radio_grados: float
    tipo: str = "desechos"  # "desechos" | "ocupada"


@dataclass(frozen=True)
class Escenario:
    regiones: Tuple[Region, ...]
    zonas: Tuple[Zona, ...] = ()
    nombre: str = "base"
    n_satelites: int = 8
    alt_min_km: float = 500.0
    alt_max_km: float = 2000.0
    semiangulo_sensor_grados: float = 30.0
    separacion_min_grados: float = 2.0
    umbral_cobertura: float = 0.95
    # Pesos del fitness (suman 1): cobertura simple, cobertura doble, resolución.
    w_cobertura: float = 0.70
    w_redundancia: float = 0.15
    w_resolucion: float = 0.15
    anillos_muestra: int = 3
    notas: Tuple[str, ...] = field(default_factory=tuple)

    # --- cambios de condiciones -----------------------------------------
    def renombrado(self, nombre: str, nota: str = "") -> "Escenario":
        notas = self.notas + ((nota,) if nota else ())
        return replace(self, nombre=nombre, notas=notas)

    def con_zona(self, zona: Zona, nombre: str = None) -> "Escenario":
        return replace(
            self,
            zonas=self.zonas + (zona,),
            nombre=nombre or f"{self.nombre}+{zona.nombre}",
            notas=self.notas + (f"nueva zona {zona.nombre} ({zona.radio_grados}°)",),
        )

    def sin_zona(self, nombre_zona: str, nombre: str = None) -> "Escenario":
        return replace(
            self,
            zonas=tuple(z for z in self.zonas if z.nombre != nombre_zona),
            nombre=nombre or f"{self.nombre}-{nombre_zona}",
        )

    def con_prioridad(self, nombre_region: str, prioridad: str, nombre: str = None) -> "Escenario":
        if prioridad not in PESOS_PRIORIDAD:
            raise ValueError(f"prioridad desconocida {prioridad!r}")
        if nombre_region not in {r.nombre for r in self.regiones}:
            raise KeyError(nombre_region)
        regiones = tuple(
            replace(r, prioridad=prioridad) if r.nombre == nombre_region else r for r in self.regiones
        )
        return replace(
            self,
            regiones=regiones,
            nombre=nombre or f"{self.nombre}:{nombre_region}={prioridad}",
            notas=self.notas + (f"prioridad {nombre_region} -> {prioridad}",),
        )

    def con_parametros(self, nombre: str = None, **parametros) -> "Escenario":
        return replace(self, nombre=nombre or self.nombre, **parametros)

    @property
    def zonas_desechos(self):
        return tuple(z for z in self.zonas if z.tipo == "desechos")

    @property
    def zonas_ocupadas(self):
        return tuple(z for z in self.zonas if z.tipo == "ocupada")

    def a_dict(self) -> dict:
        return asdict(self)

    def a_json(self, **kw) -> str:
        return json.dumps(self.a_dict(), ensure_ascii=False, **kw)


REGIONES = (
    Region("Península de Yucatán", 20.5, -89.0, "alta"),
    Region("Amazonía brasileña", -3.5, -62.0, "alta"),
    Region("California", 36.8, -119.4, "media"),
    Region("Sahel", 15.0, 0.0, "media"),
    Region("Japón", 36.0, 138.0, "media"),
    Region("Interior de Australia", -25.0, 133.0, "normal"),
)

ZONAS_DESECHOS = (
    Zona("Atlántico Norte", 45.0, -35.0, 12.0, "desechos"),
    Zona("Pacífico Sur", -30.0, -120.0, 10.0, "desechos"),
    Zona("Polar Norte", 78.0, 30.0, 8.0, "desechos"),
)

ZONAS_OCUPADAS = (
    Zona("Objeto A", 10.0, 45.0, 5.0, "ocupada"),
    Zona("Objeto B", -15.0, 110.0, 5.0, "ocupada"),
)


def escenario_inicial() -> Escenario:
    """Etapa 1: solo regiones, sin restricciones espaciales."""
    return Escenario(regiones=REGIONES, zonas=(), nombre="etapa1-sin-restricciones")


def escenario_base() -> Escenario:
    """Etapa 2: regiones + zonas de desechos + posiciones ocupadas."""
    return Escenario(regiones=REGIONES, zonas=ZONAS_DESECHOS + ZONAS_OCUPADAS, nombre="base")

Writing astroshield/escenario.py


In [5]:
%%writefile astroshield/evaluacion.py
"""Modelo de cobertura, verificación de restricciones y fitness.

Una configuración es un arreglo de reales con forma (n_satelites, 3) cuyas
columnas son (latitud en grados, longitud en grados, altitud en km) del punto
subsatelital de cada satélite. Una población es un arreglo (P, n_satelites, 3).

Modelo de cobertura
-------------------
Cada satélite lleva un sensor que apunta al nadir con semiángulo eta. Su
huella sobre el suelo es un casquete esférico de ángulo central

    lambda(h) = asin((R + h) / R * sin(eta)) - eta

(o el ángulo del horizonte acos(R / (R + h)) si el sensor ve más allá de él).
Cada región es un disco de 300 km discretizado en puntos de muestra; un punto
está visto por un satélite cuando su distancia angular al punto subsatelital
es <= lambda(h). Una región está "cubierta" cuando la fracción de sus puntos
vistos por al menos un satélite alcanza `umbral_cobertura`.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List

import numpy as np

from .escenario import Escenario
from .geometria import RADIO_TIERRA_KM, a_unitario, distancia_angular, grados_a_km, km_a_grados, muestras_disco

TOLERANCIA_FACTIBILIDAD = 1e-9


def semiangulo_huella(alt_km, semiangulo_sensor_grados):
    """Ángulo central (grados) de la huella para la(s) altitud(es) `alt_km`."""
    alt_km = np.asarray(alt_km, dtype=float)
    eta = np.radians(semiangulo_sensor_grados)
    razon = (RADIO_TIERRA_KM + alt_km) / RADIO_TIERRA_KM * np.sin(eta)
    horizonte = np.arccos(RADIO_TIERRA_KM / (RADIO_TIERRA_KM + alt_km))
    lam = np.where(razon < 1.0, np.arcsin(np.minimum(razon, 1.0)) - eta, horizonte)
    return np.degrees(np.minimum(lam, horizonte))


def calidad_resolucion(alt_km, alt_min_km, alt_max_km):
    """1.0 en la altitud mínima permitida, 0.0 en la máxima (lineal)."""
    return 1.0 - (np.asarray(alt_km) - alt_min_km) / (alt_max_km - alt_min_km)


@dataclass
class ResultadoRegion:
    nombre: str
    prioridad: str
    peso: float
    cobertura: float  # fracción de puntos vistos por >= 1 satélite
    redundancia: float  # fracción de puntos vistos por >= 2 satélites
    resolucion: float  # media de la mejor calidad de resolución por punto (0 si no se ve)
    puntaje: float
    cubierta: bool
    satelites: List[int]  # satélites que ven al menos un punto


@dataclass
class Violacion:
    satelite: int
    restriccion: str  # "desechos" | "ocupada" | "separacion"
    objetivo: str
    distancia_grados: float
    limite_grados: float

    @property
    def intrusion_grados(self) -> float:
        return self.limite_grados - self.distancia_grados


@dataclass
class Evaluacion:
    configuracion: np.ndarray
    fitness: float
    factible: bool
    violacion: float  # intrusión total en grados (0 cuando es factible)
    regiones: List[ResultadoRegion]
    violaciones: List[Violacion] = field(default_factory=list)

    @property
    def n_cubiertas(self) -> int:
        return sum(r.cubierta for r in self.regiones)

    @property
    def cobertura_ponderada(self) -> float:
        w = np.array([r.peso for r in self.regiones])
        c = np.array([r.cobertura for r in self.regiones])
        return float((w * c).sum() / w.sum())

    def tabla_regiones(self):
        import pandas as pd

        return pd.DataFrame(
            [
                {
                    "región": r.nombre,
                    "prioridad": r.prioridad,
                    "peso": r.peso,
                    "cobertura": round(r.cobertura, 3),
                    "redundancia": round(r.redundancia, 3),
                    "resolución": round(r.resolucion, 3),
                    "puntaje": round(r.puntaje, 3),
                    "cubierta": "sí" if r.cubierta else "NO",
                    "satélites": ",".join(map(str, r.satelites)) or "-",
                }
                for r in self.regiones
            ]
        )

    def tabla_satelites(self, escenario: Escenario):
        import pandas as pd

        lam = semiangulo_huella(self.configuracion[:, 2], escenario.semiangulo_sensor_grados)
        return pd.DataFrame(
            {
                "sat": np.arange(len(self.configuracion)),
                "lat": self.configuracion[:, 0].round(2),
                "lon": self.configuracion[:, 1].round(2),
                "alt_km": self.configuracion[:, 2].round(0),
                "huella_km": grados_a_km(lam).round(0),
            }
        )


class Evaluador:
    """Precalcula la geometría del escenario y evalúa poblaciones rápidamente."""

    def __init__(self, escenario: Escenario):
        self.escenario = escenario
        puntos, duenio = [], []
        for i, r in enumerate(escenario.regiones):
            s = muestras_disco(r.lat, r.lon, km_a_grados(r.radio_km), escenario.anillos_muestra)
            puntos.append(s)
            duenio.extend([i] * len(s))
        puntos = np.vstack(puntos)
        self.puntos = puntos
        self.region_de_punto = np.array(duenio)
        self.puntos_unitarios = a_unitario(puntos[:, 0], puntos[:, 1])  # (M, 3)
        conteos = np.bincount(self.region_de_punto, minlength=len(escenario.regiones))
        self.inicio_region = np.concatenate([[0], np.cumsum(conteos)[:-1]])
        self.puntos_por_region = conteos
        self.pesos = np.array([r.peso for r in escenario.regiones])
        if escenario.zonas:
            self.zonas_unitarias = a_unitario(
                np.array([z.lat for z in escenario.zonas]), np.array([z.lon for z in escenario.zonas])
            )
            self.radios_zona = np.array([z.radio_grados for z in escenario.zonas])
        else:
            self.zonas_unitarias = np.zeros((0, 3))
            self.radios_zona = np.zeros(0)
        self.n_evaluaciones = 0

    # ------------------------------------------------------------- por lote
    def _media_por_region(self, por_punto):
        """(P, M) -> (P, R): media sobre los puntos de cada región."""
        sumas = np.add.reduceat(por_punto, self.inicio_region, axis=1)
        return sumas / self.puntos_por_region

    def evaluar_poblacion(self, poblacion):
        """Evaluación vectorizada de una población (P, S, 3).

        Devuelve un dict de arreglos: fitness (P,), violacion (P,), factible (P,),
        cobertura/redundancia/resolucion/puntaje (P, R).
        """
        sc = self.escenario
        poblacion = np.asarray(poblacion, dtype=float)
        self.n_evaluaciones += len(poblacion)
        sat_u = a_unitario(poblacion[..., 0], poblacion[..., 1])  # (P, S, 3)
        alt = poblacion[..., 2]
        lam = semiangulo_huella(alt, sc.semiangulo_sensor_grados)  # (P, S)

        cos_ang = np.einsum("psk,mk->psm", sat_u, self.puntos_unitarios)  # (P, S, M)
        dentro = cos_ang >= np.cos(np.radians(lam))[..., None]
        conteo = dentro.sum(axis=1)  # (P, M)
        q = calidad_resolucion(alt, sc.alt_min_km, sc.alt_max_km)  # (P, S)
        mejor_q = np.where(dentro, q[..., None], 0.0).max(axis=1)  # (P, M)

        cob = self._media_por_region((conteo >= 1).astype(float))
        red = self._media_por_region((conteo >= 2).astype(float))
        res = self._media_por_region(mejor_q)
        puntaje = sc.w_cobertura * cob + sc.w_redundancia * red + sc.w_resolucion * res
        fitness = (puntaje * self.pesos).sum(axis=1) / self.pesos.sum()

        violacion = np.zeros(len(poblacion))
        if len(self.radios_zona):
            ang_zona = np.degrees(
                np.arccos(np.clip(np.einsum("psk,zk->psz", sat_u, self.zonas_unitarias), -1, 1))
            )
            violacion += np.clip(self.radios_zona - ang_zona, 0, None).sum(axis=(1, 2))
        if sc.separacion_min_grados > 0 and poblacion.shape[1] > 1:
            ang_sat = np.degrees(
                np.arccos(np.clip(np.einsum("psk,pqk->psq", sat_u, sat_u), -1, 1))
            )
            iu = np.triu_indices(poblacion.shape[1], k=1)
            violacion += np.clip(sc.separacion_min_grados - ang_sat[:, iu[0], iu[1]], 0, None).sum(axis=1)

        return {
            "fitness": fitness,
            "violacion": violacion,
            "factible": violacion <= TOLERANCIA_FACTIBILIDAD,
            "cobertura": cob,
            "redundancia": red,
            "resolucion": res,
            "puntaje": puntaje,
            "dentro": dentro,
        }

    # ------------------------------------------------------------- individual
    def evaluar(self, configuracion) -> Evaluacion:
        """Evaluación detallada de una configuración (S, 3)."""
        configuracion = np.asarray(configuracion, dtype=float)
        b = self.evaluar_poblacion(configuracion[None])
        sc = self.escenario
        dentro = b["dentro"][0]  # (S, M)
        regiones = []
        for i, r in enumerate(sc.regiones):
            sl = slice(self.inicio_region[i], self.inicio_region[i] + self.puntos_por_region[i])
            sats = [int(s) for s in np.flatnonzero(dentro[:, sl].any(axis=1))]
            cob = float(b["cobertura"][0, i])
            regiones.append(
                ResultadoRegion(
                    nombre=r.nombre,
                    prioridad=r.prioridad,
                    peso=r.peso,
                    cobertura=cob,
                    redundancia=float(b["redundancia"][0, i]),
                    resolucion=float(b["resolucion"][0, i]),
                    puntaje=float(b["puntaje"][0, i]),
                    cubierta=cob >= sc.umbral_cobertura - 1e-12,
                    satelites=sats,
                )
            )
        return Evaluacion(
            configuracion=configuracion.copy(),
            fitness=float(b["fitness"][0]),
            factible=bool(b["factible"][0]),
            violacion=float(b["violacion"][0]),
            regiones=regiones,
            violaciones=listar_violaciones(configuracion, sc),
        )


def listar_violaciones(configuracion, escenario: Escenario) -> List[Violacion]:
    """Lista legible de todas las restricciones que incumple una configuración."""
    salida = []
    lat, lon = configuracion[:, 0], configuracion[:, 1]
    for z in escenario.zonas:
        d = distancia_angular(lat, lon, z.lat, z.lon)
        for s in np.flatnonzero(d < z.radio_grados):
            salida.append(Violacion(int(s), z.tipo, z.nombre, float(d[s]), z.radio_grados))
    if escenario.separacion_min_grados > 0:
        n = len(configuracion)
        for i in range(n):
            for j in range(i + 1, n):
                d = float(distancia_angular(lat[i], lon[i], lat[j], lon[j]))
                if d < escenario.separacion_min_grados:
                    salida.append(Violacion(i, "separacion", f"sat {j}", d, escenario.separacion_min_grados))
    return salida


def altitud_minima_vista_completa(radio_km, escenario: Escenario, desfase_grados=0.0):
    """Altitud más baja cuya huella cubre un disco de `radio_km` cuyo centro está
    a `desfase_grados` del punto subsatelital (None si ninguna lo logra)."""
    necesario = km_a_grados(radio_km) + desfase_grados
    alts = np.arange(escenario.alt_min_km, escenario.alt_max_km + 1e-9, 1.0)
    ok = semiangulo_huella(alts, escenario.semiangulo_sensor_grados) >= necesario
    return float(alts[ok][0]) if ok.any() else None


def cota_referencia(escenario: Escenario) -> dict:
    """Valor optimista de referencia para el fitness (una relajación).

    Ignora las zonas prohibidas y supone que cada región tiene su propio
    satélite justo sobre su centro, a la altitud más baja que ve el disco
    completo; los satélites sobrantes (n_satelites - n_regiones) se asignan a
    las regiones de mayor peso como redundancia completa. Es válida para este
    escenario porque ninguna huella (máx. ~11°) alcanza dos regiones a la vez
    (separación mínima ~30°).
    """
    regiones = sorted(escenario.regiones, key=lambda r: -r.peso)
    sobrantes = max(escenario.n_satelites - len(regiones), 0)
    total, detalle = 0.0, {}
    for k, r in enumerate(regiones):
        h = altitud_minima_vista_completa(r.radio_km, escenario)
        if h is None:
            q, cob = 0.0, 0.0
        else:
            q, cob = float(calidad_resolucion(h, escenario.alt_min_km, escenario.alt_max_km)), 1.0
        red = 1.0 if (k < sobrantes and cob > 0) else 0.0
        s = escenario.w_cobertura * cob + escenario.w_redundancia * red + escenario.w_resolucion * q
        detalle[r.nombre] = {"alt_km": h, "puntaje": s}
        total += r.peso * s
    return {"fitness": total / sum(r.peso for r in regiones), "regiones": detalle}


def aceptacion(ev: Evaluacion, escenario: Escenario) -> dict:
    """Criterios del equipo para "suficientemente buena" (ver docs/01_definicion_problema.md)."""
    altas = [r for r in ev.regiones if r.prioridad == "alta"]
    # Solo se exige redundancia en tantas regiones Alta como satélites sobrantes
    # haya (n_satelites - n_regiones); pedir más sería inalcanzable.
    sobrantes = max(escenario.n_satelites - len(escenario.regiones), 0)
    necesarias = min(len(altas), sobrantes)
    criterios = {
        "C1 válida (sin violaciones)": ev.factible,
        "C2 todas las regiones cubiertas": ev.n_cubiertas == len(ev.regiones),
        f"C3 ≥{necesarias} regiones Alta con redundancia ≥ 95%": sum(r.redundancia >= 0.95 for r in altas) >= necesarias,
        "C4 fitness ≥ 90% de la cota de referencia": ev.fitness
        >= 0.90 * cota_referencia(escenario)["fitness"],
    }
    criterios["ACEPTADA"] = all(criterios.values())
    return criterios

Writing astroshield/evaluacion.py


In [6]:
%%writefile astroshield/algoritmo_genetico.py
"""Algoritmo genético con codificación real para el problema de la constelación.

Genoma: arreglo (n_satelites, 3) con (lat, lon, alt_km).

Ciclo por generación:
  1. ordenar la población (manejo de restricciones: reglas de factibilidad de Deb por defecto)
  2. elitismo: copiar sin cambios a los mejores `tam_elite` individuos
  3. selección de padres por torneo
  4. cruce alineado: los satélites del padre B se emparejan con los del padre A
     con el algoritmo húngaro antes de mezclarlos, porque el orden de los
     satélites en el genoma es arbitrario (simetría de permutación)
  5. mutación: paso gaussiano sobre la esfera + ajuste de altitud, ambos con
     sigma que decae exponencialmente; una reubicación aleatoria poco frecuente
     mantiene la exploración
  6. reparación (probabilística): los satélites dentro de una zona prohibida
     se empujan hasta su borde por el gran círculo desde el centro de la zona
"""
from __future__ import annotations

import time
from dataclasses import asdict, dataclass, field
from typing import Callable, List, Optional

import numpy as np
from scipy.optimize import linear_sum_assignment

from .escenario import Escenario
from .evaluacion import Evaluacion, Evaluador
from .geometria import a_unitario, desde_unitario, destino, distancia_angular, paso_gaussiano, puntos_aleatorios, rumbo_inicial

MODOS_RESTRICCION = ("deb", "penalizacion", "ninguno")


@dataclass
class ParametrosAG:
    tam_poblacion: int = 120
    generaciones: int = 400
    tam_torneo: int = 4
    prob_cruce: float = 0.9
    prob_mezcla: float = 0.3  # probabilidad por satélite de cruce aritmético (en vez de intercambio)
    cruce_alineado: bool = True  # emparejar satélites antes del cruce
    prob_mutacion: float = 0.15  # por satélite
    sigma_inicial_grados: float = 12.0
    sigma_final_grados: float = 0.05
    sigma_alt_inicial_km: float = 250.0
    sigma_alt_final_km: float = 2.0
    prob_reubicacion: float = 0.02  # reubicación aleatoria por satélite
    prob_reparacion: float = 0.3  # probabilidad por hijo de aplicar la reparación
    tam_elite: int = 4
    manejo_restricciones: str = "deb"  # "deb" | "penalizacion" | "ninguno"
    factor_penalizacion: float = 0.05  # fitness perdido por grado de intrusión
    paciencia: Optional[int] = None  # parar tras N generaciones sin mejora

    def __post_init__(self):
        if self.manejo_restricciones not in MODOS_RESTRICCION:
            raise ValueError(f"manejo_restricciones debe ser uno de {MODOS_RESTRICCION}")
        if not 0 <= self.tam_elite < self.tam_poblacion:
            raise ValueError("tam_elite debe ser menor que tam_poblacion")

    def a_dict(self):
        return asdict(self)


@dataclass
class ResultadoAG:
    escenario: Escenario
    parametros: ParametrosAG
    semilla: int
    mejor: np.ndarray
    eval_mejor: Evaluacion
    mejor_inicial: np.ndarray
    eval_inicial: Evaluacion
    historial: List[dict]
    poblacion_final: np.ndarray
    evaluaciones: int
    tiempo_s: float
    motivo_parada: str
    # generación -> mejor configuración, guardada en la generación 0 y en cada mejora
    instantaneas: dict = field(default_factory=dict)

    @property
    def encontro_factible(self) -> bool:
        return self.eval_mejor.factible

    def tabla_historial(self):
        import pandas as pd

        return pd.DataFrame(self.historial)


# ---------------------------------------------------------------- operadores
def poblacion_aleatoria(rng, escenario: Escenario, n: int) -> np.ndarray:
    s = escenario.n_satelites
    lat, lon = puntos_aleatorios(rng, n * s)
    alt = rng.uniform(escenario.alt_min_km, escenario.alt_max_km, n * s)
    return np.stack([lat, lon, alt], axis=-1).reshape(n, s, 3)


def valores_rango(lote: dict, modo: str, penalizacion: float) -> np.ndarray:
    """Mayor es mejor. Deb: cualquier factible le gana a cualquier infactible; las
    infactibles se ordenan por su violación total."""
    f, v = lote["fitness"], lote["violacion"]
    if modo == "ninguno":
        return f.copy()
    if modo == "penalizacion":
        return f - penalizacion * v
    return np.where(lote["factible"], f, -1.0 - v)


def alinear(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Reordena los satélites de b para que b[i] sea el más cercano a a[i]."""
    costo = distancia_angular(a[:, None, 0], a[:, None, 1], b[None, :, 0], b[None, :, 1])
    _, columnas = linear_sum_assignment(costo)
    return b[columnas]


def cruzar(rng, a, b, prob_mezcla, alineado=True):
    if alineado:
        b = alinear(a, b)
    s = len(a)
    h1, h2 = a.copy(), b.copy()
    intercambio = rng.random(s) < 0.5
    h1[intercambio], h2[intercambio] = b[intercambio], a[intercambio]
    mezcla = rng.random(s) < prob_mezcla
    if mezcla.any():
        w = rng.random(mezcla.sum())[:, None]
        ua, ub = a_unitario(a[mezcla, 0], a[mezcla, 1]), a_unitario(b[mezcla, 0], b[mezcla, 1])
        for hijo, ww in ((h1, w), (h2, 1 - w)):
            v = ww * ua + (1 - ww) * ub
            ok = np.linalg.norm(v, axis=1) > 1e-6  # omite pares antipodales
            lat, lon = desde_unitario(np.where(ok[:, None], v, ua))
            idx = np.flatnonzero(mezcla)
            hijo[idx, 0], hijo[idx, 1] = lat, lon
            hijo[idx, 2] = ww[:, 0] * a[mezcla, 2] + (1 - ww[:, 0]) * b[mezcla, 2]
    return h1, h2


def reflejar_altitud(alt, minimo, maximo):
    rango = maximo - minimo
    x = np.mod(alt - minimo, 2 * rango)
    return minimo + np.where(x > rango, 2 * rango - x, x)


def mutar(rng, individuo, escenario, prob, sigma_grados, sigma_alt, prob_reubicacion):
    s = len(individuo)
    m = rng.random(s) < prob
    if m.any():
        lat, lon = paso_gaussiano(rng, individuo[m, 0], individuo[m, 1], sigma_grados)
        individuo[m, 0], individuo[m, 1] = lat, lon
        individuo[m, 2] = reflejar_altitud(
            individuo[m, 2] + rng.normal(0, sigma_alt, m.sum()), escenario.alt_min_km, escenario.alt_max_km
        )
    r = rng.random(s) < prob_reubicacion
    if r.any():
        lat, lon = puntos_aleatorios(rng, r.sum())
        individuo[r, 0], individuo[r, 1] = lat, lon
        individuo[r, 2] = rng.uniform(escenario.alt_min_km, escenario.alt_max_km, r.sum())
    return individuo


def reparar(rng, individuo, escenario: Escenario, margen_grados=0.25, max_iter=4):
    """Saca a los satélites de las zonas prohibidas (en el mismo arreglo). Las zonas
    pueden traslaparse, así que el empuje se repite unas veces; lo que quede lo
    resuelve la selección."""
    for _ in range(max_iter):
        movido = False
        for z in escenario.zonas:
            d = distancia_angular(individuo[:, 0], individuo[:, 1], z.lat, z.lon)
            malos = np.flatnonzero(d < z.radio_grados)
            for i in malos:
                if d[i] < 1e-6:
                    rumbo = rng.uniform(0, 360)
                else:
                    rumbo = rumbo_inicial(z.lat, z.lon, individuo[i, 0], individuo[i, 1])
                individuo[i, 0], individuo[i, 1] = destino(z.lat, z.lon, rumbo, z.radio_grados + margen_grados)
                movido = True
        if not movido:
            break
    return individuo


def diversidad(rng, poblacion, n_pares=12):
    """Distancia angular media entre los satélites alineados de pares aleatorios."""
    if len(poblacion) < 2:
        return 0.0
    salida = []
    for _ in range(n_pares):
        i, j = rng.choice(len(poblacion), 2, replace=False)
        b = alinear(poblacion[i], poblacion[j])
        salida.append(distancia_angular(poblacion[i][:, 0], poblacion[i][:, 1], b[:, 0], b[:, 1]).mean())
    return float(np.mean(salida))


# ---------------------------------------------------------------- principal
def ejecutar_ag(
    escenario: Escenario,
    parametros: ParametrosAG = None,
    semilla: int = 0,
    poblacion_inicial: np.ndarray = None,
    al_registrar: Callable[[dict], None] = None,
) -> ResultadoAG:
    """Ejecuta el AG y devuelve la mejor configuración junto con el historial completo."""
    cfg = parametros or ParametrosAG()
    rng = np.random.default_rng(semilla)
    ev = Evaluador(escenario)
    p = cfg.tam_poblacion

    pob = poblacion_aleatoria(rng, escenario, p)
    if poblacion_inicial is not None:  # arranque en caliente (cambio de condiciones)
        semillas = np.asarray(poblacion_inicial, dtype=float)[:p]
        pob[: len(semillas)] = semillas

    t0 = time.perf_counter()
    lote = ev.evaluar_poblacion(pob)
    rango = valores_rango(lote, cfg.manejo_restricciones, cfg.factor_penalizacion)
    i0 = int(np.argmax(rango))
    mejor_inicial = pob[i0].copy()
    mejor, mejor_rango = pob[i0].copy(), rango[i0]
    historial, instantaneas = [], {0: mejor_inicial.copy()}
    sin_mejora, motivo_parada = 0, "generaciones completadas"

    def registrar(gen, sigma, sigma_alt):
        fact = lote["factible"]
        f = lote["fitness"]
        fila = {
            "generacion": gen,
            "mejor_rango": float(rango.max()),
            "fitness_mejor": float(f[fact].max()) if fact.any() else np.nan,
            "fitness_media": float(f.mean()),
            "fitness_desv": float(f.std()),
            "fitness_peor": float(f.min()),
            "fraccion_factible": float(fact.mean()),
            "violacion_media": float(lote["violacion"].mean()),
            "mejor_rango_historico": float(mejor_rango),
            "regiones_cubiertas_mejor": int(
                (lote["cobertura"][int(np.argmax(rango))] >= escenario.umbral_cobertura - 1e-12).sum()
            ),
            "diversidad_grados": diversidad(rng, pob),
            "sigma_grados": float(sigma),
            "sigma_alt_km": float(sigma_alt),
            "evaluaciones": ev.n_evaluaciones,
        }
        historial.append(fila)
        if al_registrar:
            al_registrar(fila)

    registrar(0, cfg.sigma_inicial_grados, cfg.sigma_alt_inicial_km)
    for gen in range(1, cfg.generaciones + 1):
        t = (gen - 1) / max(cfg.generaciones - 1, 1)
        sigma = cfg.sigma_inicial_grados * (cfg.sigma_final_grados / cfg.sigma_inicial_grados) ** t
        sigma_alt = cfg.sigma_alt_inicial_km * (cfg.sigma_alt_final_km / cfg.sigma_alt_inicial_km) ** t

        orden = np.argsort(-rango)
        hijos = [pob[i].copy() for i in orden[: cfg.tam_elite]]
        while len(hijos) < p:
            candidatos = rng.integers(0, p, size=(2, cfg.tam_torneo))
            pa, pb = (pob[c[np.argmax(rango[c])]] for c in candidatos)
            if rng.random() < cfg.prob_cruce:
                h1, h2 = cruzar(rng, pa, pb, cfg.prob_mezcla, cfg.cruce_alineado)
            else:
                h1, h2 = pa.copy(), pb.copy()
            for h in (h1, h2):
                mutar(rng, h, escenario, cfg.prob_mutacion, sigma, sigma_alt, cfg.prob_reubicacion)
                if cfg.manejo_restricciones != "ninguno" and rng.random() < cfg.prob_reparacion:
                    reparar(rng, h, escenario)
                hijos.append(h)
        pob = np.array(hijos[:p])
        lote = ev.evaluar_poblacion(pob)
        rango = valores_rango(lote, cfg.manejo_restricciones, cfg.factor_penalizacion)

        i = int(np.argmax(rango))
        if rango[i] > mejor_rango + 1e-9:
            mejor, mejor_rango, sin_mejora = pob[i].copy(), rango[i], 0
            instantaneas[gen] = mejor.copy()
        else:
            sin_mejora += 1
        registrar(gen, sigma, sigma_alt)
        if cfg.paciencia and sin_mejora >= cfg.paciencia:
            motivo_parada = f"estancamiento ({cfg.paciencia} generaciones sin mejora)"
            break

    instantaneas[historial[-1]["generacion"]] = mejor.copy()
    return ResultadoAG(
        escenario=escenario,
        parametros=cfg,
        semilla=semilla,
        mejor=mejor,
        eval_mejor=ev.evaluar(mejor),
        mejor_inicial=mejor_inicial,
        eval_inicial=ev.evaluar(mejor_inicial),
        historial=historial,
        poblacion_final=pob,
        evaluaciones=ev.n_evaluaciones,
        tiempo_s=time.perf_counter() - t0,
        motivo_parada=motivo_parada,
        instantaneas=instantaneas,
    )

Writing astroshield/algoritmo_genetico.py


In [7]:
%%writefile astroshield/referencias.py
"""Métodos de referencia para demostrar que el AG es mejor que alternativas triviales."""
from __future__ import annotations

import numpy as np

from .algoritmo_genetico import poblacion_aleatoria, valores_rango
from .escenario import Escenario
from .evaluacion import Evaluador
from .geometria import destino


def busqueda_aleatoria(escenario: Escenario, n_evaluaciones: int, semilla: int = 0, lote: int = 2000):
    """La mejor de `n_evaluaciones` configuraciones aleatorias uniformes (orden de Deb).
    Devuelve (mejor_configuracion, curva_del_mejor_fitness_hasta_el_momento)."""
    rng = np.random.default_rng(semilla)
    ev = Evaluador(escenario)
    mejor, mejor_rango, curva = None, -np.inf, []
    hechas = 0
    while hechas < n_evaluaciones:
        n = min(lote, n_evaluaciones - hechas)
        pob = poblacion_aleatoria(rng, escenario, n)
        b = ev.evaluar_poblacion(pob)
        r = valores_rango(b, "deb", 0.0)
        i = int(np.argmax(r))
        if r[i] > mejor_rango:
            mejor, mejor_rango = pob[i].copy(), r[i]
        hechas += n
        curva.append((hechas, float(mejor_rango)))
    return mejor, curva


def colocacion_voraz(escenario: Escenario, paso_alt_km: float = 50.0):
    """Heurística constructiva: agrega satélites uno por uno, eligiendo cada vez el
    candidato factible (alrededor de los centros de las regiones x rejilla de
    altitudes) con la mayor ganancia de fitness. Imita lo que haría un diseñador."""
    ev = Evaluador(escenario)
    alts = np.arange(escenario.alt_min_km, escenario.alt_max_km + 1e-9, paso_alt_km)
    candidatos = []
    for r in escenario.regiones:
        puntos = [(r.lat, r.lon)]
        for dist in (1.0, 2.0, 3.5, 5.0, 7.0):
            rumbos = np.arange(0, 360, 45.0)
            la, lo = destino(r.lat, r.lon, rumbos, dist)
            puntos.extend(zip(la, lo))
        for la, lo in puntos:
            for h in alts:
                candidatos.append((la, lo, h))
    candidatos = np.array(candidatos)

    elegidos = np.zeros((0, 3))
    for _ in range(escenario.n_satelites):
        pob = np.concatenate(
            [np.repeat(elegidos[None], len(candidatos), axis=0), candidatos[:, None, :]], axis=1
        )
        sc_parcial = escenario.con_parametros(n_satelites=pob.shape[1])
        parcial = Evaluador(sc_parcial)
        b = parcial.evaluar_poblacion(pob)
        r = valores_rango(b, "deb", 0.0)
        elegidos = pob[int(np.argmax(r))]
    return elegidos, ev.evaluar(elegidos)

Writing astroshield/referencias.py


In [8]:
%%writefile astroshield/experimentos.py
"""Experimentos reproducibles detrás de los Entregables 3 y 4.

Cada función devuelve datos simples (dicts / DataFrames / ResultadoAG) para que
el notebook los muestre y herramientas/ejecutar_experimentos.py los guarde en JSON.
"""
from __future__ import annotations

import time
from dataclasses import replace

import numpy as np
import pandas as pd
from scipy import stats

from .algoritmo_genetico import ParametrosAG, ResultadoAG, alinear, ejecutar_ag
from .escenario import Escenario, Zona, escenario_base, escenario_inicial
from .evaluacion import Evaluador, aceptacion, cota_referencia
from .geometria import distancia_angular
from .referencias import busqueda_aleatoria, colocacion_voraz


# ------------------------------------------------------------------ auxiliares
def generacion_convergencia(resultado: ResultadoAG, fraccion: float = 0.99):
    """Primera generación cuyo mejor valor histórico alcanza `fraccion` del valor final."""
    curva = np.array([h["mejor_rango_historico"] for h in resultado.historial])
    final = curva[-1]
    if final <= 0:
        return None
    return int(np.argmax(curva >= fraccion * final))


def generacion_cobertura_completa(resultado: ResultadoAG):
    n = len(resultado.escenario.regiones)
    for h in resultado.historial:
        if h["regiones_cubiertas_mejor"] == n and h["fraccion_factible"] > 0 and not np.isnan(h["fitness_mejor"]):
            return h["generacion"]
    return None


def resumir(resultado: ResultadoAG) -> dict:
    ev, sc = resultado.eval_mejor, resultado.escenario
    return {
        "escenario": sc.nombre,
        "semilla": resultado.semilla,
        "fitness": round(ev.fitness, 4),
        "fitness_inicial": round(resultado.eval_inicial.fitness, 4),
        "factible": ev.factible,
        "violacion_grados": round(ev.violacion, 3),
        "regiones_cubiertas": ev.n_cubiertas,
        "cobertura_ponderada": round(ev.cobertura_ponderada, 4),
        "aceptada": aceptacion(ev, sc)["ACEPTADA"],
        "gen_convergencia": generacion_convergencia(resultado),
        "gen_cobertura_completa": generacion_cobertura_completa(resultado),
        "generaciones": resultado.historial[-1]["generacion"],
        "evaluaciones": resultado.evaluaciones,
        "tiempo_s": round(resultado.tiempo_s, 2),
        "cota": round(cota_referencia(sc)["fitness"], 4),
    }


def distancia_soluciones(a, b) -> float:
    """Distancia angular media (grados) entre satélites tras el emparejamiento óptimo."""
    b = alinear(a, b)
    return float(distancia_angular(a[:, 0], a[:, 1], b[:, 0], b[:, 1]).mean())


def margenes_zonas(configuracion, escenario: Escenario) -> pd.DataFrame:
    """Para cada zona: distancia del satélite más cercano y su margen."""
    filas = []
    for z in escenario.zonas:
        d = distancia_angular(configuracion[:, 0], configuracion[:, 1], z.lat, z.lon)
        filas.append(
            {
                "zona": z.nombre,
                "tipo": z.tipo,
                "radio_grados": z.radio_grados,
                "sat_mas_cercano": int(d.argmin()),
                "distancia_grados": round(float(d.min()), 2),
                "margen_grados": round(float(d.min() - z.radio_grados), 2),
                "respeta": bool(d.min() >= z.radio_grados),
            }
        )
    return pd.DataFrame(filas)


# ----------------------------------------------------------------- experimentos
def e1_inicial(cfg: ParametrosAG = None, semilla: int = 0) -> ResultadoAG:
    """Etapa 1: sin restricciones espaciales."""
    return ejecutar_ag(escenario_inicial(), cfg, semilla=semilla)


def e2_base(cfg: ParametrosAG = None, semilla: int = 0) -> ResultadoAG:
    """Ejecución de referencia de la etapa 2 (todas las restricciones), usada por E2, E3, E4, E8 y E9."""
    return ejecutar_ag(escenario_base(), cfg, semilla=semilla)


def e3_e4_auditoria_restricciones(resultados) -> pd.DataFrame:
    """Márgenes de cada mejor solución a cada zona, en muchas ejecuciones."""
    marcos = []
    for r in resultados:
        df = margenes_zonas(r.mejor, r.escenario)
        df.insert(0, "semilla", r.semilla)
        marcos.append(df)
    return pd.concat(marcos, ignore_index=True)


def e4_objeto_junto_a_region(cfg: ParametrosAG = None, semilla: int = 0) -> ResultadoAG:
    """Posición ocupada reubicada junto al Sahel (a 2° de su centro)."""
    sc = escenario_base().sin_zona("Objeto A").con_zona(
        Zona("Objeto A (reubicado)", 15.0, 2.0, 5.0, "ocupada"), nombre="objeto-junto-al-Sahel"
    )
    return ejecutar_ag(sc, cfg, semilla=semilla)


def escenarios_espacio_reducido():
    base = escenario_base()
    r1 = base.con_zona(Zona("Desechos Golfo de México", 23.0, -90.0, 6.0), nombre="R1 +zona sobre Yucatán")
    r2 = r1.con_parametros(nombre="R2 R1 + techo 1200 km", alt_max_km=1200.0)
    r3 = r2.con_zona(Zona("Desechos Pacífico Norte", 40.0, 140.0, 5.0), nombre="R3 R2 + zona sobre Japón")
    return [base.renombrado("R0 base"), r1, r2, r3]


def e5_espacio_reducido(cfg: ParametrosAG = None, semilla: int = 0):
    """Reduce el espacio utilizable paso a paso. También ejecuta R1 *sin* manejo
    de restricciones para mostrar lo que las restricciones evitan."""
    resultados = [ejecutar_ag(sc, cfg, semilla=semilla) for sc in escenarios_espacio_reducido()]
    sin_manejo = ejecutar_ag(
        escenarios_espacio_reducido()[1], replace(cfg or ParametrosAG(), manejo_restricciones="ninguno"),
        semilla=semilla,
    )
    return resultados, sin_manejo


def escenarios_inviables():
    base = escenario_base()
    return {
        "I1 Amazonía inalcanzable": base.con_zona(
            Zona("Bloqueo Amazonía", -3.5, -62.0, 15.0), nombre="I1 Amazonía inalcanzable"
        ),
        "I2 solo 5 satélites": base.con_parametros(nombre="I2 solo 5 satélites", n_satelites=5),
        "I3 separación 80° imposible": base.con_parametros(
            nombre="I3 separación 80° imposible", separacion_min_grados=80.0
        ),
    }


def e6_inviables(cfg: ParametrosAG = None, semilla: int = 0):
    return {k: ejecutar_ag(sc, cfg, semilla=semilla) for k, sc in escenarios_inviables().items()}


def e7_variacion(cfg: ParametrosAG = None, semillas=range(10), progreso: bool = False):
    """Ejecuciones repetidas + métodos de referencia con el mismo presupuesto de evaluaciones."""
    sc = escenario_base()
    ejecuciones = []
    for k, s in enumerate(semillas, 1):
        ejecuciones.append(ejecutar_ag(sc, cfg, semilla=s))
        if progreso:
            print(f"  semilla {s}: fitness {ejecuciones[-1].eval_mejor.fitness:.4f} ({k}/{len(semillas)})", flush=True)
    tabla = pd.DataFrame([resumir(r) for r in ejecuciones])
    presupuesto = ejecuciones[0].evaluaciones
    aleatoria = [busqueda_aleatoria(sc, presupuesto, semilla=1000 + s)[1][-1][1] for s in semillas]
    _, ev_voraz = colocacion_voraz(sc)
    f_ag = tabla["fitness"].to_numpy()
    mw = stats.mannwhitneyu(f_ag, aleatoria, alternative="greater")
    n = len(ejecuciones)
    dist = np.full((n, n), 0.0)
    for i in range(n):
        for j in range(i + 1, n):
            dist[i, j] = dist[j, i] = distancia_soluciones(ejecuciones[i].mejor, ejecuciones[j].mejor)
    iu = np.triu_indices(n, 1)
    resumen = {
        "n_ejecuciones": n,
        "presupuesto_evaluaciones": int(presupuesto),
        "ag_media": float(f_ag.mean()),
        "ag_desv": float(f_ag.std(ddof=1)) if n > 1 else 0.0,
        "ag_min": float(f_ag.min()),
        "ag_max": float(f_ag.max()),
        "ag_cv_porcentaje": float(100 * f_ag.std(ddof=1) / f_ag.mean()) if n > 1 else 0.0,
        "ag_aceptadas": int(tabla["aceptada"].sum()),
        "aleatoria_media": float(np.mean(aleatoria)),
        "aleatoria_max": float(np.max(aleatoria)),
        "voraz": float(ev_voraz.fitness),
        "cota": float(cota_referencia(sc)["fitness"]),
        "p_mannwhitney_ag_mayor_aleatoria": float(mw.pvalue),
        "distancia_media_soluciones_grados": float(dist[iu].mean()) if n > 1 else 0.0,
        "distancia_max_soluciones_grados": float(dist[iu].max()) if n > 1 else 0.0,
    }
    return ejecuciones, tabla, resumen, aleatoria, dist


def analisis_fallas(configuracion, escenario: Escenario) -> pd.DataFrame:
    """N-1: lo que se pierde si falla cada satélite."""
    ev_completa = Evaluador(escenario).evaluar(configuracion)
    sc_menos = escenario.con_parametros(n_satelites=len(configuracion) - 1)
    ev = Evaluador(sc_menos)
    filas = []
    for s in range(len(configuracion)):
        e = ev.evaluar(np.delete(configuracion, s, axis=0))
        perdidas = [r.nombre for r, f in zip(e.regiones, ev_completa.regiones) if f.cubierta and not r.cubierta]
        filas.append(
            {
                "satelite_fallado": s,
                "fitness": round(e.fitness, 4),
                "cobertura_ponderada": round(e.cobertura_ponderada, 4),
                "regiones_cubiertas": e.n_cubiertas,
                "regiones_perdidas": ", ".join(perdidas) or "ninguna",
            }
        )
    return pd.DataFrame(filas)


def e8_robustez(base: ResultadoAG, cfg: ParametrosAG = None, semilla: int = 0):
    """Prueba propia: valor del término de redundancia ante la falla de un satélite."""
    sc_sin_red = escenario_base().con_parametros(
        nombre="sin término de redundancia", w_cobertura=0.80, w_redundancia=0.0, w_resolucion=0.20
    )
    sin_red = ejecutar_ag(sc_sin_red, cfg, semilla=semilla)
    sc = escenario_base()
    return {
        "con_redundancia": analisis_fallas(base.mejor, sc),
        "sin_redundancia": analisis_fallas(sin_red.mejor, sc),
        "ejecucion_sin_redundancia": sin_red,
    }


def revalidar(configuracion, anterior: Escenario, nuevo: Escenario) -> dict:
    """¿Una configuración anterior sigue siendo válida y buena bajo las nuevas condiciones?"""
    a, b = Evaluador(anterior).evaluar(configuracion), Evaluador(nuevo).evaluar(configuracion)
    return {
        "valida_antes": a.factible,
        "valida_ahora": b.factible,
        "fitness_antes": round(a.fitness, 4),
        "fitness_ahora": round(b.fitness, 4),
        "regiones_cubiertas_ahora": b.n_cubiertas,
        "violaciones": [
            f"sat {v.satelite} en {v.objetivo} ({v.distancia_grados:.2f}° < {v.limite_grados}°)" for v in b.violaciones
        ],
        "aceptada_ahora": aceptacion(b, nuevo)["ACEPTADA"],
    }


def e9_cambio_condiciones(base: ResultadoAG, cfg: ParametrosAG = None, semilla: int = 0):
    """Sección 3.3: cambio de prioridad y una zona que de pronto deja de estar disponible."""
    cfg = cfg or ParametrosAG()
    anterior = base.escenario
    salida = {}
    # (a) El Interior de Australia pasa a prioridad Alta.
    sc_a = anterior.con_prioridad("Interior de Australia", "alta", nombre="C1 Australia -> Alta")
    salida["prioridad"] = {
        "escenario": sc_a,
        "revalidacion": revalidar(base.mejor, anterior, sc_a),
        "reoptimizado": ejecutar_ag(sc_a, cfg, semilla=semilla, poblacion_inicial=base.poblacion_final),
    }
    # (b) Zona de desechos nueva justo donde está el satélite que observa Japón.
    japon = next(r for r in base.eval_mejor.regiones if r.nombre == "Japón")
    if japon.satelites:
        s = japon.satelites[0]
    else:  # ejecuciones cortas: si nadie ve Japón, se usa el satélite más cercano
        centro = next(r for r in anterior.regiones if r.nombre == "Japón")
        s = int(np.argmin(distancia_angular(base.mejor[:, 0], base.mejor[:, 1], centro.lat, centro.lon)))
    lat, lon = base.mejor[s, 0], base.mejor[s, 1]
    sc_b = anterior.con_zona(Zona("Desechos nuevos", float(lat), float(lon), 3.0), nombre="C2 nueva zona sobre Japón")
    corto = replace(cfg, generaciones=min(cfg.generaciones, 150))
    salida["zona_nueva"] = {
        "escenario": sc_b,
        "revalidacion": revalidar(base.mejor, anterior, sc_b),
        "desde_poblacion_previa": ejecutar_ag(sc_b, corto, semilla=semilla, poblacion_inicial=base.poblacion_final),
        "desde_cero": ejecutar_ag(sc_b, corto, semilla=semilla),
    }
    return salida


ABLACIONES = {
    "configuración final": {},
    "sin alineación en cruce": {"cruce_alineado": False},
    "sin reparación": {"prob_reparacion": 0.0},
    "sin elitismo": {"tam_elite": 0},
    "mutación alta (0.5)": {"prob_mutacion": 0.5},
    "penalización en vez de Deb": {"manejo_restricciones": "penalizacion"},
    "población 40": {"tam_poblacion": 40},
}


def e10_ablacion(cfg: ParametrosAG = None, semillas=range(3), escenario: Escenario = None, progreso: bool = False):
    """Efecto de cada operador o parámetro sobre la calidad final."""
    cfg = cfg or ParametrosAG()
    sc = escenario or escenarios_espacio_reducido()[1]  # aquí las restricciones sí afectan
    filas = []
    for nombre, cambio in ABLACIONES.items():
        if progreso:
            print(f"  variante: {nombre}", flush=True)
        for s in semillas:
            r = ejecutar_ag(sc, replace(cfg, **cambio), semilla=s)
            filas.append(
                {
                    "variante": nombre,
                    "semilla": s,
                    "fitness": r.eval_mejor.fitness,
                    "valida": r.eval_mejor.factible,
                    "gen_convergencia": generacion_convergencia(r),
                }
            )
    df = pd.DataFrame(filas)
    agregado = (
        df.groupby("variante", sort=False)
        .agg(fitness_media=("fitness", "mean"), fitness_desv=("fitness", "std"),
             validas=("valida", "sum"), gen_convergencia_media=("gen_convergencia", "mean"))
        .round(4)
        .reset_index()
    )
    return df, agregado


# ------------------------------------------------------------ todo junto
PASOS = 9


def ejecutar_todo(cfg: ParametrosAG = None, n_semillas: int = 10, n_ablacion: int = 5, progreso: bool = False) -> dict:
    """Ejecuta E1-E10 en el mismo orden que la ejecución de referencia y devuelve
    todos los resultados. Lo usan el notebook y herramientas/ejecutar_experimentos.py."""
    cfg = cfg or ParametrosAG()
    t0 = time.perf_counter()

    def paso(k, texto):
        if progreso:
            print(f"[{k}/{PASOS}] {texto}  ({time.perf_counter() - t0:.0f} s)", flush=True)

    paso(1, "E1 · etapa 1 sin restricciones")
    e1 = e1_inicial(cfg)
    paso(2, "E2 · ejecución principal con todas las restricciones")
    base = e2_base(cfg)
    paso(3, f"E7 · variación entre {n_semillas} semillas + búsqueda aleatoria + voraz")
    ejecuciones, tabla, resumen, aleatoria, dist = e7_variacion(cfg, semillas=range(n_semillas), progreso=progreso)
    paso(4, "E3/E4 · auditoría de zonas y objeto junto al Sahel")
    auditoria = e3_e4_auditoria_restricciones(ejecuciones)
    junto = e4_objeto_junto_a_region(cfg)
    paso(5, "E5 · espacio reducido")
    niveles, sin_manejo = e5_espacio_reducido(cfg)
    paso(6, "E6 · condiciones difíciles o inviables")
    inviables = e6_inviables(cfg)
    paso(7, "E8 · prueba propia: falla de un satélite")
    robustez = e8_robustez(base, cfg)
    paso(8, "E9 · cambio de condiciones")
    cambios = e9_cambio_condiciones(base, cfg)
    paso(9, f"E10 · ablación de operadores ({n_ablacion} semillas por variante)")
    ablacion, ablacion_agregada = e10_ablacion(cfg, semillas=range(n_ablacion), progreso=progreso)
    tiempo = time.perf_counter() - t0
    if progreso:
        print(f"Listo en {tiempo:.0f} s.", flush=True)
    return {
        "parametros": cfg,
        "e1": e1,
        "base": base,
        "e7": {"ejecuciones": ejecuciones, "tabla": tabla, "resumen": resumen, "aleatoria": aleatoria, "distancias": dist},
        "auditoria": auditoria,
        "e4": junto,
        "e5": {"niveles": niveles, "sin_manejo": sin_manejo},
        "e6": inviables,
        "e8": robustez,
        "e9": cambios,
        "e10": {"detalle": ablacion, "agregado": ablacion_agregada},
        "tiempo_s": tiempo,
    }


def _texto_violaciones(ev, limite=None):
    textos = [
        f"S{v.satelite}: {v.restriccion} {v.objetivo} ({v.distancia_grados:.2f}° < {v.limite_grados}°)"
        for v in ev.violaciones
    ]
    return textos[:limite] if limite else textos


def resumen_resultados(todo: dict) -> dict:
    """Evidencia en formato JSON (outputs/resultados.json) que citan los documentos."""
    e1, base, e7 = todo["e1"], todo["base"], todo["e7"]
    auditoria, junto = todo["auditoria"], todo["e4"]
    robustez, cambios = todo["e8"], todo["e9"]
    ejec = e7["ejecuciones"]
    return {
        "parametros_ag": todo["parametros"].a_dict(),
        "E1": {**resumir(e1), "regiones": e1.eval_mejor.tabla_regiones().to_dict("records"),
               "aceptacion": aceptacion(e1.eval_mejor, e1.escenario)},
        "E2": {**resumir(base), "regiones": base.eval_mejor.tabla_regiones().to_dict("records"),
               "satelites": base.eval_mejor.tabla_satelites(base.escenario).to_dict("records"),
               "aceptacion": aceptacion(base.eval_mejor, base.escenario),
               "fraccion_factible_inicial": base.historial[0]["fraccion_factible"],
               "cota_detalle": cota_referencia(base.escenario)},
        "E7": {"resumen": e7["resumen"], "ejecuciones": e7["tabla"].to_dict("records"), "aleatoria": e7["aleatoria"]},
        "E3_E4": {
            "todas_respetan": bool(auditoria["respeta"].all()),
            "margen_min_por_zona": auditoria.groupby("zona")["margen_grados"].min().to_dict(),
            "fraccion_factible_inicial_media": float(np.mean([r.historial[0]["fraccion_factible"] for r in ejec])),
            "fraccion_factible_final_media": float(np.mean([r.historial[-1]["fraccion_factible"] for r in ejec])),
        },
        "E4_junto_al_Sahel": {**resumir(junto), "margenes": margenes_zonas(junto.mejor, junto.escenario).to_dict("records"),
                              "regiones": junto.eval_mejor.tabla_regiones().to_dict("records")},
        "E5": {
            "niveles": [{**resumir(r), "regiones": r.eval_mejor.tabla_regiones().to_dict("records"),
                         "satelites": r.eval_mejor.tabla_satelites(r.escenario).to_dict("records")}
                        for r in todo["e5"]["niveles"]],
            "R1_sin_manejo": {**resumir(todo["e5"]["sin_manejo"]),
                              "violaciones": _texto_violaciones(todo["e5"]["sin_manejo"].eval_mejor)},
        },
        "E6": {k: {**resumir(r), "regiones": r.eval_mejor.tabla_regiones().to_dict("records"),
                   "violaciones": _texto_violaciones(r.eval_mejor, 12), "n_violaciones": len(r.eval_mejor.violaciones)}
               for k, r in todo["e6"].items()},
        "E8": {
            **{k: robustez[k].to_dict("records") for k in ("con_redundancia", "sin_redundancia")},
            "resumen": {k: {"fitness_media_N-1": float(robustez[k]["fitness"].mean()),
                            "peor_regiones_cubiertas": int(robustez[k]["regiones_cubiertas"].min()),
                            "fallas_sin_perdida": int((robustez[k]["regiones_perdidas"] == "ninguna").sum())}
                        for k in ("con_redundancia", "sin_redundancia")},
        },
        "E9": {
            "prioridad": {"revalidacion": cambios["prioridad"]["revalidacion"],
                          "reoptimizado": resumir(cambios["prioridad"]["reoptimizado"])},
            "zona_nueva": {"revalidacion": cambios["zona_nueva"]["revalidacion"],
                           "desde_poblacion_previa": resumir(cambios["zona_nueva"]["desde_poblacion_previa"]),
                           "desde_cero": resumir(cambios["zona_nueva"]["desde_cero"])},
        },
        "E10": todo["e10"]["agregado"].to_dict("records"),
        "tiempo_s": round(todo["tiempo_s"], 1),
    }

Writing astroshield/experimentos.py


In [9]:
%%writefile astroshield/visualizacion.py
"""Gráficas estáticas de análisis (matplotlib) para los documentos del repositorio.

La vista interactiva (globo 3D, mapa 2D y gráficas) está en centro_control.py;
estas figuras PNG se guardan en outputs/ y se citan desde docs/.
"""
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from .escenario import Escenario
from .evaluacion import Evaluador, semiangulo_huella
from .geometria import circulo, km_a_grados

# Paleta de AstroShield (personalizada, tema oscuro de "centro de control").
PALETA = {
    "fondo": "#0B1026",
    "panel": "#131B3D",
    "rejilla": "#26305C",
    "texto": "#E6ECFF",
    "tenue": "#8D97C9",
    "final": "#22D3EE",  # constelación final
    "inicial": "#A78BFA",  # constelación inicial
    "cubierta": "#34D399",
    "no_cubierta": "#F87171",
    "desechos": "#FB923C",
    "ocupada": "#F472B6",
    "media": "#FACC15",
}
SERIES = ["#22D3EE", "#FACC15", "#34D399", "#F472B6", "#A78BFA", "#FB923C", "#60A5FA", "#F87171"]


def aplicar_tema():
    plt.rcParams.update(
        {
            "figure.facecolor": PALETA["fondo"],
            "axes.facecolor": PALETA["panel"],
            "axes.edgecolor": PALETA["rejilla"],
            "axes.labelcolor": PALETA["texto"],
            "axes.titlecolor": PALETA["texto"],
            "axes.titleweight": "bold",
            "axes.grid": True,
            "grid.color": PALETA["rejilla"],
            "grid.linewidth": 0.6,
            "xtick.color": PALETA["tenue"],
            "ytick.color": PALETA["tenue"],
            "text.color": PALETA["texto"],
            "legend.facecolor": PALETA["panel"],
            "legend.edgecolor": PALETA["rejilla"],
            "axes.prop_cycle": plt.cycler(color=SERIES),
            "figure.dpi": 110,
            "savefig.facecolor": PALETA["fondo"],
            "savefig.bbox": "tight",
            "font.size": 10,
        }
    )


# ---------------------------------------------------------------- gráficas
def graficar_evolucion(resultado, titulo=None):
    """Mejor / media / dispersión del fitness, factibilidad y diversidad por generación."""
    h = resultado.tabla_historial()
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True, height_ratios=[2.2, 1])
    g = h["generacion"]
    ax1.fill_between(g, h["fitness_media"] - h["fitness_desv"], h["fitness_media"] + h["fitness_desv"],
                     color=PALETA["media"], alpha=0.15, label="media ± desv. estándar")
    ax1.plot(g, h["fitness_media"], color=PALETA["media"], lw=1.4, label="fitness media")
    ax1.plot(g, h["fitness_mejor"], color=PALETA["final"], lw=2.2, label="mejor válida de la generación")
    ax1.plot(g, h["fitness_peor"], color=PALETA["tenue"], lw=0.8, ls=":", label="peor")
    from .evaluacion import cota_referencia

    cota = cota_referencia(resultado.escenario)["fitness"]
    ax1.axhline(cota, color=PALETA["cubierta"], ls="--", lw=1, label=f"cota de referencia {cota:.3f}")
    ax1.set_ylabel("fitness")
    ax1.set_ylim(0, 1)
    ax1.legend(loc="lower right", fontsize=8)
    ax1.set_title(titulo or f"Evolución — {resultado.escenario.nombre} (semilla {resultado.semilla})")
    ax2.plot(g, h["fraccion_factible"], color=PALETA["cubierta"], lw=1.6, label="fracción válida")
    ax2.set_ylabel("válidas")
    ax2.set_ylim(-0.02, 1.02)
    ax3 = ax2.twinx()
    ax3.plot(g, h["diversidad_grados"], color=PALETA["ocupada"], lw=1.2, label="diversidad (°)")
    ax3.set_ylabel("diversidad (°)", color=PALETA["ocupada"])
    ax3.grid(False)
    ax2.set_xlabel("generación")
    lineas = ax2.get_lines() + ax3.get_lines()
    ax2.legend(lineas, [l.get_label() for l in lineas], loc="center right", fontsize=8)
    fig.tight_layout()
    return fig


def graficar_ejecuciones(ejecuciones, titulo="Curvas de convergencia de todas las ejecuciones"):
    fig, ax = plt.subplots(figsize=(10, 4.5))
    curvas = []
    for r in ejecuciones:
        y = [max(h["mejor_rango_historico"], 0) for h in r.historial]
        curvas.append(y)
        ax.plot(y, lw=1, alpha=0.55)
    m = np.mean(curvas, axis=0)
    ax.plot(m, color="white", lw=2.4, label=f"media de {len(ejecuciones)} ejecuciones")
    ax.set_xlabel("generación")
    ax.set_ylabel("mejor fitness hasta el momento")
    ax.set_ylim(0, 1)
    ax.set_title(titulo)
    ax.legend()
    fig.tight_layout()
    return fig


def graficar_regiones(evaluaciones: dict, titulo="Calidad por región"):
    """Barras agrupadas de cobertura por región para varias evaluaciones etiquetadas."""
    etiquetas = [r.nombre for r in next(iter(evaluaciones.values())).regiones]
    x = np.arange(len(etiquetas))
    w = 0.8 / len(evaluaciones)
    fig, ax = plt.subplots(figsize=(11, 4.2))
    for k, (nombre, ev) in enumerate(evaluaciones.items()):
        cob = [r.cobertura for r in ev.regiones]
        red = [r.redundancia for r in ev.regiones]
        barras = ax.bar(x + k * w, cob, w * 0.95, label=f"{nombre} · cobertura", color=SERIES[k % len(SERIES)])
        ax.bar(x + k * w, red, w * 0.45, color="white", alpha=0.55,
               label="redundancia (≥2 satélites)" if k == 0 else None)
        for b, v in zip(barras, cob):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.0%}", ha="center", fontsize=7)
    ax.axhline(0.95, color=PALETA["cubierta"], ls="--", lw=1)
    ax.set_xticks(x + w * (len(evaluaciones) - 1) / 2)
    ax.set_xticklabels([f"{l}\n({r.prioridad})" for l, r in zip(etiquetas, next(iter(evaluaciones.values())).regiones)],
                       fontsize=8)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("fracción de la región")
    ax.set_title(titulo)
    ax.legend(fontsize=8, loc="upper right", ncol=2)
    fig.tight_layout()
    return fig


def graficar_mapa_mision(escenario: Escenario, configuracion, inicial=None, titulo=None):
    """Mapa 2-D equirectangular (vista de análisis, complementa al globo 3D)."""
    ev = Evaluador(escenario).evaluar(configuracion)
    fig, ax = plt.subplots(figsize=(12, 5.6))
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_aspect("equal")

    def dibujar_casquete(lat, lon, radio, **kw):
        la, lo = circulo(lat, lon, radio, 120)
        saltos = np.flatnonzero(np.abs(np.diff(lo)) > 180)
        for seg_la, seg_lo in zip(np.split(la, saltos + 1), np.split(lo, saltos + 1)):
            ax.plot(seg_lo, seg_la, **kw)

    for z in escenario.zonas:
        c = PALETA["desechos"] if z.tipo == "desechos" else PALETA["ocupada"]
        dibujar_casquete(z.lat, z.lon, z.radio_grados, color=c, lw=1.6, ls="--" if z.tipo == "desechos" else "-")
        ax.text(z.lon, z.lat, z.nombre, color=c, fontsize=7, ha="center", va="center")
    lam = semiangulo_huella(configuracion[:, 2], escenario.semiangulo_sensor_grados)
    for s, (la, lo, h) in enumerate(configuracion):
        dibujar_casquete(la, lo, lam[s], color=PALETA["final"], lw=1, alpha=0.8)
    if inicial is not None:
        ax.scatter(inicial[:, 1], inicial[:, 0], marker="x", color=PALETA["inicial"], s=40,
                   label="satélites iniciales (gen 0)")
    ax.scatter(configuracion[:, 1], configuracion[:, 0], marker="*", color=PALETA["final"], s=120, zorder=5,
               edgecolor="white", linewidth=0.5, label="satélites finales")
    for s, (la, lo, h) in enumerate(configuracion):
        ax.text(lo + 2, la + 2, f"S{s} · {h:.0f} km", fontsize=7, color=PALETA["final"])
    for r, rr in zip(escenario.regiones, ev.regiones):
        c = PALETA["cubierta"] if rr.cubierta else PALETA["no_cubierta"]
        dibujar_casquete(r.lat, r.lon, km_a_grados(r.radio_km), color=c, lw=2)
        ax.text(r.lon, r.lat - 4.5, f"{r.nombre}\n{rr.cobertura:.0%}", fontsize=7, color=c, ha="center", va="top")
    ax.set_xlabel("longitud (°)")
    ax.set_ylabel("latitud (°)")
    ax.set_xticks(range(-180, 181, 30))
    ax.set_yticks(range(-90, 91, 30))
    ax.set_title(titulo or f"Mapa de misión — fitness {ev.fitness:.4f}, {ev.n_cubiertas}/{len(ev.regiones)} regiones cubiertas")
    ax.legend(loc="lower left", fontsize=8)
    fig.tight_layout()
    return fig

Writing astroshield/visualizacion.py


In [10]:
%%writefile astroshield/plantilla_centro_control.py
"""Plantilla HTML del centro de control (HTML + CSS + JavaScript en una sola página).

centro_control.py reemplaza __DATOS__ por el JSON de la ejecución y __ALTO__ por
la altura en píxeles. Librerías (versión fija, desde CDN): d3 para el mapa 2D y
las gráficas, topojson + world-atlas para los continentes y globe.gl para el globo 3D.
"""

PLANTILLA = r"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>AstroShield · Centro de control</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600;700&family=Inter:wght@400;600&display=swap" rel="stylesheet">
<style>
:root{
  --fondo:#070B1A; --fondo2:#0B1026; --panel:rgba(16,23,52,.82); --linea:#26305C; --borde:rgba(34,211,238,.28);
  --texto:#E6ECFF; --tenue:#8D97C9; --final:#22D3EE; --inicial:#A78BFA; --ok:#34D399; --mal:#F87171;
  --desechos:#FB923C; --ocupada:#F472B6; --media:#FACC15;
  --mono:'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;
  --sans:'Inter',system-ui,-apple-system,'Segoe UI',sans-serif;
}
*{box-sizing:border-box}
html,body{margin:0;background:var(--fondo);color:var(--texto);font-family:var(--mono);font-size:12px}
button,select,input{font:inherit;color:inherit}
#app{position:relative;height:__ALTO__px;display:grid;grid-template-columns:minmax(0,1fr);grid-template-rows:58px minmax(0,1fr) 50px 36px 392px;gap:6px;padding:8px;
  background:radial-gradient(1200px 500px at 50% 0%,#101a44 0%,var(--fondo) 60%)}
.panel{position:relative;background:var(--panel);border:1px solid var(--borde);border-radius:6px}
.panel::before,.panel::after{content:"";position:absolute;width:10px;height:10px;border-color:var(--final);border-style:solid;opacity:.8}
.panel::before{top:-1px;left:-1px;border-width:2px 0 0 2px;border-top-left-radius:6px}
.panel::after{bottom:-1px;right:-1px;border-width:0 2px 2px 0;border-bottom-right-radius:6px}
h2{margin:0 0 6px;font-size:10px;letter-spacing:.18em;color:var(--final);font-weight:700}
h3{margin:10px 0 6px;font-size:11px;letter-spacing:.12em;color:var(--texto);font-weight:700}
.tenue{color:var(--tenue)} .ok{color:var(--ok)} .mal{color:var(--mal)} .cian{color:var(--final)}
/* ---------- cabecera ---------- */
#cab{display:flex;align-items:center;gap:10px;padding:6px 12px;min-width:0}
.marca{display:flex;flex-direction:column;line-height:1.15;min-width:150px}
.marca b{font-size:17px;letter-spacing:.08em}.marca b span{color:var(--final)}
.marca small{font-size:9px;letter-spacing:.3em;color:var(--tenue)}
.sel{display:flex;flex-direction:column;gap:2px;font-size:9px;letter-spacing:.2em;color:var(--tenue)}
.sel select{background:#0d1534;border:1px solid var(--linea);border-radius:4px;padding:4px 6px;font-size:11px;letter-spacing:0;color:var(--texto);max-width:250px}
#kpis{display:flex;gap:6px;flex:1;justify-content:center;min-width:0}
.kpi{min-width:84px;padding:4px 8px;border:1px solid var(--linea);border-radius:4px;background:rgba(255,255,255,.03)}
.kpi small{display:block;font-size:8.5px;letter-spacing:.16em;color:var(--tenue)}
.kpi b{font-size:15px;font-weight:700}
.interruptor{display:flex;align-items:center;gap:6px;font-weight:700;font-size:11px;color:var(--tenue)}
.interruptor .on{color:var(--final)}
#modo3d{position:relative;width:46px;height:24px;border-radius:12px;border:1px solid var(--final);background:rgba(34,211,238,.18);cursor:pointer;padding:0}
#modo3d i{position:absolute;top:3px;left:3px;width:16px;height:16px;border-radius:50%;background:var(--final);transition:left .2s}
#modo3d[aria-checked="true"] i{left:25px}
#modo3d:disabled{opacity:.4;cursor:not-allowed}
.btn{background:#0d1534;border:1px solid var(--linea);border-radius:4px;padding:5px 9px;cursor:pointer;font-size:11px}
.btn:hover{border-color:var(--final)} .btn.on{background:var(--final);color:#04121a;border-color:var(--final);font-weight:700}
/* ---------- misión ---------- */
#mision{display:grid;grid-template-columns:236px minmax(0,1fr) 262px;gap:6px;min-height:0}
#izq,#der{padding:10px;overflow:auto;min-height:0}
#centro{position:relative;overflow:hidden;min-height:0}
#globo,#mapa{position:absolute;inset:0}
#mapa{width:100%;height:100%;display:none;cursor:grab}
#aviso{position:absolute;bottom:60px;left:50%;transform:translateX(-50%);background:rgba(248,113,113,.15);border:1px solid var(--mal);color:#fecaca;padding:4px 10px;border-radius:4px;font-size:11px;display:none;z-index:3;max-width:90%}
#nota-vista{position:absolute;top:8px;left:10px;max-width:60%;font-size:10.5px;color:var(--tenue);z-index:2;pointer-events:none;background:rgba(7,11,26,.72);padding:4px 8px;border-radius:4px}
#nota-vista b{color:var(--texto);font-size:12px;display:block}
#leyenda{position:absolute;bottom:8px;left:10px;font-size:10px;color:var(--tenue);z-index:2;background:rgba(7,11,26,.7);padding:5px 8px;border-radius:4px;line-height:1.6}
.punto{display:inline-block;width:8px;height:8px;border-radius:50%;margin:0 4px 0 8px;vertical-align:middle}
.punto:first-child{margin-left:0}
.region{padding:6px 6px 7px;border:1px solid transparent;border-radius:4px;cursor:pointer;margin-bottom:3px}
.region:hover{border-color:var(--linea)} .region.sel{border-color:var(--final);background:rgba(34,211,238,.07)}
.region .fila{display:flex;justify-content:space-between;align-items:center;gap:4px}
.region .nom{font-family:var(--sans);font-size:11.5px;font-weight:600;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.prio{font-size:8.5px;letter-spacing:.1em;padding:1px 4px;border-radius:3px;border:1px solid var(--linea);color:var(--tenue)}
.prio.alta{color:var(--media);border-color:rgba(250,204,21,.5)}
.barra{position:relative;height:6px;background:#1a2350;border-radius:3px;margin:5px 0 3px;overflow:hidden}
.barra i{position:absolute;left:0;top:0;bottom:0;border-radius:3px}
.barra i.red{background:rgba(255,255,255,.55);top:2px;bottom:2px}
.region small{font-size:9.5px;color:var(--tenue)}
#detalle-region{margin-top:8px;border-top:1px solid var(--linea);padding-top:8px;font-size:10.5px;line-height:1.6}
.mini{margin-bottom:6px}
.mini-tit{display:flex;justify-content:space-between;font-size:9px;letter-spacing:.14em;color:var(--tenue)}
.mini-tit b{color:var(--texto);letter-spacing:0}
table{border-collapse:collapse;width:100%}
td,th{padding:3px 4px;border-bottom:1px solid var(--linea);text-align:right;white-space:nowrap}
th{font-size:9px;letter-spacing:.12em;color:var(--tenue);font-weight:600}
td:first-child,th:first-child{text-align:left}
#satelites td,#satelites th{font-size:10px;padding:3px 2px}
/* ---------- línea de tiempo ---------- */
#linea{display:flex;align-items:center;gap:8px;padding:0 10px;min-width:0}
#gen{flex:1;accent-color:var(--final);min-width:120px}
#gen-txt{min-width:120px;font-weight:700}
#play{width:34px}
.capas{display:flex;gap:8px;align-items:center;font-size:10.5px;color:var(--tenue)}
.capas label,#linea>label{display:flex;gap:3px;align-items:center;cursor:pointer;white-space:nowrap}
input[type=checkbox]{accent-color:var(--final)}
/* ---------- pestañas ---------- */
#pestanas{display:flex;gap:4px;overflow-x:auto}
#pestanas button{flex:none;background:transparent;border:1px solid var(--linea);border-bottom-color:transparent;border-radius:5px 5px 0 0;padding:7px 12px;cursor:pointer;color:var(--tenue);font-size:11px;letter-spacing:.05em}
#pestanas button[aria-selected="true"]{color:var(--texto);border-color:var(--final);background:rgba(34,211,238,.1);font-weight:700}
#detalle{padding:12px 14px;overflow:auto}
.rejilla{display:grid;gap:10px}
.r2{grid-template-columns:repeat(2,minmax(0,1fr))} .r3{grid-template-columns:repeat(3,minmax(0,1fr))}
.caja{border:1px solid var(--linea);border-radius:5px;padding:8px 10px;background:rgba(255,255,255,.02);min-width:0}
.caja h3{margin-top:0}
.prueba{font-family:var(--sans);font-size:12px;line-height:1.55}
.prueba .etq{font-family:var(--mono);font-size:9px;letter-spacing:.18em;color:var(--final);display:block;margin-top:6px}
.prueba ul{margin:3px 0 0;padding-left:18px} .prueba li{margin:2px 0}
.prueba b{color:#fff}
.ver{background:transparent;border:1px solid var(--final);color:var(--final);border-radius:4px;padding:2px 7px;cursor:pointer;font-size:10px}
.ver:hover{background:var(--final);color:#04121a}
.grande{font-size:22px;font-weight:700}
.chips{display:flex;flex-wrap:wrap;gap:6px}
.chip{border:1px solid var(--linea);border-radius:4px;padding:4px 8px}
.chip small{display:block;font-size:8.5px;letter-spacing:.14em;color:var(--tenue)}
code{font-family:var(--mono);color:var(--final)}
/* ---------- gráficas ---------- */
svg.graf{display:block;overflow:visible}
.graf .eje text{fill:var(--tenue);font-size:9.5px;font-family:var(--mono)}
.graf .eje path,.graf .eje line{stroke:var(--linea)}
.graf .rej line{stroke:rgba(141,151,201,.13)} .graf .rej path{display:none}
.graf .tit-eje{fill:var(--tenue);font-size:9.5px}
.graf .cursor{stroke:#fff;stroke-width:1;stroke-dasharray:3 3;opacity:.7}
.leyenda-g{display:flex;flex-wrap:wrap;gap:4px 12px;font-size:10px;color:var(--tenue);margin-top:4px}
.leyenda-g i{display:inline-block;width:14px;height:3px;margin-right:4px;vertical-align:middle;border-radius:2px}
#tip{position:fixed;pointer-events:none;background:rgba(7,11,26,.95);border:1px solid var(--final);border-radius:4px;padding:5px 8px;font-size:10.5px;display:none;z-index:20;line-height:1.5;max-width:280px}
.mapa-tierra{fill:#16204a;stroke:#2e3b78;stroke-width:.5}
.mapa-esfera{fill:#0a1230}.mapa-ret{fill:none;stroke:rgba(141,151,201,.12);stroke-width:.5}
@media (max-width:1180px){
  #mision{grid-template-columns:208px minmax(0,1fr) 236px}
  .marca{min-width:0} .marca b{font-size:15px} .sel select{max-width:180px}
  .kpi{min-width:0;padding:4px 6px} .kpi b{font-size:13px} .kpi small{letter-spacing:.08em}
  #gen-txt{min-width:0} #gen-txt .tenue,.capas>span{display:none}
  #linea{gap:6px;font-size:11px} .capas{gap:5px}
  #cab .btn{padding:5px 6px}
}
@media (max-width:900px){
  #app{height:auto;grid-template-rows:auto}
  #mision{grid-template-columns:1fr} #centro{height:420px} #izq,#der{max-height:none}
  #kpis{flex-wrap:wrap} #cab{flex-wrap:wrap} #linea{flex-wrap:wrap;padding:8px} #detalle{max-height:none}
  .r2,.r3{grid-template-columns:1fr}
}
</style>
<script src="https://cdnjs.cloudflare.com/ajax/libs/d3/7.8.5/d3.min.js" onerror="(window.__fallas=window.__fallas||[]).push('d3')"></script>
<script src="https://cdn.jsdelivr.net/npm/topojson-client@3.1.0/dist/topojson-client.min.js" onerror="(window.__fallas=window.__fallas||[]).push('topojson')"></script>
<script src="https://unpkg.com/globe.gl@2.33.1/dist/globe.gl.min.js" onerror="(window.__fallas=window.__fallas||[]).push('globe.gl')"></script>
</head>
<body>
<div id="app">
  <header id="cab" class="panel">
    <div class="marca"><b>ASTRO<span>SHIELD</span></b><small>CENTRO DE CONTROL</small></div>
    <label class="sel">CONFIGURACIÓN EN EL MAPA<select id="vista" aria-label="Configuración que se muestra en el mapa"></select></label>
    <div id="kpis"></div>
    <div class="interruptor"><span id="et2d">2D</span>
      <button id="modo3d" role="switch" aria-checked="true" aria-label="Vista 3D activada; pulsa para cambiar a 2D" title="Cambiar entre globo 3D y mapa 2D"><i></i></button>
      <span id="et3d" class="on">3D</span></div>
    <button class="btn" id="descargar" title="Descargar esta página como archivo HTML">⬇ HTML</button>
  </header>
  <main id="mision">
    <section id="izq" class="panel"><h2>REGIONES</h2><div id="regiones"></div><div id="detalle-region"></div></section>
    <section id="centro" class="panel">
      <div id="globo"></div><svg id="mapa" role="img" aria-label="Mapa 2D de la constelación"></svg>
      <div id="nota-vista"></div><div id="aviso" role="status"></div>
      <div id="leyenda">
        <span class="punto" style="background:var(--ok)"></span>cubierta<span class="punto" style="background:var(--mal)"></span>no cubierta
        <span class="punto" style="background:var(--desechos)"></span>desechos<span class="punto" style="background:var(--ocupada)"></span>ocupada<br>
        <span class="punto" style="background:var(--final)"></span>satélite y huella<span class="punto" style="background:var(--inicial)"></span>configuración inicial
        <span class="punto" style="background:var(--mal);border:1px solid #fff"></span>viola una restricción
      </div>
    </section>
    <section id="der" class="panel">
      <h2>TELEMETRÍA</h2>
      <div class="mini"><div class="mini-tit">MEJOR FITNESS / MEDIA <b id="m-fit"></b></div><div id="mini-fitness"></div></div>
      <div class="mini"><div class="mini-tit">FRACCIÓN VÁLIDA <b id="m-fact"></b></div><div id="mini-fact"></div></div>
      <div class="mini"><div class="mini-tit">DIVERSIDAD (°) <b id="m-div"></b></div><div id="mini-div"></div></div>
      <h2 style="margin-top:8px">SATÉLITES</h2><table id="satelites"></table>
    </section>
  </main>
  <div id="linea" class="panel">
    <button class="btn" id="play" title="Reproducir la evolución" aria-label="Reproducir la evolución">▶</button>
    <input type="range" id="gen" min="0" max="0" value="0" aria-label="Generación">
    <span id="gen-txt"></span>
    <button class="btn" id="b-ini">Inicial</button><button class="btn" id="b-fin">Final</button>
    <label title="Superpone la configuración inicial y une cada satélite con su posición final"><input type="checkbox" id="comparar"> Comparar con inicial</label>
    <span class="capas"><span>CAPAS</span>
      <label><input type="checkbox" id="c-huellas" checked>Huellas</label>
      <label><input type="checkbox" id="c-desechos" checked>Desechos</label>
      <label><input type="checkbox" id="c-ocupadas" checked>Ocupadas</label>
      <label><input type="checkbox" id="c-giro" checked>Giro</label></span>
  </div>
  <nav id="pestanas" role="tablist" aria-label="Detalle de las pruebas"></nav>
  <section id="detalle" class="panel" role="tabpanel"></section>
</div>
<div id="tip"></div>
<script id="datos" type="application/json">__DATOS__</script>
<script>
'use strict';
const FUENTE = '<!doctype html>\n' + document.documentElement.outerHTML;
const D = JSON.parse(document.getElementById('datos').textContent);
const P = D.paleta;
const $ = s => document.querySelector(s);
const RT = 6371;
const miles = new Intl.NumberFormat('es-MX');
const f4 = x => x == null ? '—' : Number(x).toFixed(4);
const fx = (x, n) => x == null ? '—' : Number(x).toFixed(n);
const pct = (x, n = 0) => x == null ? '—' : (100 * x).toFixed(n) + '%';
const esc = s => String(s).replace(/[&<>"]/g, c => ({'&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;'}[c]));
const md = s => esc(s).replace(/\*\*(.+?)\*\*/g, '<b>$1</b>');
const sino = b => b ? '<span class="ok">sí</span>' : '<span class="mal">no</span>';
const marca = b => b ? '<span class="ok">✔</span>' : '<span class="mal">✖</span>';
const rgba = (h, a) => { const n = parseInt(h.slice(1), 16); return `rgba(${n >> 16},${(n >> 8) & 255},${n & 255},${a})`; };
const VISTAS = Object.fromEntries(D.vistas.map(v => [v.id, v]));
const FALLAS = window.__fallas || [];
const E = {vista: VISTAS[D.vista_inicial] || D.vistas[0], gen: 0, modo3d: true, region: null, comparar: false,
  capas: {huellas: true, desechos: true, ocupadas: true, giro: true}, reproduciendo: null};

// ---------------------------------------------------------------- geometría (igual que geometria.py)
const rad = x => x * Math.PI / 180, grad = x => x * 180 / Math.PI;
function destino(lat, lon, rumbo, dist) {
  const p1 = rad(lat), l1 = rad(lon), b = rad(rumbo), d = rad(dist);
  const p2 = Math.asin(Math.sin(p1) * Math.cos(d) + Math.cos(p1) * Math.sin(d) * Math.cos(b));
  const l2 = l1 + Math.atan2(Math.sin(b) * Math.sin(d) * Math.cos(p1), Math.cos(d) - Math.sin(p1) * Math.sin(p2));
  return [grad(p2), ((grad(l2) + 180) % 360 + 360) % 360 - 180];
}
function distAng(la1, lo1, la2, lo2) {
  const p1 = rad(la1), p2 = rad(la2), dp = p2 - p1, dl = rad(lo2 - lo1);
  const a = Math.sin(dp / 2) ** 2 + Math.cos(p1) * Math.cos(p2) * Math.sin(dl / 2) ** 2;
  return grad(2 * Math.asin(Math.sqrt(Math.min(Math.max(a, 0), 1))));
}
function muestrasDisco(lat, lon, radio, anillos) {
  const pts = [[lat, lon]];
  for (let k = 1; k <= anillos; k++) {
    const n = 8 * k, desfase = k % 2 ? 0 : 180 / n;
    for (let j = 0; j < n; j++) pts.push(destino(lat, lon, desfase + j * 360 / n, radio * k / anillos));
  }
  return pts;
}
function casquete(lat, lon, r) {
  // globe.gl exige que cada anillo cierre exactamente en su primer punto.
  const c = d3.geoCircle().center([lon, lat]).radius(r).precision(1.5)();
  c.coordinates.forEach(a => { a[a.length - 1] = a[0].slice(); });
  return c;
}

// ---------------------------------------------------------------- estado
function cuadroEn(v, gen) {
  if (v.gen_total == null) return v.cuadros[0];
  let c = v.cuadros[0];
  for (const k of v.cuadros) { if (k.g <= gen) c = k; else break; }
  return c;
}
const escDe = v => D.escenarios[v.escenario];
function criterios(c, sc) {
  const altas = sc.regiones.map((r, i) => [r, i]).filter(([r]) => r.prioridad === 'alta');
  const nec = Math.min(altas.length, Math.max(sc.n_satelites - sc.regiones.length, 0));
  return [
    ['C1', 'Válida (sin violaciones)', !!c.ok],
    ['C2', 'Todas las regiones cubiertas', c.c === sc.regiones.length],
    ['C3', `≥${nec} regiones Alta con redundancia ≥ 95%`, altas.filter(([, i]) => c.r[i][1] >= 0.95).length >= nec],
    ['C4', 'Fitness ≥ 90% de la cota de referencia', c.f >= 0.9 * sc.cota - 1e-9]];
}
const colorRegion = ok => ok ? P.cubierta : P.no_cubierta;

// ---------------------------------------------------------------- tooltip
const tip = $('#tip');
function mostrarTip(ev, html) { tip.innerHTML = html; tip.style.display = 'block';
  const x = Math.min(ev.clientX + 14, window.innerWidth - tip.offsetWidth - 8);
  tip.style.left = x + 'px'; tip.style.top = (ev.clientY + 14) + 'px'; }
const ocultarTip = () => { tip.style.display = 'none'; };

// ---------------------------------------------------------------- gráficas SVG (d3)
function grafica(cont, o) {
  cont.innerHTML = '';
  const W = Math.max(cont.clientWidth, 160), H = o.alto || 150;
  const m = {t: 8, r: o.margenDer || 14, b: o.ejeX ? 30 : 18, l: o.margenIzq || 42};
  const svg = d3.select(cont).append('svg').attr('class', 'graf').attr('width', W).attr('height', H);
  const sx = d3.scaleLinear(o.x, [m.l, W - m.r]), sy = d3.scaleLinear(o.y, [H - m.b, m.t]);
  svg.append('g').attr('class', 'rej').attr('transform', `translate(${m.l},0)`)
    .call(d3.axisLeft(sy).ticks(o.ticksY || 4).tickSize(-(W - m.l - m.r)).tickFormat(''));
  svg.append('g').attr('class', 'eje').attr('transform', `translate(0,${H - m.b})`)
    .call(d3.axisBottom(sx).ticks(Math.max(2, Math.floor(W / 70))).tickSizeOuter(0).tickFormat(o.fx || null));
  svg.append('g').attr('class', 'eje').attr('transform', `translate(${m.l},0)`)
    .call(d3.axisLeft(sy).ticks(o.ticksY || 4).tickSizeOuter(0).tickFormat(o.fy || null));
  if (o.ejeX) svg.append('text').attr('class', 'tit-eje').attr('x', W - m.r).attr('y', H - 2).attr('text-anchor', 'end').text(o.ejeX);
  if (o.ejeY) svg.append('text').attr('class', 'tit-eje').attr('x', 2).attr('y', m.t - 1).attr('dy', '.7em')
    .attr('transform', `rotate(-90 ${8} ${m.t + 40})`).attr('text-anchor', 'end').text(o.ejeY);
  const xsDe = s => s.x || s.y.map((_, i) => i);
  for (const b of o.bandas || []) {
    const xs = b.x || b.y0.map((_, i) => i);
    svg.append('path').attr('fill', b.color).attr('d', d3.area().defined((_, i) => b.y0[i] != null && b.y1[i] != null)
      .x((_, i) => sx(xs[i])).y0((_, i) => sy(b.y0[i])).y1((_, i) => sy(b.y1[i]))(xs));
  }
  for (const h of o.h || []) {
    svg.append('line').attr('x1', m.l).attr('x2', W - m.r).attr('y1', sy(h.y)).attr('y2', sy(h.y))
      .attr('stroke', h.color).attr('stroke-dasharray', '5 4').attr('stroke-width', 1);
    if (h.txt) svg.append('text').attr('x', W - m.r - 2).attr('y', sy(h.y) - 3).attr('text-anchor', 'end')
      .attr('fill', h.color).attr('font-size', 9.5).text(h.txt);
  }
  for (const v of o.v || []) {
    svg.append('line').attr('x1', sx(v.x)).attr('x2', sx(v.x)).attr('y1', m.t).attr('y2', H - m.b)
      .attr('stroke', v.color).attr('stroke-dasharray', '2 3');
    if (v.txt) svg.append('text').attr('x', sx(v.x) + 3).attr('y', m.t + 9).attr('fill', v.color).attr('font-size', 9.5).text(v.txt);
  }
  for (const s of o.series) {
    const xs = xsDe(s);
    svg.append('path').attr('fill', 'none').attr('stroke', s.color).attr('stroke-width', s.ancho || 1.6)
      .attr('stroke-dasharray', s.guion || null).attr('opacity', s.opacidad || 1)
      .attr('d', d3.line().defined((_, i) => s.y[i] != null).x((_, i) => sx(xs[i])).y((_, i) => sy(s.y[i]))(xs));
  }
  const cursor = svg.append('line').attr('class', 'cursor').attr('y1', m.t).attr('y2', H - m.b).style('display', 'none');
  const nombradas = o.series.filter(s => s.nombre);
  if (nombradas.length) {
    svg.append('rect').attr('x', m.l).attr('y', m.t).attr('width', W - m.l - m.r).attr('height', H - m.t - m.b)
      .attr('fill', 'transparent')
      .on('mousemove', ev => {
        const x = sx.invert(d3.pointer(ev)[0]);
        const filas = nombradas.map(s => { const xs = xsDe(s); const i = d3.minIndex(xs, v => Math.abs(v - x));
          return `<span style="color:${s.color}">■</span> ${esc(s.nombre)}: <b>${s.y[i] == null ? '—' : (o.fy ? o.fy(s.y[i]) : fx(s.y[i], 4))}</b>`; });
        mostrarTip(ev, `${esc(o.nombreX || 'x')} <b>${Math.round(x)}</b><br>` + filas.join('<br>'));
      }).on('mouseleave', ocultarTip);
  }
  if (o.leyenda !== false && nombradas.length > 1) {
    const l = document.createElement('div'); l.className = 'leyenda-g';
    l.innerHTML = nombradas.map(s => `<span><i style="background:${s.color}"></i>${esc(s.nombre)}</span>`).join('')
      + (o.h || []).filter(h => h.nombre).map(h => `<span><i style="background:${h.color}"></i>${esc(h.nombre)}</span>`).join('');
    cont.appendChild(l);
  }
  return {cursor(x) { if (x == null) cursor.style('display', 'none'); else cursor.style('display', null).attr('x1', sx(x)).attr('x2', sx(x)); }};
}

function barrasH(cont, filas, o) {
  cont.innerHTML = '';
  const W = Math.max(cont.clientWidth, 200), fh = o.altoFila || 22, m = {t: 4, r: 54, b: 22, l: o.margenIzq || 150};
  const H = m.t + m.b + fh * filas.length;
  const svg = d3.select(cont).append('svg').attr('class', 'graf').attr('width', W).attr('height', H);
  const sx = d3.scaleLinear(o.x, [m.l, W - m.r]).clamp(true);
  svg.append('g').attr('class', 'rej').attr('transform', `translate(0,${H - m.b})`)
    .call(d3.axisBottom(sx).ticks(5).tickSize(-(H - m.t - m.b)).tickFormat(''));
  svg.append('g').attr('class', 'eje').attr('transform', `translate(0,${H - m.b})`).call(d3.axisBottom(sx).ticks(5).tickFormat(o.fx || null));
  filas.forEach((f, i) => {
    const y = m.t + i * fh, x0 = sx(o.x[0] < 0 && f.valor >= 0 ? 0 : o.x[0]);
    svg.append('text').attr('x', m.l - 6).attr('y', y + fh / 2 + 3).attr('text-anchor', 'end').attr('fill', '#E6ECFF')
      .attr('font-size', 10).text(f.nombre);
    const xa = Math.min(x0, sx(f.valor)), xb = Math.max(x0, sx(f.valor));
    svg.append('rect').attr('x', xa).attr('y', y + 4).attr('width', Math.max(xb - xa, 1)).attr('height', fh - 8)
      .attr('fill', f.color || P.final).attr('rx', 2)
      .on('mousemove', ev => mostrarTip(ev, f.tip || `${esc(f.nombre)}: <b>${o.fv ? o.fv(f.valor) : f.valor}</b>`)).on('mouseleave', ocultarTip);
    if (f.error) svg.append('line').attr('x1', sx(f.valor - f.error)).attr('x2', sx(f.valor + f.error))
      .attr('y1', y + fh / 2).attr('y2', y + fh / 2).attr('stroke', '#fff').attr('stroke-width', 1.2);
    svg.append('text').attr('x', W - m.r + 4).attr('y', y + fh / 2 + 3).attr('fill', '#8D97C9').attr('font-size', 9.5)
      .text(o.fv ? o.fv(f.valor) : f.valor);
  });
  for (const v of o.v || []) {
    svg.append('line').attr('x1', sx(v.x)).attr('x2', sx(v.x)).attr('y1', m.t).attr('y2', H - m.b).attr('stroke', v.color).attr('stroke-dasharray', '4 3');
    svg.append('text').attr('x', sx(v.x)).attr('y', H - m.b + 18).attr('text-anchor', 'middle').attr('fill', v.color).attr('font-size', 9).text(v.txt || '');
  }
}

function barrasAgrupadas(cont, grupos, series, o) {
  cont.innerHTML = '';
  const W = Math.max(cont.clientWidth, 200), H = o.alto || 170, m = {t: 8, r: 8, b: 30, l: 42};
  const svg = d3.select(cont).append('svg').attr('class', 'graf').attr('width', W).attr('height', H);
  const x0 = d3.scaleBand(grupos, [m.l, W - m.r]).padding(0.25), x1 = d3.scaleBand(series.map(s => s.nombre), [0, x0.bandwidth()]).padding(0.08);
  const sy = d3.scaleLinear(o.y, [H - m.b, m.t]).clamp(true);
  svg.append('g').attr('class', 'rej').attr('transform', `translate(${m.l},0)`).call(d3.axisLeft(sy).ticks(4).tickSize(-(W - m.l - m.r)).tickFormat(''));
  svg.append('g').attr('class', 'eje').attr('transform', `translate(0,${H - m.b})`).call(d3.axisBottom(x0).tickSizeOuter(0));
  svg.append('g').attr('class', 'eje').attr('transform', `translate(${m.l},0)`).call(d3.axisLeft(sy).ticks(4).tickFormat(o.fy || null));
  if (o.ejeX) svg.append('text').attr('class', 'tit-eje').attr('x', W - m.r).attr('y', H - 2).attr('text-anchor', 'end').text(o.ejeX);
  series.forEach(s => s.valores.forEach((v, i) => {
    svg.append('rect').attr('x', x0(grupos[i]) + x1(s.nombre)).attr('width', x1.bandwidth()).attr('y', sy(v))
      .attr('height', Math.max(sy(o.y[0]) - sy(v), 0)).attr('fill', s.color).attr('rx', 1.5)
      .on('mousemove', ev => mostrarTip(ev, `${esc(grupos[i])} · ${esc(s.nombre)}<br><b>${fx(v, 4)}</b>${s.tips ? '<br>' + esc(s.tips[i]) : ''}`))
      .on('mouseleave', ocultarTip);
  }));
  const l = document.createElement('div'); l.className = 'leyenda-g';
  l.innerHTML = series.map(s => `<span><i style="background:${s.color}"></i>${esc(s.nombre)}</span>`).join('');
  cont.appendChild(l);
}

// ---------------------------------------------------------------- cabecera, regiones, satélites
function llenarSelector() {
  const sel = $('#vista'), grupos = {};
  D.vistas.forEach(v => (grupos[v.grupo] = grupos[v.grupo] || []).push(v));
  sel.innerHTML = Object.entries(grupos).map(([g, vs]) => `<optgroup label="${esc(g)}">` +
    vs.map(v => `<option value="${esc(v.id)}">${esc(v.titulo)}</option>`).join('') + '</optgroup>').join('');
  sel.value = E.vista.id;
  sel.onchange = () => seleccionarVista(sel.value, false);
}

function renderKPIs(c, sc) {
  const cr = criterios(c, sc), cumple = cr.filter(x => x[2]).length;
  const k = (t, v, cls = '', tit = '') => `<div class="kpi" title="${esc(tit)}"><small>${t}</small><b class="${cls}">${v}</b></div>`;
  $('#kpis').innerHTML =
    k('FITNESS', f4(c.f), 'cian', 'Calidad de la configuración, de 0 a 1') +
    k('% DE LA COTA', sc.cota ? fx(100 * c.f / sc.cota, 1) + '%' : '—', '', `Cota de referencia ${f4(sc.cota)}`) +
    k('REGIONES', `${c.c}/${sc.regiones.length}`, c.c === sc.regiones.length ? 'ok' : 'mal', 'Regiones cubiertas (≥ 95% de sus puntos)') +
    k('VIOLACIONES', c.ok ? '0' : `${c.x.length} · ${fx(c.v, 1)}°`, c.ok ? 'ok' : 'mal', c.ok ? 'Configuración válida' : c.x.join('\n')) +
    k('CRITERIOS', `${cumple}/4 ${cumple === 4 ? 'ACEPTADA' : 'NO ACEPTADA'}`, cumple === 4 ? 'ok' : 'mal',
      cr.map(x => `${x[0]} ${x[1]}: ${x[2] ? 'sí' : 'no'}`).join('\n'));
}

function renderRegiones(c, sc) {
  $('#regiones').innerHTML = sc.regiones.map((r, i) => {
    const [cob, red, , cub, sats] = c.r[i], col = colorRegion(cub);
    return `<div class="region${E.region === i ? ' sel' : ''}" data-i="${i}">
      <div class="fila"><span class="nom"><span class="punto" style="background:${col}"></span>${esc(r.nombre)}</span>
      <span class="prio ${r.prioridad}">${r.prioridad.toUpperCase()}</span></div>
      <div class="barra"><i style="width:${100 * cob}%;background:${col}"></i><i class="red" style="width:${100 * red}%"></i></div>
      <small>cobertura <b style="color:${col}">${pct(cob)}</b> · redund. ${pct(red)} · ${sats.length ? sats.map(s => 'S' + s).join(' ') : 'sin satélites'}</small></div>`;
  }).join('');
  document.querySelectorAll('.region').forEach(el => el.onclick = () => {
    const i = +el.dataset.i; E.region = E.region === i ? null : i; render();
    if (E.region != null) enfocar(sc.regiones[i].lat, sc.regiones[i].lon);
  });
  const det = $('#detalle-region');
  if (E.region == null || E.region >= sc.regiones.length) {
    det.innerHTML = '<span class="tenue">Haz clic en una región para ver su detalle y centrarla en el mapa.</span>'; return;
  }
  const r = sc.regiones[E.region], [cob, red, res, cub, sats] = c.r[E.region], w = sc.pesos_fitness;
  det.innerHTML = `<h2>${esc(r.nombre.toUpperCase())}</h2>
    Prioridad <b>${r.prioridad}</b> (peso ${r.peso}) · centro ${fx(r.lat, 1)}°, ${fx(r.lon, 1)}° · radio ${r.radio_km} km<br>
    Cobertura <b>${pct(cob, 1)}</b> · redundancia <b>${pct(red, 1)}</b> · resolución <b>${fx(res, 3)}</b><br>
    Puntaje = ${w[0]}·${fx(cob, 3)} + ${w[1]}·${fx(red, 3)} + ${w[2]}·${fx(res, 3)} = <b>${fx(w[0] * cob + w[1] * red + w[2] * res, 4)}</b><br>
    Estado: ${cub ? '<b class="ok">CUBIERTA</b>' : '<b class="mal">NO CUBIERTA</b>'} · satélites: ${sats.map(s => 'S' + s).join(', ') || 'ninguno'}`;
}

function renderSatelites(c, ini) {
  const filas = c.s.map((s, i) => {
    const mov = ini && ini.s[i] ? distAng(ini.s[i][0], ini.s[i][1], s[0], s[1]) : null;
    return `<tr title="${s[4] ? 'Viola una restricción' : ''}"><td><span class="punto" style="background:${s[4] ? P.no_cubierta : P.final}"></span>S${i}</td>
      <td>${fx(s[0], 2)}</td><td>${fx(s[1], 2)}</td><td>${fx(s[2], 0)}</td>
      <td>${mov == null ? fx(s[3] * Math.PI / 180 * RT, 0) : fx(mov, 1) + '°'}</td></tr>`;
  }).join('');
  $('#satelites').innerHTML = `<tr><th>SAT</th><th>LAT</th><th>LON</th><th title="Altitud en km">ALT</th>${ini
    ? '<th title="Distancia desde la posición inicial">MOVIDO</th>' : '<th title="Radio de la huella del sensor, en km">HUELLA</th>'}</tr>` + filas;
}

let minis = null;
function renderMinis() {
  const v = E.vista, h = v.hist;
  if (!h) {
    ['#mini-fitness', '#mini-fact', '#mini-div'].forEach(s => { $(s).innerHTML = '<div class="tenue" style="padding:10px 0">Configuración fija: sin historial de evolución.</div>'; });
    ['#m-fit', '#m-fact', '#m-div'].forEach(s => { $(s).textContent = ''; }); minis = null; return;
  }
  const n = h.mejor.length - 1, nf = 'generación';
  minis = [
    grafica($('#mini-fitness'), {alto: 92, x: [0, n], y: [0, 1], margenIzq: 30, nombreX: nf, leyenda: false,
      series: [{y: h.media, color: P.media, ancho: 1.1, nombre: 'media'}, {y: h.mejor, color: P.final, ancho: 1.8, nombre: 'mejor histórico'}],
      h: [{y: D.escenarios[v.escenario].cota, color: rgba(P.cubierta, .7)}]}),
    grafica($('#mini-fact'), {alto: 70, x: [0, n], y: [0, 1], margenIzq: 30, nombreX: nf, ticksY: 2, fy: d3.format('.0%'),
      series: [{y: h.fact, color: P.cubierta, nombre: 'fracción válida'}]}),
    grafica($('#mini-div'), {alto: 70, x: [0, n], y: [0, Math.max(5, d3.max(h.div))], margenIzq: 30, nombreX: nf, ticksY: 2,
      series: [{y: h.div, color: P.ocupada, nombre: 'diversidad (°)'}], fy: d3.format('.0f')})];
}
function cursorMinis() {
  const h = E.vista.hist; if (!minis || !h) return;
  const g = Math.min(E.gen, h.mejor.length - 1);
  minis.forEach(m => m.cursor(g));
  $('#m-fit').textContent = f4(h.mejor[g]); $('#m-fact').textContent = pct(h.fact[g]); $('#m-div').textContent = fx(h.div[g], 1) + '°';
}

// ---------------------------------------------------------------- globo 3D
function hayWebGL() {
  try { const c = document.createElement('canvas'); return !!(window.WebGLRenderingContext && (c.getContext('webgl') || c.getContext('experimental-webgl'))); }
  catch (e) { return false; }
}
let G = null, etiquetas = [];
function iniciarGlobo() {
  if (typeof Globe === 'undefined' || !hayWebGL()) return null;
  const el = $('#globo'), cont = $('#centro'), IMG = 'https://unpkg.com/three-globe@2.33.0/example/img/';
  const g = Globe({animateIn: false})(el).width(cont.clientWidth).height(cont.clientHeight)
    .backgroundColor('rgba(0,0,0,0)').globeImageUrl(IMG + 'earth-blue-marble.jpg').bumpImageUrl(IMG + 'earth-topology.png')
    .showAtmosphere(true).atmosphereColor(P.final).atmosphereAltitude(0.16)
    .polygonGeoJsonGeometry('geom').polygonCapCurvatureResolution(2).polygonSideColor(() => 'rgba(0,0,0,0)')
    .polygonStrokeColor('borde').polygonCapColor('relleno').polygonAltitude('alt').polygonLabel('tip').polygonsTransitionDuration(0)
    .pointLat('lat').pointLng('lng').pointAltitude('alt').pointRadius('r').pointColor('color').pointLabel('tip').pointsMerge(false)
    .pointsTransitionDuration(300)
    .arcStartLat('a0').arcStartLng('o0').arcEndLat('a1').arcEndLng('o1').arcColor('color').arcStroke(0.45)
    .arcDashLength(0.5).arcDashGap(0.15).arcDashAnimateTime(2200).arcAltitudeAutoScale(0.35)
    .htmlLat('lat').htmlLng('lng').htmlAltitude(d => d.alt || 0.012)
    .htmlElement(d => { const s = document.createElement('span'); s.textContent = d.texto;
      Object.assign(s.style, {color: d.color, fontFamily: 'JetBrains Mono, monospace', fontSize: Math.round(15 * d.tam) + 'px',
        whiteSpace: 'nowrap', pointerEvents: 'none', textShadow: '0 1px 3px #000', transition: 'opacity .2s'});
      d.el = s; return s; })
    .ringLat('lat').ringLng('lng').ringColor(d => t => rgba(d.color, 1 - t)).ringMaxRadius(4).ringPropagationSpeed(1.2).ringRepeatPeriod(1700);
  g.pointOfView({lat: 15, lng: -60, altitude: 2.4});
  g.controls().autoRotate = true; g.controls().autoRotateSpeed = 0.4;
  g.controls().addEventListener('change', visibilidadEtiquetas);
  return g;
}
function visibilidadEtiquetas() {
  if (!G) return;
  const cam = G.camera().position;
  etiquetas.forEach(d => { if (!d.el) return; const p = G.getCoords(d.lat, d.lng, d.alt || 0.012);
    d.el.style.opacity = (p.x * cam.x + p.y * cam.y + p.z * cam.z) > 0 ? 1 : 0; });
}
function renderGlobo(c, sc, ini) {
  const pol = [];
  sc.zonas.forEach(z => {
    const des = z.tipo === 'desechos'; if (des ? !E.capas.desechos : !E.capas.ocupadas) return;
    const col = des ? P.desechos : P.ocupada;
    pol.push({geom: casquete(z.lat, z.lon, z.radio), relleno: rgba(col, .26), borde: col, alt: .006,
      tip: `<b>${esc(z.nombre)}</b><br>${des ? 'Zona de desechos' : 'Posición ocupada'} · radio ${z.radio}°`});
  });
  sc.regiones.forEach((r, i) => { const col = colorRegion(c.r[i][3]);
    pol.push({geom: casquete(r.lat, r.lon, r.radio_km / RT * 180 / Math.PI), relleno: rgba(col, E.region === i ? .85 : .55), borde: col, alt: .009,
      tip: `<b>${esc(r.nombre)}</b> (${r.prioridad})<br>cobertura ${pct(c.r[i][0])} · redundancia ${pct(c.r[i][1])}`}); });
  if (E.capas.huellas) c.s.forEach((s, i) => pol.push({geom: casquete(s[0], s[1], s[3]), relleno: rgba(s[4] ? P.no_cubierta : P.final, .1),
    borde: rgba(s[4] ? P.no_cubierta : P.final, .9), alt: .003, tip: `Huella de S${i} · radio ${fx(s[3] * Math.PI / 180 * RT, 0)} km`}));
  if (ini && E.capas.huellas) ini.s.forEach((s, i) => pol.push({geom: casquete(s[0], s[1], s[3]), relleno: 'rgba(0,0,0,0)',
    borde: rgba(P.inicial, .85), alt: .002, tip: `Huella inicial de S${i}`}));
  G.polygonsData(pol);
  const pts = c.s.map((s, i) => ({lat: s[0], lng: s[1], alt: s[2] / RT, r: .5, color: s[4] ? P.no_cubierta : P.final,
    tip: `<b>S${i}</b><br>${fx(s[0], 2)}°, ${fx(s[1], 2)}°<br>altitud ${fx(s[2], 0)} km · huella ${fx(s[3] * Math.PI / 180 * RT, 0)} km${s[4] ? '<br><b>¡viola una restricción!</b>' : ''}`}));
  if (ini) ini.s.forEach((s, i) => pts.push({lat: s[0], lng: s[1], alt: s[2] / RT, r: .3, color: P.inicial, tip: `S${i} inicial · ${fx(s[2], 0)} km`}));
  G.pointsData(pts);
  G.arcsData(ini ? c.s.map((s, i) => ({a0: ini.s[i][0], o0: ini.s[i][1], a1: s[0], o1: s[1], color: [P.inicial, P.final]})) : []);
  etiquetas = c.s.map((s, i) => ({lat: s[0], lng: s[1], alt: s[2] / RT + .012, texto: 'S' + i, tam: 1, color: s[4] ? P.no_cubierta : P.final}))
    .concat(sc.regiones.map((r, i) => ({lat: r.lat, lng: r.lon, texto: r.nombre, tam: .78, color: colorRegion(c.r[i][3])})))
    .concat(sc.zonas.filter(z => z.tipo === 'desechos' ? E.capas.desechos : E.capas.ocupadas)
      .map(z => ({lat: z.lat, lng: z.lon, texto: z.nombre, tam: .7, color: z.tipo === 'desechos' ? P.desechos : P.ocupada})));
  G.htmlElementsData(etiquetas);
  requestAnimationFrame(visibilidadEtiquetas);
  G.ringsData(sc.regiones.map((r, i) => ({lat: r.lat, lng: r.lon, color: colorRegion(c.r[i][3])})));
  G.controls().autoRotate = E.capas.giro && E.reproduciendo == null;
}

// ---------------------------------------------------------------- mapa 2D
const MAPA = {listo: false};
function iniciarMapa() {
  const svg = d3.select('#mapa');
  MAPA.svg = svg; MAPA.g = svg.append('g');
  ['esfera', 'ret', 'tierra', 'zonas', 'regiones', 'huellas', 'fantasma', 'arcos', 'sats', 'etiq'].forEach(k => { MAPA[k] = MAPA.g.append('g'); });
  MAPA.zoom = d3.zoom().scaleExtent([1, 10]).on('zoom', ev => { MAPA.g.attr('transform', ev.transform); MAPA.k = ev.transform.k; escalarTextos(); });
  svg.call(MAPA.zoom); MAPA.k = 1;
  ajustarMapa();
  fetch('https://cdn.jsdelivr.net/npm/world-atlas@2.0.2/land-110m.json').then(r => r.json()).then(t => {
    MAPA.tierraGeo = topojson.feature(t, t.objects.land); dibujarFondo();
  }).catch(() => { MAPA.sinTierra = true; });
  MAPA.listo = true;
}
function ajustarMapa() {
  const el = $('#centro'), w = el.clientWidth, h = el.clientHeight;
  MAPA.svg.attr('viewBox', `0 0 ${w} ${h}`);
  MAPA.proy = d3.geoEquirectangular().fitExtent([[8, 62], [w - 8, h - 50]], {type: 'Sphere'});
  MAPA.ruta = d3.geoPath(MAPA.proy);
  dibujarFondo();
}
function dibujarFondo() {
  if (!MAPA.ruta) return;
  MAPA.esfera.selectAll('*').remove(); MAPA.ret.selectAll('*').remove(); MAPA.tierra.selectAll('*').remove();
  MAPA.esfera.append('path').attr('class', 'mapa-esfera').attr('d', MAPA.ruta({type: 'Sphere'}));
  MAPA.ret.append('path').attr('class', 'mapa-ret').attr('d', MAPA.ruta(d3.geoGraticule10()));
  if (MAPA.tierraGeo) MAPA.tierra.append('path').attr('class', 'mapa-tierra').attr('d', MAPA.ruta(MAPA.tierraGeo));
}
function escalarTextos() { const k = MAPA.k || 1;
  MAPA.etiq.selectAll('text').attr('font-size', d => (d.tam || 10) / k).attr('stroke-width', 2.5 / k);
  MAPA.sats.selectAll('circle').attr('r', d => d.r / k); }
function renderMapa(c, sc, ini) {
  if (!MAPA.listo) return;
  const ruta = MAPA.ruta, pr = MAPA.proy;
  const capa = (g, datos, attrs) => { g.selectAll('*').remove();
    datos.forEach(d => { const p = g.append('path').attr('d', ruta(d.geom)).attr('vector-effect', 'non-scaling-stroke');
      Object.entries(attrs(d)).forEach(([k, v]) => p.attr(k, v));
      if (d.tip) p.on('mousemove', ev => mostrarTip(ev, d.tip)).on('mouseleave', ocultarTip); }); };
  capa(MAPA.zonas, sc.zonas.filter(z => z.tipo === 'desechos' ? E.capas.desechos : E.capas.ocupadas).map(z => ({z, geom: casquete(z.lat, z.lon, z.radio),
    tip: `<b>${esc(z.nombre)}</b><br>${z.tipo === 'desechos' ? 'Zona de desechos' : 'Posición ocupada'} · radio ${z.radio}°`})),
    d => { const col = d.z.tipo === 'desechos' ? P.desechos : P.ocupada;
      return {fill: rgba(col, .22), stroke: col, 'stroke-width': 1.3, 'stroke-dasharray': d.z.tipo === 'desechos' ? '5 3' : null}; });
  capa(MAPA.regiones, sc.regiones.map((r, i) => ({i, geom: casquete(r.lat, r.lon, r.radio_km / RT * 180 / Math.PI),
    tip: `<b>${esc(r.nombre)}</b> (${r.prioridad})<br>cobertura ${pct(c.r[i][0])} · redundancia ${pct(c.r[i][1])}`})),
    d => { const col = colorRegion(c.r[d.i][3]); return {fill: rgba(col, E.region === d.i ? .9 : .6), stroke: col, 'stroke-width': 1.5}; });
  capa(MAPA.huellas, E.capas.huellas ? c.s.map((s, i) => ({s, geom: casquete(s[0], s[1], s[3]), tip: `Huella de S${i}`})) : [],
    d => ({fill: rgba(d.s[4] ? P.no_cubierta : P.final, .1), stroke: rgba(d.s[4] ? P.no_cubierta : P.final, .9), 'stroke-width': 1}));
  capa(MAPA.fantasma, ini && E.capas.huellas ? ini.s.map(s => ({geom: casquete(s[0], s[1], s[3])})) : [],
    () => ({fill: 'none', stroke: rgba(P.inicial, .8), 'stroke-width': 1, 'stroke-dasharray': '3 3'}));
  capa(MAPA.arcos, ini ? c.s.map((s, i) => ({geom: {type: 'LineString', coordinates: [[ini.s[i][1], ini.s[i][0]], [s[1], s[0]]]}})) : [],
    () => ({fill: 'none', stroke: rgba(P.inicial, .9), 'stroke-width': 1.2}));
  MAPA.sats.selectAll('*').remove(); MAPA.etiq.selectAll('*').remove();
  const puntos = c.s.map((s, i) => ({s, i, r: 4.5, color: s[4] ? P.no_cubierta : P.final, xy: pr([s[1], s[0]])}));
  if (ini) ini.s.forEach((s, i) => puntos.push({s, i, r: 2.8, color: P.inicial, xy: pr([s[1], s[0]]), ini: true}));
  MAPA.sats.selectAll('circle').data(puntos).join('circle').attr('cx', d => d.xy[0]).attr('cy', d => d.xy[1]).attr('r', d => d.r)
    .attr('fill', d => d.color).attr('stroke', '#04121a').attr('stroke-width', 1).attr('vector-effect', 'non-scaling-stroke')
    .on('mousemove', (ev, d) => mostrarTip(ev, `<b>S${d.i}</b>${d.ini ? ' (inicial)' : ''}<br>${fx(d.s[0], 2)}°, ${fx(d.s[1], 2)}° · ${fx(d.s[2], 0)} km`))
    .on('mouseleave', ocultarTip);
  const textos = c.s.map((s, i) => ({texto: 'S' + i, xy: pr([s[1], s[0]]), dx: 6, dy: -5, color: s[4] ? P.no_cubierta : P.final, tam: 10}))
    .concat(sc.regiones.map((r, i) => ({texto: r.nombre, xy: pr([r.lon, r.lat]), dx: 0, dy: 16, color: colorRegion(c.r[i][3]), tam: 9.5, centro: true})))
    .concat(sc.zonas.filter(z => z.tipo === 'desechos' ? E.capas.desechos : E.capas.ocupadas)
      .map(z => ({texto: z.nombre, xy: pr([z.lon, z.lat]), dx: 0, dy: 3, color: z.tipo === 'desechos' ? P.desechos : P.ocupada, tam: 9, centro: true})));
  MAPA.etiq.selectAll('text').data(textos).join('text').attr('x', d => d.xy[0] + d.dx).attr('y', d => d.xy[1] + d.dy)
    .attr('text-anchor', d => d.centro ? 'middle' : 'start').attr('fill', d => d.color).attr('font-family', 'JetBrains Mono, monospace')
    .attr('paint-order', 'stroke').attr('stroke', '#070B1A').attr('pointer-events', 'none').text(d => d.texto);
  escalarTextos();
}

// ---------------------------------------------------------------- render principal
function render() {
  const v = E.vista, sc = escDe(v), c = cuadroEn(v, E.gen);
  const ini = E.comparar && v.cuadros.length > 1 ? v.cuadros[0] : null;
  renderKPIs(c, sc); renderRegiones(c, sc); renderSatelites(c, ini);
  if (E.modo3d && G) renderGlobo(c, sc, ini); else renderMapa(c, sc, ini);
  $('#nota-vista').innerHTML = `<b>${esc(v.titulo)}</b>${esc(sc.nombre)}${v.semilla != null ? ' · semilla ' + v.semilla : ''}` +
    (v.nota ? '<br>' + md(v.nota) : '') + (c.x.length ? `<br><span class="mal">Violaciones: ${esc(c.x.join(' · '))}</span>` : '');
  const fija = v.gen_total == null;
  $('#gen').disabled = fija; $('#play').disabled = fija; $('#b-ini').disabled = fija; $('#comparar').disabled = fija || v.cuadros.length < 2;
  $('#gen-txt').innerHTML = fija ? 'CONFIGURACIÓN FIJA' : `GEN <span class="cian">${E.gen}</span> / ${v.gen_total} <span class="tenue">· cuadro ${v.cuadros.indexOf(c) + 1}/${v.cuadros.length}</span>`;
  $('#b-ini').classList.toggle('on', !fija && E.gen === 0); $('#b-fin').classList.toggle('on', fija || E.gen === v.gen_total);
  cursorMinis(); if (graficaEvolucion && v.id === 'base') graficaEvolucion.forEach(g => g.cursor(E.gen));
}
let pendiente = false;
const pedirRender = () => { if (!pendiente) { pendiente = true; requestAnimationFrame(() => { pendiente = false; render(); }); } };

function seleccionarVista(id, irAlMapa = true) {
  const v = VISTAS[id]; if (!v) return;
  detener(); E.vista = v; E.gen = v.gen_total == null ? 0 : v.gen_total;
  if (E.region != null && E.region >= escDe(v).regiones.length) E.region = null;
  $('#vista').value = id; const s = $('#gen'); s.max = v.gen_total || 0; s.value = E.gen;
  renderMinis(); render();
  if (irAlMapa) { const m = $('#mision'); m.animate([{boxShadow: '0 0 0 2px #22D3EE'}, {boxShadow: '0 0 0 0 transparent'}], 900);
    m.scrollIntoView({behavior: 'smooth', block: 'start'}); }
}
function enfocar(lat, lon) {
  if (E.modo3d && G) { G.controls().autoRotate = false; $('#c-giro').checked = false; E.capas.giro = false; G.pointOfView({lat, lng: lon, altitude: 1.5}, 900); }
  else if (MAPA.listo) { const [x, y] = MAPA.proy([lon, lat]), el = $('#centro');
    MAPA.svg.transition().duration(700).call(MAPA.zoom.transform, d3.zoomIdentity.translate(el.clientWidth / 2, el.clientHeight / 2).scale(3).translate(-x, -y)); }
}
function detener() { if (E.reproduciendo != null) { clearInterval(E.reproduciendo); E.reproduciendo = null; $('#play').textContent = '▶'; } }
function reproducir() {
  const v = E.vista; if (v.gen_total == null) return;
  if (E.reproduciendo != null) { detener(); render(); return; }
  if (E.gen >= v.gen_total) E.gen = 0;
  const paso = Math.max(1, Math.round(v.gen_total / 240));
  $('#play').textContent = '⏸';
  E.reproduciendo = setInterval(() => { E.gen = Math.min(v.gen_total, E.gen + paso); $('#gen').value = E.gen; pedirRender();
    if (E.gen >= v.gen_total) { detener(); pedirRender(); } }, 45);
}
function fijarModo(tres) {
  E.modo3d = tres && !!G;
  const b = $('#modo3d'); b.setAttribute('aria-checked', String(E.modo3d));
  b.setAttribute('aria-label', E.modo3d ? 'Vista 3D activada; pulsa para cambiar a 2D' : 'Vista 2D activada; pulsa para cambiar a 3D');
  $('#et3d').classList.toggle('on', E.modo3d); $('#et2d').classList.toggle('on', !E.modo3d);
  $('#globo').style.display = E.modo3d ? 'block' : 'none'; $('#mapa').style.display = E.modo3d ? 'none' : 'block';
  if (G) { if (E.modo3d) G.resumeAnimation(); else G.pauseAnimation(); }
  if (!E.modo3d && !MAPA.listo) iniciarMapa();
  render();
}

// ---------------------------------------------------------------- pestañas
const bloque = t => `<div class="prueba"><span class="etq">CONDICIONES</span>${md(t.condiciones)}
  <span class="etq">RESULTADO</span>${md(t.resultado)}
  <span class="etq">INTERPRETACIÓN</span><ul>${t.interpretacion.map(x => `<li>${md(x)}</li>`).join('')}</ul></div>`;
const botonVer = (id, txt = 'Ver en el mapa') => VISTAS[id] ? `<button class="ver" data-vista="${esc(id)}">${esc(txt)}</button>` : '';
let graficaEvolucion = null;

const PESTANAS = [
  ['evolucion', 'Evolución', pEvolucion], ['ejecuciones', 'Ejecuciones', pEjecuciones], ['restricciones', 'Restricciones', pRestricciones],
  ['limite', 'Escenarios límite', pLimite], ['propia', 'Prueba propia', pPropia], ['calibracion', 'Calibración', pCalibracion],
  ['modelo', 'Modelo y verificación', pModelo]];

function pEvolucion(el) {
  const ev = D.evolucion, n = ev.fitness_media.length - 1, sup = ev.fitness_media.map((m, i) => m == null ? null : Math.min(1, m + ev.fitness_desv[i])),
    inf = ev.fitness_media.map((m, i) => m == null ? null : Math.max(0, m - ev.fitness_desv[i]));
  el.innerHTML = `<div class="rejilla" style="grid-template-columns:minmax(0,1.5fr) minmax(0,1fr)">
    <div class="caja"><h3>E2 · CALIDAD POR GENERACIÓN ${botonVer('base', 'Reproducir en el mapa')}</h3><div id="g-ev1"></div></div>
    <div class="caja"><h3>FACTIBILIDAD, DIVERSIDAD Y MUTACIÓN</h3><div id="g-ev2"></div><div id="g-ev3"></div></div></div>
    <div class="caja" style="margin-top:10px">${bloque(ev.texto)}</div>`;
  const v = [];
  if (ev.gen_cobertura_completa != null) v.push({x: ev.gen_cobertura_completa, color: P.cubierta, txt: 'todas cubiertas'});
  if (ev.gen_convergencia != null) v.push({x: ev.gen_convergencia, color: P.media, txt: '99%'});
  graficaEvolucion = [
    grafica($('#g-ev1'), {alto: 205, x: [0, n], y: [0, 1], ejeX: 'generación', nombreX: 'generación', v,
      bandas: [{y0: inf, y1: sup, color: rgba(P.media, .14)}],
      series: [{y: ev.fitness_peor, color: P.tenue, ancho: .8, guion: '2 3', nombre: 'peor'},
        {y: ev.fitness_media, color: P.media, ancho: 1.3, nombre: 'media (± desv.)'},
        {y: ev.fitness_mejor, color: P.final, ancho: 2.2, nombre: 'mejor válida de la generación'}],
      h: [{y: ev.cota, color: P.cubierta, txt: `cota ${f4(ev.cota)}`, nombre: 'cota de referencia'}]}),
    grafica($('#g-ev2'), {alto: 110, x: [0, n], y: [0, Math.max(5, d3.max(ev.div))], nombreX: 'generación',
      series: [{y: ev.div, color: P.ocupada, nombre: 'diversidad (°)'}, {y: ev.fact.map(x => x * Math.max(5, d3.max(ev.div))), color: P.cubierta, nombre: 'fracción válida (escalada)'}]}),
    grafica($('#g-ev3'), {alto: 92, x: [0, n], y: [0, d3.max(ev.sigma)], ejeX: 'generación', nombreX: 'generación', ticksY: 3,
      series: [{y: ev.sigma, color: P.inicial, nombre: 'σ de mutación (°)'}]})];
  if (E.vista.id === 'base') graficaEvolucion.forEach(g => g.cursor(E.gen));
}

function pEjecuciones(el) {
  const ej = D.ejecuciones, r = ej.resumen;
  el.innerHTML = `<div class="rejilla r3">
    <div class="caja"><h3>E7 · CONVERGENCIA DE ${r.n_ejecuciones} SEMILLAS</h3><div id="g-e7a"></div></div>
    <div class="caja"><h3>MISMO PRESUPUESTO: AG CONTRA ALEATORIA</h3><div id="g-e7b"></div></div>
    <div class="caja"><h3>DISTANCIA ENTRE SOLUCIONES (°)</h3><div id="g-e7c"></div></div></div>
    <div class="rejilla r2" style="margin-top:10px"><div class="caja" style="overflow:auto"><h3>TABLA POR SEMILLA</h3><table id="t-e7"></table></div>
    <div class="caja">${bloque(ej.texto)}</div></div>`;
  const curvas = ej.filas.map(f => VISTAS[f.vista].hist.mejor), n = curvas[0].length - 1;
  const media = curvas[0].map((_, i) => d3.mean(curvas, c => c[i]));
  grafica($('#g-e7a'), {alto: 170, x: [0, n], y: [0, 1], ejeX: 'generación', nombreX: 'generación', leyenda: false,
    series: curvas.map((c, k) => ({y: c, color: d3.schemeTableau10[k % 10], ancho: 1, opacidad: .55})).concat([{y: media, color: '#fff', ancho: 2.2, nombre: 'media'}]),
    h: [{y: r.cota, color: P.cubierta, txt: 'cota'}]});
  const c2 = $('#g-e7b'); c2.innerHTML = '';
  const W = Math.max(c2.clientWidth, 200), H = 170, m = {t: 8, r: 70, b: 22, l: 42};
  const todos = ej.filas.map(f => f.fitness).concat(ej.aleatoria);
  const sy = d3.scaleLinear([Math.max(0, d3.min(todos) - .05), Math.min(1, Math.max(r.cota, d3.max(todos)) + .02)], [H - m.b, m.t]);
  const sx = d3.scaleBand(['AG', 'Aleatoria'], [m.l, W - m.r]).padding(.4);
  const svg = d3.select(c2).append('svg').attr('class', 'graf').attr('width', W).attr('height', H);
  svg.append('g').attr('class', 'rej').attr('transform', `translate(${m.l},0)`).call(d3.axisLeft(sy).ticks(5).tickSize(-(W - m.l - m.r)).tickFormat(''));
  svg.append('g').attr('class', 'eje').attr('transform', `translate(${m.l},0)`).call(d3.axisLeft(sy).ticks(5));
  svg.append('g').attr('class', 'eje').attr('transform', `translate(0,${H - m.b})`).call(d3.axisBottom(sx).tickSizeOuter(0));
  [[r.cota, P.cubierta, `cota ${f4(r.cota)}`, -4], [r.voraz, P.media, `voraz ${f4(r.voraz)}`, 11]].forEach(([y, col, t, dy]) => {
    svg.append('line').attr('x1', m.l).attr('x2', W - m.r).attr('y1', sy(y)).attr('y2', sy(y)).attr('stroke', col).attr('stroke-dasharray', '5 4');
    svg.append('text').attr('x', W - m.r + 4).attr('y', sy(y) + dy).attr('fill', col).attr('font-size', 9.5).text(t); });
  const pts = ej.filas.map((f, i) => ({g: 'AG', y: f.fitness, t: `semilla ${f.semilla}: ${f4(f.fitness)}`, vista: f.vista, j: i}))
    .concat(ej.aleatoria.map((y, i) => ({g: 'Aleatoria', y, t: `aleatoria ${i}: ${f4(y)}`, j: i})));
  svg.selectAll('circle').data(pts).join('circle').attr('cx', d => sx(d.g) + sx.bandwidth() * (.2 + .6 * ((d.j * 37) % 10) / 10))
    .attr('cy', d => sy(d.y)).attr('r', 4).attr('fill', d => d.g === 'AG' ? P.final : P.inicial).attr('opacity', .85)
    .style('cursor', d => d.vista ? 'pointer' : null)
    .on('mousemove', (ev, d) => mostrarTip(ev, esc(d.t) + (d.vista ? '<br><span class="tenue">clic: ver en el mapa</span>' : '')))
    .on('mouseleave', ocultarTip).on('click', (ev, d) => d.vista && seleccionarVista(d.vista));
  [['AG', d3.mean(ej.filas, f => f.fitness)], ['Aleatoria', d3.mean(ej.aleatoria)]].forEach(([g, y]) =>
    svg.append('line').attr('x1', sx(g)).attr('x2', sx(g) + sx.bandwidth()).attr('y1', sy(y)).attr('y2', sy(y)).attr('stroke', '#fff').attr('stroke-width', 2));
  const n2 = ej.distancias.length, maxd = d3.max(ej.distancias.flat()) || 1, col = d3.scaleSequential([0, maxd], d3.interpolateViridis);
  $('#g-e7c').innerHTML = `<table style="table-layout:fixed">${'<tr><th></th>' + ej.filas.map(f => `<th>s${f.semilla}</th>`).join('') + '</tr>'}${ej.distancias.map((fila, i) =>
    `<tr><th>s${ej.filas[i].semilla}</th>${fila.map((d, j) => `<td title="s${ej.filas[i].semilla} ↔ s${ej.filas[j].semilla}: ${fx(d, 1)}°" style="background:${i === j ? 'transparent' : col(d)};color:${d > maxd * .6 ? '#04121a' : '#E6ECFF'};text-align:center;font-size:9px;padding:2px">${i === j ? '' : fx(d, 0)}</td>`).join('')}</tr>`).join('')}</table>
    <div class="leyenda-g">Distancia media entre satélites emparejados de dos soluciones (${n2}×${n2}).</div>`;
  $('#t-e7').innerHTML = '<tr><th>SEMILLA</th><th>FITNESS</th><th>INICIAL</th><th>REG.</th><th>ACEPTADA</th><th>GEN 6/6</th><th>GEN 99%</th><th></th></tr>' +
    ej.filas.map(f => `<tr><td>${f.semilla}</td><td>${f4(f.fitness)}</td><td>${f4(f.fitness_inicial)}</td><td>${f.regiones_cubiertas}</td>
    <td>${f.aceptada ? marca(true) : marca(false) + ' ' + f.fallidos.join(', ')}</td><td>${f.gen_cobertura_completa ?? '—'}</td><td>${f.gen_convergencia ?? '—'}</td>
    <td>${botonVer(f.vista, 'Ver')}</td></tr>`).join('');
}

function pRestricciones(el) {
  const R = D.restricciones, T = R.texto;
  el.innerHTML = `<div class="rejilla" style="grid-template-columns:minmax(0,1fr) minmax(0,1fr)">
    <div class="caja"><h3>E3/E4 · MARGEN MÍNIMO POR ZONA EN TODAS LAS SEMILLAS</h3><div id="g-zonas"></div>
      <div class="chips" style="margin-top:6px"><div class="chip"><small>VÁLIDAS AL INICIO</small><b class="grande">${pct(R.fact_inicial)}</b></div>
      <div class="chip"><small>VÁLIDAS AL FINAL</small><b class="grande ok">${pct(R.fact_final)}</b></div>
      <div class="chip"><small>OBJETO JUNTO AL SAHEL</small><b class="grande">${f4(R.junto.fitness)}</b> <span class="tenue">vs ${f4(R.junto.fitness_base)}</span> ${botonVer('e4')}</div></div></div>
    <div class="caja">${bloque(T.auditoria)}</div>
    <div class="caja"><h3>E5 · ESPACIO REDUCIDO: FITNESS Y COBERTURA POR NIVEL</h3><div id="g-e5"></div><table id="t-e5" style="margin-top:6px"></table>
      <h3>R1 SIN MANEJO DE RESTRICCIONES ${botonVer('e5-R1-sin')}</h3>
      <div>Fitness aparente <b>${f4(R.sin_manejo.fitness)}</b> · intrusión <b class="mal">${fx(R.sin_manejo.violacion, 2)}°</b><br>
      <span class="mal">${R.sin_manejo.violaciones.map(esc).join('<br>') || 'sin violaciones'}</span></div></div>
    <div class="caja">${bloque(T.e5)}</div></div>`;
  barrasH($('#g-zonas'), R.zonas.map(z => ({nombre: `${z.zona} (${z.radio}°)`, valor: z.margen_min, color: z.todas ? (z.tipo === 'desechos' ? P.desechos : P.ocupada) : P.no_cubierta,
    tip: `<b>${esc(z.zona)}</b><br>distancia mínima ${fx(z.distancia_min, 2)}° · margen ${fx(z.margen_min, 2)}°<br>${z.todas ? 'todas las semillas la respetan' : '¡alguna semilla la invade!'}`})),
    {x: [Math.min(0, d3.min(R.zonas, z => z.margen_min)), d3.max(R.zonas, z => z.margen_min) * 1.05], fv: v => fx(v, 1) + '°', margenIzq: 160,
     v: [{x: 0, color: P.no_cubierta}]});
  barrasH($('#g-e5'), R.niveles.map(n => ({nombre: n.nombre, valor: n.fitness, color: n.factible ? P.final : P.no_cubierta,
    tip: `<b>${esc(n.nombre)}</b><br>fitness ${f4(n.fitness)} · cota ${f4(n.cota)}<br>altitudes: ${n.altitudes.join(', ')} km`})),
    {x: [Math.max(0, d3.min(R.niveles, n => n.fitness) - .1), 1], fv: f4, margenIzq: 190});
  const regiones = D.escenarios[VISTAS[R.niveles[0].vista].escenario].regiones;
  const col = d3.scaleLinear([0, .95, 1], [P.no_cubierta, P.media, P.cubierta]);
  $('#t-e5').innerHTML = `<tr><th>NIVEL</th>${regiones.map(r => `<th title="${esc(r.nombre)}">${esc(r.nombre.split(' ').slice(-1)[0].slice(0, 9))}</th>`).join('')}<th></th></tr>` +
    R.niveles.map(n => `<tr><td>${esc(n.nombre.split(' ')[0])}</td>${n.cobertura.map(c => `<td style="color:${col(c)}">${pct(c)}</td>`).join('')}<td>${botonVer(n.vista, 'Ver')}</td></tr>`).join('');
}

function pLimite(el) {
  const L = D.limite, e9 = L.e9;
  const fila = (t, a, b) => `<tr><td>${t}</td><td>${a}</td><td>${b}</td></tr>`;
  el.innerHTML = `<div class="rejilla r3">${L.casos.map(c => `<div class="caja"><h3>E6 · ${esc(c.clave.toUpperCase())} ${botonVer(c.vista)}</h3>
      <div class="prueba">${md(c.dificultad)}</div>
      <div class="chips" style="margin:6px 0"><div class="chip"><small>FITNESS</small><b>${f4(c.fitness)}</b></div>
      <div class="chip"><small>VÁLIDA</small><b>${sino(c.factible)}</b></div><div class="chip"><small>REGIONES</small><b>${c.cubiertas}/${c.regiones}</b></div>
      <div class="chip"><small>VIOLACIONES</small><b>${c.n_violaciones}</b></div></div>
      <div class="prueba">${md(c.texto)}</div></div>`).join('')}</div>
    <div class="rejilla" style="grid-template-columns:minmax(0,1fr) minmax(0,1fr) minmax(0,1.2fr);margin-top:10px">
      <div class="caja"><h3>E9 · ¿SIGUE SIENDO VÁLIDA LA SOLUCIÓN ANTERIOR?</h3><table>
        <tr><th></th><th>(a) PRIORIDAD</th><th>(b) ZONA NUEVA</th></tr>
        ${fila('Válida antes', sino(e9.a.valida_antes), sino(e9.b.valida_antes))}${fila('Válida ahora', sino(e9.a.valida_ahora), sino(e9.b.valida_ahora))}
        ${fila('Fitness antes', f4(e9.a.fitness_antes), f4(e9.b.fitness_antes))}${fila('Fitness ahora', f4(e9.a.fitness_ahora), f4(e9.b.fitness_ahora))}
        ${fila('Aceptada ahora', sino(e9.a.aceptada_ahora), sino(e9.b.aceptada_ahora))}
        ${fila('Reoptimizada', f4(e9.a.reoptimizado), `${f4(e9.b.previa)} / ${f4(e9.b.cero)}`)}
        ${fila('', botonVer('e9-a-anterior', 'Anterior') + ' ' + botonVer('e9-a-reopt', 'Nueva'), botonVer('e9-b-anterior', 'Anterior') + ' ' + botonVer('e9-b-previa', 'Previa') + ' ' + botonVer('e9-b-cero', 'Cero'))}</table>
        <div class="mal" style="margin-top:6px">${e9.b.violaciones.map(esc).join('<br>')}</div></div>
      <div class="caja"><h3>E9b · REOPTIMIZACIÓN TRAS LA ZONA NUEVA</h3><div id="g-e9"></div></div>
      <div class="caja">${bloque(e9.texto)}</div></div>`;
  const hp = VISTAS['e9-b-previa'].hist.mejor, hc = VISTAS['e9-b-cero'].hist.mejor;
  grafica($('#g-e9'), {alto: 170, x: [0, Math.max(hp.length, hc.length) - 1], y: [0, 1], ejeX: 'generación', nombreX: 'generación',
    series: [{y: hc, color: P.inicial, nombre: 'desde cero'}, {y: hp, color: P.final, ancho: 2, nombre: 'desde la población previa'}]});
}

function pPropia(el) {
  const Pr = D.propia;
  el.innerHTML = `<div class="rejilla" style="grid-template-columns:minmax(0,1.3fr) minmax(0,1fr)">
    <div class="caja"><h3>E8 · FITNESS SI FALLA CADA SATÉLITE (N-1) ${botonVer('base', 'Con redundancia')} ${botonVer('e8-sin', 'Sin redundancia')}</h3><div id="g-e8"></div>
      <table style="margin-top:6px"><tr><th>FALLA</th><th>CON: REGIONES PERDIDAS</th><th>SIN: REGIONES PERDIDAS</th></tr>
      ${Pr.con.map((f, i) => `<tr><td>S${f.satelite_fallado}</td><td class="${f.regiones_perdidas === 'ninguna' ? 'ok' : 'mal'}" style="text-align:left">${esc(f.regiones_perdidas)}</td>
        <td class="${Pr.sin[i].regiones_perdidas === 'ninguna' ? 'ok' : 'mal'}" style="text-align:left">${esc(Pr.sin[i].regiones_perdidas)}</td></tr>`).join('')}</table></div>
    <div class="caja">${bloque(Pr.texto)}</div></div>`;
  barrasAgrupadas($('#g-e8'), Pr.con.map(f => 'S' + f.satelite_fallado), [
    {nombre: 'con término de redundancia', color: P.final, valores: Pr.con.map(f => f.fitness), tips: Pr.con.map(f => 'perdidas: ' + f.regiones_perdidas)},
    {nombre: 'sin término de redundancia', color: P.inicial, valores: Pr.sin.map(f => f.fitness), tips: Pr.sin.map(f => 'perdidas: ' + f.regiones_perdidas)}],
    {y: [Math.max(0, d3.min(Pr.con.concat(Pr.sin), f => f.fitness) - .1), 1], alto: 165, ejeX: 'satélite que falla'});
}

function pCalibracion(el) {
  const C = D.calibracion;
  el.innerHTML = `<div class="rejilla" style="grid-template-columns:minmax(0,1.3fr) minmax(0,1fr)">
    <div class="caja"><h3>E10 · ABLACIÓN DE OPERADORES (${C.n_semillas} SEMILLAS, ESCENARIO R1)</h3><div id="g-e10"></div>
      <table style="margin-top:6px"><tr><th>VARIANTE</th><th>FITNESS</th><th>DESV.</th><th>VÁLIDAS</th><th>GEN 99%</th></tr>
      ${C.filas.map(f => `<tr><td>${esc(f.variante)}</td><td>${f4(f.fitness_media)}</td><td>${f4(f.fitness_desv)}</td><td>${f.validas}/${C.n_semillas}</td><td>${fx(f.gen_convergencia_media, 0)}</td></tr>`).join('')}</table></div>
    <div class="caja">${bloque(C.texto)}</div></div>`;
  const ref = C.filas[0].fitness_media;
  barrasH($('#g-e10'), C.filas.map((f, i) => ({nombre: f.variante, valor: f.fitness_media, error: f.fitness_desv || 0,
    color: i === 0 ? P.cubierta : (f.fitness_media < ref ? P.final : P.media)})),
    {x: [Math.max(0, d3.min(C.filas, f => f.fitness_media - (f.fitness_desv || 0)) - .01), Math.min(1, d3.max(C.filas, f => f.fitness_media + (f.fitness_desv || 0)) + .005)],
     fv: f4, margenIzq: 170, v: [{x: ref, color: P.cubierta, txt: 'final'}]});
}

function verificarEnNavegador() {
  // Recalcula la cobertura del cuadro final con los datos dibujados, de forma independiente a Python.
  const v = VISTAS.base, c = v.cuadros[v.cuadros.length - 1], sc = escDe(v), rad_r = 180 / Math.PI / RT;
  const res = sc.regiones.map((r, i) => {
    const pts = muestrasDisco(r.lat, r.lon, r.radio_km * rad_r, sc.anillos);
    const vistos = pts.filter(p => c.s.some(s => distAng(p[0], p[1], s[0], s[1]) <= s[3] + 1e-9)).length;
    return {js: vistos / pts.length, py: c.r[i][0]};
  });
  const crJs = criterios(c, sc).map(x => x[2]), crPy = Object.entries(D.modelo.aceptacion).filter(([k]) => k !== 'ACEPTADA').map(([, v]) => v);
  return {res, maxDif: d3.max(res, x => Math.abs(x.js - x.py)), criteriosIguales: crJs.every((x, i) => x === crPy[i])};
}

function pModelo(el) {
  const M = D.modelo, V = D.verificacion, U = D.pruebas_unitarias, js = verificarEnNavegador(), pa = M.parametros_ag;
  const w = M.pesos_fitness;
  el.innerHTML = `<div class="rejilla r3">
    <div class="caja prueba"><h3>FUNCIÓN DE EVALUACIÓN</h3>
      Por región: <code>puntaje = ${w[0]}·cobertura + ${w[1]}·redundancia + ${w[2]}·resolución</code><br>
      Total: <code>fitness = Σ peso·puntaje / Σ pesos</code> ∈ [0, 1]<br>
      Pesos por prioridad: ${Object.entries(M.pesos_prioridad).map(([k, v]) => `<b>${k}</b> = ${v}`).join(' · ')}<br>
      Cota de referencia (relajación sin zonas): <b>${f4(M.cota.fitness)}</b>
      <h3>SUPUESTOS DEL MODELO</h3><ul>${M.supuestos.map(s => `<li>${esc(s)}</li>`).join('')}</ul></div>
    <div class="caja"><h3>HUELLA DEL SENSOR CONTRA ALTITUD</h3><table><tr><th>ALTITUD km</th><th>RADIO HUELLA km</th><th>CALIDAD RESOL.</th></tr>
      ${M.huella.map(h => `<tr><td>${fx(h.alt_km, 0)}</td><td>${fx(h.radio_km, 0)}</td><td>${fx(h.calidad, 3)}</td></tr>`).join('')}</table>
      <h3>PARÁMETROS DEL AG</h3><table>${[['Población', pa.tam_poblacion], ['Generaciones', pa.generaciones], ['Torneo / élite', `${pa.tam_torneo} / ${pa.tam_elite}`],
        ['Prob. cruce / mezcla', `${pa.prob_cruce} / ${pa.prob_mezcla}`], ['Cruce alineado', pa.cruce_alineado ? 'sí' : 'no'], ['Prob. mutación', pa.prob_mutacion],
        ['σ posición', `${pa.sigma_inicial_grados}° → ${pa.sigma_final_grados}°`], ['σ altitud', `${pa.sigma_alt_inicial_km} → ${pa.sigma_alt_final_km} km`],
        ['Reubicación / reparación', `${pa.prob_reubicacion} / ${pa.prob_reparacion}`], ['Restricciones', pa.manejo_restricciones]]
        .map(([k, v]) => `<tr><td>${k}</td><td>${esc(v)}</td></tr>`).join('')}</table></div>
    <div class="caja"><h3>CRITERIOS DE ACEPTACIÓN (SOLUCIÓN PRINCIPAL)</h3><table>
      ${Object.entries(M.aceptacion).map(([k, v]) => `<tr><td style="white-space:normal">${esc(k)}</td><td>${marca(v)}</td></tr>`).join('')}</table>
      <h3>VERIFICACIÓN DE CORRESPONDENCIA</h3><table>
      <tr><td style="white-space:normal">Configuración dibujada = resultado del AG (dif. máx. ${V.max_dif_configuracion.toExponential(1)})</td><td>${marca(V.max_dif_configuracion < 1e-3)}</td></tr>
      <tr><td style="white-space:normal">Fitness y regiones cubiertas iguales a Python</td><td>${marca(V.fitness_igual && V.cobertura_igual)}</td></tr>
      <tr><td style="white-space:normal">Cobertura recalculada en el navegador (dif. máx. ${fx(js.maxDif, 4)})</td><td>${marca(js.maxDif < 1e-9)}</td></tr>
      <tr><td style="white-space:normal">Criterios C1–C4 del navegador = Python</td><td>${marca(js.criteriosIguales)}</td></tr>
      <tr><td style="white-space:normal">Pruebas unitarias (pytest)</td><td>${U ? `${marca(U.fallidas === 0 && U.aprobadas > 0)} ${U.aprobadas} aprobadas${U.fallidas ? `, <span class="mal">${U.fallidas} fallidas</span>` : ''}` : '<span class="tenue">no se ejecutaron</span>'}</td></tr></table>
      <div class="leyenda-g">El navegador vuelve a muestrear los 49 puntos de cada región y comprueba qué huellas los cubren, sin usar los valores de Python.</div>
      <div class="leyenda-g">Datos: modo <b>${esc(D.modo)}</b> · generado ${esc(D.generado)} · AstroShield ${esc(D.version)}${D.tiempo_s != null ? ` · ${D.tiempo_s} s de cómputo` : ''}</div></div></div>`;
}

function abrirPestana(id) {
  document.querySelectorAll('#pestanas button').forEach(b => b.setAttribute('aria-selected', String(b.dataset.id === id)));
  const p = PESTANAS.find(x => x[0] === id), el = $('#detalle');
  graficaEvolucion = null;
  try { p[2](el); } catch (e) { el.innerHTML = `<span class="mal">No se pudo dibujar esta pestaña: ${esc(e.message)}</span>`; console.error(e); }
  el.querySelectorAll('.ver').forEach(b => b.onclick = () => seleccionarVista(b.dataset.vista));
  el.scrollTop = 0;
}

// ---------------------------------------------------------------- arranque
function iniciar() {
  if (typeof d3 === 'undefined') {
    document.getElementById('app').innerHTML = '<div class="panel" style="padding:20px;margin:20px">No se pudo cargar la librería de gráficas (d3). ' +
      'Revisa tu conexión a internet y vuelve a ejecutar la celda.</div>'; return;
  }
  llenarSelector();
  $('#pestanas').innerHTML = PESTANAS.map(([id, t]) => `<button role="tab" data-id="${id}" aria-selected="false">${t}</button>`).join('');
  document.querySelectorAll('#pestanas button').forEach(b => b.onclick = () => abrirPestana(b.dataset.id));
  G = iniciarGlobo();
  if (!G) {
    const b = $('#modo3d'); b.disabled = true; b.title = 'Vista 3D no disponible';
    const aviso = $('#aviso'); aviso.style.display = 'block';
    aviso.textContent = typeof Globe === 'undefined' ? 'No se pudo cargar el globo 3D (globe.gl): se muestra el mapa 2D.'
      : 'Tu navegador no tiene WebGL: se muestra el mapa 2D.';
  }
  $('#modo3d').onclick = () => fijarModo(!E.modo3d);
  $('#gen').oninput = e => { detener(); E.gen = +e.target.value; pedirRender(); };
  $('#play').onclick = reproducir;
  $('#b-ini').onclick = () => { detener(); E.gen = 0; $('#gen').value = 0; render(); };
  $('#b-fin').onclick = () => { detener(); E.gen = E.vista.gen_total || 0; $('#gen').value = E.gen; render(); };
  $('#comparar').onchange = e => { E.comparar = e.target.checked; render(); };
  [['huellas', 'c-huellas'], ['desechos', 'c-desechos'], ['ocupadas', 'c-ocupadas'], ['giro', 'c-giro']].forEach(([k, id]) =>
    $('#' + id).onchange = e => { E.capas[k] = e.target.checked; render(); });
  $('#descargar').onclick = () => {
    const a = document.createElement('a'); a.href = URL.createObjectURL(new Blob([FUENTE], {type: 'text/html'}));
    a.download = 'astroshield_centro_control.html'; document.body.appendChild(a); a.click(); a.remove();
  };
  new ResizeObserver(() => { const el = $('#centro'); if (G) G.width(el.clientWidth).height(el.clientHeight); if (MAPA.listo) { ajustarMapa(); render(); } }).observe($('#centro'));
  // El globo deja de dibujar cuando el panel no está a la vista (Colab tiene muchas salidas en la misma pestaña).
  if ('IntersectionObserver' in window) new IntersectionObserver(es => es.forEach(e => {
    if (!G || !E.modo3d) return; if (e.isIntersecting) G.resumeAnimation(); else G.pauseAnimation(); })).observe($('#centro'));
  fijarModo(!!G);
  seleccionarVista(E.vista.id, false);
  abrirPestana('evolucion');
  if (FALLAS.length && FALLAS.some(f => f !== 'globe.gl')) console.warn('Librerías que no cargaron:', FALLAS);
}
iniciar();
</script>
</body>
</html>
"""

Writing astroshield/plantilla_centro_control.py


In [11]:
%%writefile astroshield/centro_control.py
"""Centro de control: una sola página HTML con el globo 3D / mapa 2D y todas las
gráficas y comparativas de E1-E10.

La página se genera con los datos *reales* de la ejecución: cada configuración
que se dibuja sale de un ResultadoAG y se evalúa con el mismo Evaluador que usa
el algoritmo, así que lo que se ve es exactamente lo que produjo el AG.
"""
from __future__ import annotations

import datetime
import json
import re
import subprocess
import sys

import numpy as np

from . import __version__
from .algoritmo_genetico import alinear
from .escenario import PESOS_PRIORIDAD, Escenario
from .evaluacion import (
    Evaluador,
    aceptacion,
    altitud_minima_vista_completa,
    calidad_resolucion,
    cota_referencia,
    semiangulo_huella,
)
from .experimentos import generacion_cobertura_completa, generacion_convergencia, margenes_zonas, resumir
from .geometria import grados_a_km
from .plantilla_centro_control import PLANTILLA
from .visualizacion import PALETA

MAX_CUADROS = 24  # cuadros de la línea de tiempo en las vistas secundarias
ALTO_PX = 1060


def _r(x, n=4):
    """Redondea para el JSON; NaN e infinitos se vuelven null."""
    if x is None:
        return None
    x = float(x)
    return round(x, n) if np.isfinite(x) else None


def _pct(x, total):
    return 100.0 * x / total if total else 0.0


def _lista(nombres):
    nombres = list(nombres)
    if len(nombres) <= 1:
        return "".join(nombres)
    return ", ".join(nombres[:-1]) + " y " + nombres[-1]


# --------------------------------------------------------------- escenarios
def _escenario(sc: Escenario) -> dict:
    return {
        "nombre": sc.nombre,
        "regiones": [{"nombre": r.nombre, "lat": r.lat, "lon": r.lon, "prioridad": r.prioridad, "peso": r.peso,
                      "radio_km": r.radio_km} for r in sc.regiones],
        "zonas": [{"nombre": z.nombre, "lat": z.lat, "lon": z.lon, "radio": z.radio_grados, "tipo": z.tipo}
                  for z in sc.zonas],
        "n_satelites": sc.n_satelites,
        "alt_min_km": sc.alt_min_km,
        "alt_max_km": sc.alt_max_km,
        "semiangulo_sensor": sc.semiangulo_sensor_grados,
        "umbral_cobertura": sc.umbral_cobertura,
        "separacion_min": sc.separacion_min_grados,
        "anillos": sc.anillos_muestra,
        "pesos_fitness": [sc.w_cobertura, sc.w_redundancia, sc.w_resolucion],
        "cota": _r(cota_referencia(sc)["fitness"]),
        "notas": list(sc.notas),
    }


# ------------------------------------------------------------------- vistas
def _cuadro(evaluador: Evaluador, gen, configuracion, decimales: int = 5) -> dict:
    """Un cuadro de la línea de tiempo: configuración + su evaluación completa.

    `decimales` fija la precisión de lat/lon: 5 (≈ 1 m) en la vista principal, que
    se verifica en el navegador, y 4 (≈ 11 m) en las secundarias, para aligerar."""
    sc = evaluador.escenario
    configuracion = np.asarray(configuracion, dtype=float)
    ev = evaluador.evaluar(configuracion)
    lam = semiangulo_huella(configuracion[:, 2], sc.semiangulo_sensor_grados)
    malos = {v.satelite for v in ev.violaciones}
    return {
        "g": gen,
        "f": _r(ev.fitness),
        "ok": int(ev.factible),
        "v": _r(ev.violacion, 3),
        "c": ev.n_cubiertas,
        "s": [[_r(la, decimales), _r(lo, decimales), _r(h, decimales - 2), _r(lam[i], decimales), int(i in malos)]
              for i, (la, lo, h) in enumerate(configuracion)],
        "r": [[_r(x.cobertura), _r(x.redundancia), _r(x.resolucion), int(x.cubierta), x.satelites]
              for x in ev.regiones],
        "x": [f"S{v.satelite} → {v.objetivo} ({v.distancia_grados:.2f}° < {v.limite_grados}°)"
              for v in ev.violaciones][:12],
    }


def _adelgazar(gens, maximo):
    if maximo is None or len(gens) <= maximo:
        return gens
    idx = np.unique(np.round(np.linspace(0, len(gens) - 1, maximo)).astype(int))
    return [gens[i] for i in idx]


def _historial(resultado) -> dict:
    h = resultado.historial
    return {
        "mejor": [_r(x["mejor_rango_historico"]) if x["mejor_rango_historico"] >= 0 else None for x in h],
        "media": [_r(x["fitness_media"], 3) for x in h],
        "fact": [_r(x["fraccion_factible"], 3) for x in h],
        "div": [_r(x["diversidad_grados"], 1) for x in h],
    }


class _Constructor:
    """Junta escenarios y vistas sin duplicar escenarios iguales."""

    def __init__(self):
        self.escenarios, self.vistas = {}, []

    def escenario(self, sc: Escenario) -> str:
        datos = _escenario(sc)
        clave = sc.nombre
        while clave in self.escenarios and self.escenarios[clave] != datos:
            clave += "'"
        self.escenarios[clave] = datos
        return clave

    def vista_resultado(self, id_, grupo, titulo, resultado, nota="", max_cuadros=MAX_CUADROS):
        sc = resultado.escenario
        ev = Evaluador(sc)
        gens = _adelgazar(sorted(resultado.instantaneas), max_cuadros)
        configs = [np.asarray(resultado.instantaneas[g], dtype=float) for g in gens]
        # El orden de los satélites en el genoma es arbitrario: se alinea cada cuadro
        # con el siguiente para que S3 sea el mismo satélite en toda la línea de
        # tiempo. El último cuadro conserva el orden exacto del resultado del AG.
        for i in range(len(configs) - 2, -1, -1):
            if len(configs[i]) == len(configs[i + 1]):
                configs[i] = alinear(configs[i + 1], configs[i])
        self.vistas.append({
            "id": id_, "grupo": grupo, "titulo": titulo, "escenario": self.escenario(sc),
            "semilla": resultado.semilla, "nota": nota, "gen_total": resultado.historial[-1]["generacion"],
            "cuadros": [_cuadro(ev, int(g), cfg, 5 if max_cuadros is None else 4) for g, cfg in zip(gens, configs)],
            "hist": _historial(resultado),
        })
        return id_

    def vista_fija(self, id_, grupo, titulo, escenario: Escenario, configuracion, nota=""):
        """Una configuración evaluada bajo otro escenario (por ejemplo, una revalidación)."""
        self.vistas.append({
            "id": id_, "grupo": grupo, "titulo": titulo, "escenario": self.escenario(escenario),
            "semilla": None, "nota": nota, "gen_total": None,
            "cuadros": [_cuadro(Evaluador(escenario), None, configuracion, 4)], "hist": None,
        })
        return id_


# ------------------------------------------------------------------- textos
def _texto_evolucion(base) -> dict:
    sc, ev, cfg = base.escenario, base.eval_mejor, base.parametros
    h = base.historial
    cota = cota_referencia(sc)["fitness"]
    hmin = altitud_minima_vista_completa(300.0, sc)
    g30 = min(30, len(h) - 1)
    redundantes = [r.nombre for r in ev.regiones if r.redundancia >= 0.95]
    principales = sorted({min(r.satelites, key=lambda s: base.mejor[s, 2]) for r in ev.regiones if r.satelites})
    alts = base.mejor[principales, 2] if principales else np.array([np.nan])
    estrategia = (
        f"Cada región cubierta tiene un satélite principal entre {alts.min():.0f} y {alts.max():.0f} km, "
        f"justo arriba del mínimo de {hmin:.0f} km que cubre los 300 km."
    )
    if redundantes:
        estrategia += (f" Los satélites restantes dan redundancia a {_lista(redundantes)}. "
                       "**Nadie programó esa regla: surge de la función de evaluación.**")
    return {
        "condiciones": (f"Escenario base con {len(sc.zonas_desechos)} zonas de desechos, {len(sc.zonas_ocupadas)} "
                        f"posiciones ocupadas y separación mínima de {sc.separacion_min_grados:g}°. Población de "
                        f"{cfg.tam_poblacion}, {h[-1]['generacion']} generaciones, semilla {base.semilla}."),
        "resultado": (f"El fitness pasa de {base.eval_inicial.fitness:.4f} (mejor individuo aleatorio) a "
                      f"**{ev.fitness:.4f}**, el {_pct(ev.fitness, cota):.2f}% de la cota de referencia ({cota:.4f}). "
                      f"Regiones cubiertas: {ev.n_cubiertas}/{len(ev.regiones)} desde la generación "
                      f"{generacion_cobertura_completa(base)}; el 99% del valor final se alcanza en la generación "
                      f"{generacion_convergencia(base)}."),
        "interpretacion": [
            (f"**Primero explora y luego explota.** La diversidad baja de {h[0]['diversidad_grados']:.0f}° a "
             f"{h[g30]['diversidad_grados']:.0f}° en las primeras {g30} generaciones, con σ de mutación alto; después "
             f"σ decae de {cfg.sigma_inicial_grados:g}° a {cfg.sigma_final_grados:g}° y el AG solo afina "
             f"(diversidad final {h[-1]['diversidad_grados']:.1f}°)."),
            estrategia,
            (f"Al inicio, el {100 * h[0]['fraccion_factible']:.0f}% de la población aleatoria era válida. Con las "
             f"reglas de Deb, las inválidas pierden contra cualquier válida, y la solución final tiene "
             f"{len(ev.violaciones)} violaciones."),
        ],
    }


def _criterios_fallidos(resultado):
    criterios = aceptacion(resultado.eval_mejor, resultado.escenario)
    return [k.split(" ")[0] for k, v in criterios.items() if k != "ACEPTADA" and not v]


def _texto_ejecuciones(e7) -> dict:
    res, ejec = e7["resumen"], e7["ejecuciones"]
    n = res["n_ejecuciones"]
    validas = sum(r.eval_mejor.factible for r in ejec)
    completas = sum(r.eval_mejor.n_cubiertas == len(r.escenario.regiones) for r in ejec)
    fallidas = {r.semilla: _criterios_fallidos(r) for r in ejec}
    fallidas = {s: c for s, c in fallidas.items() if c}
    interp = [f"**Validez:** {validas}/{n} ejecuciones son válidas y {completas}/{n} cubren todas las regiones."]
    if fallidas:
        detalle = "; ".join(f"semilla {s}: {', '.join(c)}" for s, c in fallidas.items())
        interp.append(f"**No aceptadas:** {detalle}.")
        if all(c == ["C3"] for c in fallidas.values()):
            interp.append(
                "Todas fallan solo por C3: un satélite de redundancia quedó sobre una región Media en lugar de una "
                "Alta. Es un **óptimo local**: salir de ahí exige mover un satélite decenas de grados de golpe, y la "
                "mutación fina del final ya no da saltos tan grandes.")
    else:
        interp.append("**Todas las ejecuciones cumplen los cuatro criterios de aceptación.**")
    interp += [
        (f"**AG contra búsqueda aleatoria:** con el mismo presupuesto, la aleatoria se queda en "
         f"{res['aleatoria_media']:.3f} (máx. {res['aleatoria_max']:.3f}). La prueba de Mann-Whitney unilateral da "
         f"p = {res['p_mannwhitney_ag_mayor_aleatoria']:.1e}: **el AG no gana por suerte**."),
        (f"**AG contra voraz:** la heurística voraz ({res['voraz']:.4f}) solo prueba posiciones cerca de las regiones, "
         f"es decir, usa conocimiento del problema. El AG parte de cero y obtiene en promedio {res['ag_media']:.4f} "
         f"(mejor ejecución {res['ag_max']:.4f})."),
        (f"**Soluciones distintas, calidad parecida:** la distancia media entre las soluciones de dos semillas es "
         f"{res['distancia_media_soluciones_grados']:.1f}° (máx. {res['distancia_max_soluciones_grados']:.1f}°). "
         "El paisaje es multimodal; por eso se reportan distribuciones y no una sola ejecución."),
    ]
    return {
        "condiciones": (f"Escenario base, {n} semillas con los mismos parámetros y el mismo presupuesto de "
                        f"{res['presupuesto_evaluaciones']:,} evaluaciones para el AG y para la búsqueda aleatoria. "
                        "La heurística voraz agrega un satélite a la vez, siempre el de mayor ganancia."),
        "resultado": (f"Fitness {res['ag_media']:.4f} ± {res['ag_desv']:.4f} (CV {res['ag_cv_porcentaje']:.2f}%), "
                      f"rango [{res['ag_min']:.4f}, {res['ag_max']:.4f}]. {res['ag_aceptadas']}/{n} ejecuciones "
                      "cumplen todos los criterios."),
        "interpretacion": interp,
    }


def _satelite_de(resultado, nombre_region):
    region = next(r for r in resultado.eval_mejor.regiones if r.nombre == nombre_region)
    if not region.satelites:
        return None
    return min(region.satelites, key=lambda s: resultado.mejor[s, 2])


def _texto_restricciones(todo) -> dict:
    auditoria, ejec, junto, base = todo["auditoria"], todo["e7"]["ejecuciones"], todo["e4"], todo["base"]
    margenes = auditoria.groupby("zona", sort=False)["margen_grados"].min()
    ini = np.mean([r.historial[0]["fraccion_factible"] for r in ejec])
    fin = np.mean([r.historial[-1]["fraccion_factible"] for r in ejec])
    todas = bool(auditoria["respeta"].all())
    s_junto, s_base = _satelite_de(junto, "Sahel"), _satelite_de(base, "Sahel")
    sahel = next(r for r in junto.eval_mejor.regiones if r.nombre == "Sahel")
    zona = next(z for z in junto.escenario.zonas if "reubicado" in z.nombre)
    m = margenes_zonas(junto.mejor, junto.escenario)
    margen = float(m.loc[m["zona"] == zona.nombre, "margen_grados"].iloc[0])
    estres = (f"**Objeto junto al Sahel:** el margen mínimo a la zona reubicada es {margen:.2f}°. "
              f"Cobertura del Sahel {sahel.cobertura:.0%}, resolución {sahel.resolucion:.2f}.")
    if s_junto is not None and s_base is not None:
        estres += (f" El satélite del Sahel pasa de {base.mejor[s_base, 2]:.0f} km a {junto.mejor[s_junto, 2]:.0f} km "
                   "para alcanzar la región desde fuera de la zona.")
    estres += (f" La restricción tiene un **costo medible**: el fitness baja de {base.eval_mejor.fitness:.4f} a "
               f"{junto.eval_mejor.fitness:.4f}.")

    niveles, sin_manejo = todo["e5"]["niveles"], todo["e5"]["sin_manejo"]
    fits = [r.eval_mejor.fitness for r in niveles]
    monotono = all(b <= a + 1e-9 for a, b in zip(fits, fits[1:]))
    todas_cubiertas = all(r.eval_mejor.n_cubiertas == len(r.escenario.regiones) for r in niveles)
    e5 = [f"El fitness {'baja de forma monótona' if monotono else 'cambia'} al reducir el espacio: "
          + " → ".join(f"{f:.4f}" for f in fits) + "."]
    e5.append("**Las regiones siguen cubiertas en todos los niveles**: el AG compensa subiendo o alejando satélites, "
              "y lo que se pierde es resolución y redundancia." if todas_cubiertas else
              "En algunos niveles ya no se pueden cubrir todas las regiones; el AG sacrifica primero las de menor peso.")
    e5.append(f"**Sin manejo de restricciones**, R1 obtiene un fitness aparente de {sin_manejo.eval_mejor.fitness:.4f} "
              f"con {sin_manejo.eval_mejor.violacion:.1f}° de intrusión: un puntaje más alto que **no es válido**. "
              f"Con las reglas de Deb, R1 queda en {niveles[1].eval_mejor.fitness:.4f}, pero válida.")
    return {
        "auditoria": {
            "condiciones": (f"Auditoría geométrica independiente del AG: para la mejor solución de cada una de las "
                            f"{len(ejec)} semillas se mide la distancia del satélite más cercano a cada zona."),
            "resultado": (f"{'Todas las zonas se respetan' if todas else 'Hay zonas invadidas'} en las {len(ejec)} "
                          "ejecuciones. Margen mínimo por zona: "
                          + ", ".join(f"{k} {v:.1f}°" for k, v in margenes.items()) + "."),
            "interpretacion": [
                (f"La población aleatoria inicial tenía {100 * (1 - ini):.0f}% de configuraciones inválidas; al final "
                 f"queda {100 * fin:.0f}% de válidas. Las inválidas restantes son hijos recién mutados que la "
                 "selección descarta."),
                ("**Observación honesta:** en el escenario base las zonas quedan lejos de las regiones, así que en el "
                 "óptimo no están activas. Por eso las pruebas de estrés, E5 y E6 las ponen encima de las regiones."),
                estres,
            ],
        },
        "e5": {
            "condiciones": ("Se reduce el espacio en 4 niveles acumulativos: R0 base → R1 + zona de 6° a 2.6° de "
                            "Yucatán → R2 + techo de 1200 km → R3 + zona sobre Japón. R1 también se ejecuta sin "
                            "manejo de restricciones para ver qué evita el sistema."),
            "resultado": " · ".join(f"{r.escenario.nombre.split(' ')[0]}: {r.eval_mejor.fitness:.4f} "
                                    f"({r.eval_mejor.n_cubiertas}/{len(r.escenario.regiones)})" for r in niveles),
            "interpretacion": e5,
        },
    }


DIFICULTAD_E6 = {
    "I1 Amazonía inalcanzable": "Zona de 15° centrada en la Amazonía: ninguna huella (máx. ≈ 11°) la alcanza desde "
                                "fuera. **Objetivo inalcanzable.**",
    "I2 solo 5 satélites": "Solo 5 satélites para 6 regiones. **Recursos insuficientes.**",
    "I3 separación 80° imposible": "Separación mínima de 80° entre 8 satélites: imposible por geometría (problema de "
                                   "Tammes, máx. ≈ 74.9°). **No existe ninguna configuración válida.**",
}


def _texto_e6(clave, r) -> str:
    ev = r.eval_mejor
    sin_cubrir = [x for x in ev.regiones if not x.cubierta]
    if not ev.factible:
        nunca = max(h["fraccion_factible"] for h in r.historial) == 0
        return (f"No se encontró ninguna configuración válida{' en ninguna generación' if nunca else ''}. El AG lo "
                f"reporta (factible = no) y devuelve la que **menos** viola: {len(ev.violaciones)} violaciones, "
                f"{ev.violacion:.1f}° de intrusión total. **Que el algoritmo termine no significa que haya resuelto "
                "el problema.**")
    if not sin_cubrir:
        return f"Solución válida con todas las regiones cubiertas (fitness {ev.fitness:.4f})."
    nombres = _lista(f"{x.nombre} ({x.prioridad}, {x.cobertura:.0%})" for x in sin_cubrir)
    texto = (f"Solución **válida** (fitness {ev.fitness:.4f}) con {ev.n_cubiertas}/{len(ev.regiones)} regiones "
             f"cubiertas; queda sin cubrir: {nombres}. El sistema **no hace trampa**: lo reporta (C2 = falso).")
    if all(x.prioridad == "normal" for x in sin_cubrir):
        texto += " Sacrifica justo la región de **menor prioridad**: las prioridades deciden qué se pierde."
    return texto


def _texto_e9(todo) -> dict:
    cambios = todo["e9"]
    a, b = cambios["prioridad"], cambios["zona_nueva"]
    ra, rb = a["revalidacion"], b["revalidacion"]
    previa, cero = b["desde_poblacion_previa"], b["desde_cero"]
    return {
        "condiciones": ("(a) El Interior de Australia sube a prioridad Alta. (b) Aparece una zona de desechos de 3° "
                        "justo donde está el satélite que observa Japón. Primero se revalida la solución anterior y "
                        "luego se reoptimiza desde la población previa y, para comparar, desde cero."),
        "resultado": (f"(a) válida antes/ahora: {'sí' if ra['valida_antes'] else 'no'}/{'sí' if ra['valida_ahora'] else 'no'}"
                      f", fitness {ra['fitness_antes']:.4f} → {ra['fitness_ahora']:.4f}. (b) válida antes/ahora: "
                      f"{'sí' if rb['valida_antes'] else 'no'}/{'sí' if rb['valida_ahora'] else 'no'}."),
        "interpretacion": [
            (f"**(a) Prioridad:** la solución anterior {'sigue siendo válida' if ra['valida_ahora'] else 'deja de ser válida'}"
             f", porque una prioridad no cambia la geometría; su fitness cambia porque cambian los pesos. La "
             f"reoptimización llega a {a['reoptimizado'].eval_mejor.fitness:.4f} "
             f"(cota {cota_referencia(a['escenario'])['fitness']:.4f})."),
            (f"**(b) Zona nueva:** la solución anterior {'deja de ser válida' if not rb['valida_ahora'] else 'sigue siendo válida'}"
             + (f" ({'; '.join(rb['violaciones'])})" if rb["violaciones"] else "")
             + f", aunque su fitness sea {rb['fitness_ahora']:.4f}. **Puntaje y validez son cosas distintas.**"),
            (f"Desde la población previa, el AG llega a {previa.eval_mejor.fitness:.4f} "
             f"(99% en la generación {generacion_convergencia(previa)}); desde cero, con el mismo presupuesto, a "
             f"{cero.eval_mejor.fitness:.4f}. "
             + ("**Reutilizar la población anterior es más eficiente.**"
                if previa.eval_mejor.fitness >= cero.eval_mejor.fitness else
                "En esta ejecución, empezar de cero dio un resultado igual o mejor.")),
        ],
    }


def _texto_e8(robustez) -> dict:
    con, sin = robustez["con_redundancia"], robustez["sin_redundancia"]
    n = len(con)
    ok_con = int((con["regiones_perdidas"] == "ninguna").sum())
    ok_sin = int((sin["regiones_perdidas"] == "ninguna").sum())
    return {
        "condiciones": ("Pregunta propia: ¿el término de redundancia del fitness tiene valor real o solo infla el "
                        "puntaje? Se simula la falla de cada satélite (análisis N-1) en la solución base y en una "
                        "optimizada **sin** término de redundancia (pesos 0.80 / 0 / 0.20)."),
        "resultado": (f"Con redundancia, {ok_con}/{n} fallas no hacen perder ninguna región (fitness medio tras una "
                      f"falla {con['fitness'].mean():.4f}). Sin redundancia, {ok_sin}/{n} "
                      f"({sin['fitness'].mean():.4f})."),
        "interpretacion": [
            ("El término de redundancia hace la constelación **más tolerante a fallas**." if ok_con > ok_sin else
             "En esta ejecución el término de redundancia no mejoró la tolerancia a fallas."),
            (f"En el peor caso, con redundancia quedan {int(con['regiones_cubiertas'].min())} regiones cubiertas; "
             f"sin ella, {int(sin['regiones_cubiertas'].min())}."),
        ],
    }


def _texto_e10(agregado, n_semillas) -> dict:
    final = agregado.iloc[0]
    peores = agregado.sort_values("fitness_media").head(2)["variante"].tolist()
    dudosas = [
        f["variante"] for _, f in agregado.iloc[1:].iterrows()
        if abs(f["fitness_media"] - final["fitness_media"]) < max(f["fitness_desv"] or 0, final["fitness_desv"] or 0)
    ]
    interp = [f"Las variantes que más empeoran el resultado son {_lista(peores)}."]
    if dudosas:
        interp.append(f"Las diferencias de {_lista(dudosas)} son menores que la desviación estándar, así que **no son "
                      f"concluyentes con {n_semillas} semillas**; se reporta así en lugar de sobreinterpretarlas.")
    interp.append("La calibración inicial (docs/04) mostró que el número de generaciones y el ajuste fino final "
                  "(σ final de 0.05° y 2 km) son los parámetros que más mejoran el resultado.")
    return {
        "condiciones": (f"Escenario R1 (las restricciones sí afectan aquí), {n_semillas} semillas por variante. Se "
                        "apaga un operador a la vez para medir su aporte."),
        "resultado": f"Configuración final: {final['fitness_media']:.4f} ± {final['fitness_desv'] or 0:.4f}.",
        "interpretacion": interp,
    }


# -------------------------------------------------------------------- datos
def _modelo(base, cfg) -> dict:
    sc = base.escenario
    alts = [500, altitud_minima_vista_completa(300.0, sc), 750, 1000, 1500, 2000]
    cota = cota_referencia(sc)
    return {
        "parametros_ag": cfg.a_dict(),
        "pesos_prioridad": PESOS_PRIORIDAD,
        "pesos_fitness": [sc.w_cobertura, sc.w_redundancia, sc.w_resolucion],
        "huella": [{"alt_km": a, "radio_km": _r(grados_a_km(semiangulo_huella(a, sc.semiangulo_sensor_grados)), 0),
                    "calidad": _r(calidad_resolucion(a, sc.alt_min_km, sc.alt_max_km), 3)} for a in alts],
        "cota": {"fitness": _r(cota["fitness"]),
                 "regiones": {k: {"alt_km": v["alt_km"], "puntaje": _r(v["puntaje"])} for k, v in cota["regiones"].items()}},
        "aceptacion": {k: bool(v) for k, v in aceptacion(base.eval_mejor, sc).items()},
        "supuestos": [
            "La Tierra es una esfera de radio 6371 km.",
            "Cada satélite se mantiene fijo sobre un punto (station-keeping); altitud permitida de 500 a 2000 km.",
            "Sensor apuntando al nadir con semiángulo de 30°: subir agranda la huella pero empeora la resolución.",
            "Las zonas prohibidas son casquetes (conos desde el centro de la Tierra) que aplican a cualquier altitud.",
            "Separación mínima de 2° entre satélites propios.",
            "Región cubierta ⇔ al menos el 95% de sus 49 puntos de muestra está dentro de alguna huella.",
        ],
    }


def _verificacion(vista_base, base) -> dict:
    final = vista_base["cuadros"][-1]
    dibujada = np.array([s[:3] for s in final["s"]])
    return {
        "max_dif_configuracion": float(np.abs(dibujada - base.mejor).max()),
        "fitness_igual": final["f"] == _r(base.eval_mejor.fitness),
        "cobertura_igual": [r[3] for r in final["r"]] == [int(x.cubierta) for x in base.eval_mejor.regiones],
    }


def ejecutar_pruebas_unitarias(directorio: str = ".") -> dict:
    """Ejecuta pytest y devuelve el resumen que muestra el centro de control."""
    proc = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider"], cwd=directorio,
                          capture_output=True, text=True)
    lineas = [l for l in proc.stdout.strip().splitlines() if l.strip()]
    ultima = lineas[-1] if lineas else proc.stderr.strip()[-200:]
    cuenta = {k: int(n) for n, k in re.findall(r"(\d+) (passed|failed|error)", ultima)}
    return {"aprobadas": cuenta.get("passed", 0), "fallidas": cuenta.get("failed", 0) + cuenta.get("error", 0),
            "resumen": ultima, "codigo_salida": proc.returncode}


def datos_centro_control(todo: dict, pruebas_unitarias: dict = None, modo: str = "completo") -> dict:
    """Arma el JSON del centro de control a partir de los resultados de `ejecutar_todo`."""
    base, e7, cfg = todo["base"], todo["e7"], todo["parametros"]
    c = _Constructor()

    principal = "Ejecución principal"
    c.vista_resultado("base", principal, "Etapa 2 · escenario base", base,
                      "Todas las restricciones. Cada cuadro es una mejora del mejor individuo.", max_cuadros=None)
    c.vista_resultado("e1", "Etapa 1", "E1 · sin restricciones", todo["e1"], "Solo regiones, sin zonas prohibidas.")
    for r in e7["ejecuciones"]:
        c.vista_resultado(f"e7-{r.semilla}", "Variación entre semillas (E7)", f"E7 · semilla {r.semilla}", r)
    c.vista_resultado("e4", "Restricciones (E4, E5)", "E4 · objeto junto al Sahel", todo["e4"],
                      "El Objeto A se reubica a 2° del centro del Sahel.")
    niveles = todo["e5"]["niveles"]
    for r in niveles:
        c.vista_resultado(f"e5-{r.escenario.nombre.split(' ')[0]}", "Restricciones (E4, E5)",
                          f"E5 · {r.escenario.nombre}", r)
    c.vista_resultado("e5-R1-sin", "Restricciones (E4, E5)", "E5 · R1 sin manejo de restricciones",
                      todo["e5"]["sin_manejo"], "Configuración con mejor puntaje pero inválida.")
    for k, r in todo["e6"].items():
        c.vista_resultado(f"e6-{k.split(' ')[0]}", "Escenarios límite (E6)", f"E6 · {k}", r, DIFICULTAD_E6[k])
    ca, cb = todo["e9"]["prioridad"], todo["e9"]["zona_nueva"]
    grupo9 = "Cambio de condiciones (E9)"
    c.vista_fija("e9-a-anterior", grupo9, "E9a · solución anterior revalidada", ca["escenario"], base.mejor,
                 "La solución base evaluada con Australia en prioridad Alta.")
    c.vista_resultado("e9-a-reopt", grupo9, "E9a · reoptimizada", ca["reoptimizado"])
    c.vista_fija("e9-b-anterior", grupo9, "E9b · solución anterior revalidada", cb["escenario"], base.mejor,
                 "La solución base frente a la zona nueva: ahora viola una restricción.")
    c.vista_resultado("e9-b-previa", grupo9, "E9b · desde la población previa", cb["desde_poblacion_previa"])
    c.vista_resultado("e9-b-cero", grupo9, "E9b · desde cero", cb["desde_cero"])
    c.vista_resultado("e8-sin", "Prueba propia (E8)", "E8 · optimizada sin redundancia",
                      todo["e8"]["ejecucion_sin_redundancia"], "Pesos 0.80 / 0 / 0.20.")

    vista_base = c.vistas[0]
    h = base.historial
    auditoria = todo["auditoria"]
    robustez = todo["e8"]
    agregado = todo["e10"]["agregado"]
    n_ablacion = int(todo["e10"]["detalle"]["semilla"].nunique())
    textos_restr = _texto_restricciones(todo)
    return {
        "version": __version__,
        "generado": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
        "modo": modo,
        "paleta": PALETA,
        "vista_inicial": "base",
        "escenarios": c.escenarios,
        "vistas": c.vistas,
        "evolucion": {
            "fitness_mejor": [_r(x["fitness_mejor"]) for x in h],
            "fitness_media": [_r(x["fitness_media"]) for x in h],
            "fitness_desv": [_r(x["fitness_desv"]) for x in h],
            "fitness_peor": [_r(x["fitness_peor"]) for x in h],
            "fact": [_r(x["fraccion_factible"], 3) for x in h],
            "div": [_r(x["diversidad_grados"], 2) for x in h],
            "sigma": [_r(x["sigma_grados"], 3) for x in h],
            "sigma_alt": [_r(x["sigma_alt_km"], 1) for x in h],
            "cota": _r(cota_referencia(base.escenario)["fitness"]),
            "gen_convergencia": generacion_convergencia(base),
            "gen_cobertura_completa": generacion_cobertura_completa(base),
            "texto": _texto_evolucion(base),
        },
        "ejecuciones": {
            "resumen": e7["resumen"],
            "filas": [{**resumir(r), "fallidos": _criterios_fallidos(r), "vista": f"e7-{r.semilla}"}
                      for r in e7["ejecuciones"]],
            "aleatoria": [_r(x) for x in e7["aleatoria"]],
            "distancias": [[_r(x, 1) for x in fila] for fila in e7["distancias"]],
            "texto": _texto_ejecuciones(e7),
        },
        "restricciones": {
            "zonas": [{"zona": z, "tipo": g["tipo"].iloc[0], "radio": float(g["radio_grados"].iloc[0]),
                       "distancia_min": float(g["distancia_grados"].min()), "margen_min": float(g["margen_grados"].min()),
                       "todas": bool(g["respeta"].all())} for z, g in auditoria.groupby("zona", sort=False)],
            "fact_inicial": _r(np.mean([r.historial[0]["fraccion_factible"] for r in e7["ejecuciones"]]), 3),
            "fact_final": _r(np.mean([r.historial[-1]["fraccion_factible"] for r in e7["ejecuciones"]]), 3),
            "junto": {"margenes": margenes_zonas(todo["e4"].mejor, todo["e4"].escenario).to_dict("records"),
                      "fitness": _r(todo["e4"].eval_mejor.fitness), "fitness_base": _r(base.eval_mejor.fitness)},
            "niveles": [{"nombre": r.escenario.nombre, "vista": f"e5-{r.escenario.nombre.split(' ')[0]}",
                         "fitness": _r(r.eval_mejor.fitness), "factible": r.eval_mejor.factible,
                         "cubiertas": r.eval_mejor.n_cubiertas, "cota": _r(cota_referencia(r.escenario)["fitness"]),
                         "aceptada": aceptacion(r.eval_mejor, r.escenario)["ACEPTADA"],
                         "cobertura": [_r(x.cobertura, 3) for x in r.eval_mejor.regiones],
                         "altitudes": [int(round(a)) for a in r.mejor[:, 2]]} for r in niveles],
            "sin_manejo": {"fitness": _r(todo["e5"]["sin_manejo"].eval_mejor.fitness),
                           "violacion": _r(todo["e5"]["sin_manejo"].eval_mejor.violacion, 2),
                           "violaciones": [f"S{v.satelite} en {v.objetivo} ({v.distancia_grados:.2f}° < {v.limite_grados}°)"
                                           for v in todo["e5"]["sin_manejo"].eval_mejor.violaciones]},
            "texto": textos_restr,
        },
        "limite": {
            "casos": [{"clave": k, "vista": f"e6-{k.split(' ')[0]}", "dificultad": DIFICULTAD_E6[k],
                       "fitness": _r(r.eval_mejor.fitness), "factible": r.eval_mejor.factible,
                       "violacion": _r(r.eval_mejor.violacion, 2), "n_violaciones": len(r.eval_mejor.violaciones),
                       "cubiertas": r.eval_mejor.n_cubiertas, "regiones": len(r.escenario.regiones),
                       "fact_max": _r(max(x["fraccion_factible"] for x in r.historial), 3),
                       "texto": _texto_e6(k, r)} for k, r in todo["e6"].items()],
            "e9": {
                "a": {**ca["revalidacion"], "reoptimizado": _r(ca["reoptimizado"].eval_mejor.fitness),
                      "cota": _r(cota_referencia(ca["escenario"])["fitness"])},
                "b": {**cb["revalidacion"], "previa": _r(cb["desde_poblacion_previa"].eval_mejor.fitness),
                      "cero": _r(cb["desde_cero"].eval_mejor.fitness)},
                "texto": _texto_e9(todo),
            },
        },
        "propia": {
            "con": robustez["con_redundancia"].to_dict("records"),
            "sin": robustez["sin_redundancia"].to_dict("records"),
            "texto": _texto_e8(robustez),
        },
        "calibracion": {
            "filas": [{k: (_r(v) if isinstance(v, float) else v) for k, v in f.items()}
                      for f in agregado.to_dict("records")],
            "n_semillas": n_ablacion,
            "texto": _texto_e10(agregado, n_ablacion),
        },
        "modelo": _modelo(base, cfg),
        "pruebas_unitarias": pruebas_unitarias,
        "verificacion": _verificacion(vista_base, base),
        "tiempo_s": _r(todo["tiempo_s"], 0),
    }


def agregar_laboratorio(datos: dict, resultado, configuracion_anterior, escenario_anterior: Escenario) -> dict:
    """Agrega (o reemplaza) las vistas del laboratorio y las deja seleccionadas."""
    c = _Constructor()
    c.escenarios = dict(datos["escenarios"])
    c.vista_fija("lab-anterior", "Laboratorio", "Laboratorio · solución base revalidada", resultado.escenario,
                 configuracion_anterior, f"Solución de «{escenario_anterior.nombre}» bajo las nuevas condiciones.")
    c.vista_resultado("lab", "Laboratorio", "Laboratorio · reoptimizada", resultado,
                      "; ".join(resultado.escenario.notas) or "Escenario personalizado.")
    datos = dict(datos)
    datos["escenarios"] = c.escenarios
    datos["vistas"] = [v for v in datos["vistas"] if not v["id"].startswith("lab")] + c.vistas
    datos["vista_inicial"] = "lab"
    return datos


# ---------------------------------------------------------------- salida
def html_centro_control(datos: dict, alto: int = ALTO_PX) -> str:
    """Página HTML autocontenida (UTF-8) con el centro de control."""
    js = json.dumps(datos, ensure_ascii=False, separators=(",", ":"), default=_json_default,
                    allow_nan=False).replace("</", "<\\/")
    return PLANTILLA.replace("__ALTO__", str(alto)).replace("__DATOS__", js)


def _json_default(x):
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return _r(x, 6)
    if isinstance(x, np.bool_):
        return bool(x)
    raise TypeError(type(x))


def guardar_centro_control(datos: dict, ruta: str, alto: int = ALTO_PX) -> str:
    html = html_centro_control(datos, alto)
    with open(ruta, "w", encoding="utf-8") as fh:
        fh.write(html)
    return ruta


def en_colab() -> bool:
    return "google.colab" in sys.modules


def mostrar_centro_control(datos: dict, manejador=None, alto: int = ALTO_PX):
    """Muestra el centro de control en el notebook. Si recibe el `manejador` de una
    salida anterior, la actualiza en su lugar en vez de crear otro bloque."""
    from IPython.display import HTML, display

    html = html_centro_control(datos, alto)
    if not en_colab():
        # Jupyter no aísla las salidas: se envuelve en un iframe propio.
        srcdoc = html.replace("&", "&amp;").replace('"', "&quot;")
        html = (f'<iframe srcdoc="{srcdoc}" style="width:100%;height:{alto + 8}px;border:0" '
                'allow="fullscreen"></iframe>')
    if manejador is None:
        return display(HTML(html), display_id=True)
    manejador.update(HTML(html))
    return manejador

Writing astroshield/centro_control.py


In [12]:
%%writefile pruebas/test_modelo.py
import numpy as np
import pytest

from astroshield.escenario import Zona, escenario_base, escenario_inicial
from astroshield.evaluacion import (
    Evaluador,
    aceptacion,
    altitud_minima_vista_completa,
    cota_referencia,
    listar_violaciones,
    semiangulo_huella,
)
from astroshield.geometria import destino, distancia_angular, km_a_grados, muestras_disco, normalizar_lon


def configuracion_encima(escenario, alt=900.0):
    """Un satélite sobre el centro de cada región; los sobrantes, lejos, sobre océanos."""
    filas = [(r.lat, r.lon, alt) for r in escenario.regiones]
    sobrantes = [(-60.0, -150.0, alt), (-60.0, 60.0, alt)]
    return np.array((filas + sobrantes)[: escenario.n_satelites])


# ---------------------------------------------------------------- geometría
def test_distancia_angular_valores_conocidos():
    assert distancia_angular(0, 0, 0, 90) == pytest.approx(90)
    assert distancia_angular(90, 0, -90, 0) == pytest.approx(180)
    assert distancia_angular(10, 179, 10, -179) == pytest.approx(2 * np.cos(np.radians(10)), rel=1e-3)


def test_destino_recorre_la_distancia_pedida():
    lat, lon = destino(20.5, -89.0, np.arange(0, 360, 30), 7.0)
    assert np.allclose(distancia_angular(20.5, -89.0, lat, lon), 7.0)


def test_normalizar_lon_rango():
    assert normalizar_lon(190) == pytest.approx(-170)
    assert normalizar_lon(-180) == pytest.approx(-180)


def test_muestras_disco_cubren_exactamente_la_region():
    pts = muestras_disco(0, 0, 3.0, anillos=3)
    assert len(pts) == 1 + 8 + 16 + 24
    d = distancia_angular(0, 0, pts[:, 0], pts[:, 1])
    assert d.max() == pytest.approx(3.0)


# ---------------------------------------------------------------- cobertura
def test_huella_crece_con_la_altitud():
    lam = semiangulo_huella(np.linspace(500, 2000, 20), 30)
    assert np.all(np.diff(lam) > 0)


def test_altitud_minima_no_ve_la_region_completa():
    sc = escenario_inicial()
    assert semiangulo_huella(500, 30) < km_a_grados(300)
    assert 500 < altitud_minima_vista_completa(300, sc) < 600


def test_satelite_encima_cubre_toda_su_region():
    sc = escenario_inicial()
    ev = Evaluador(sc).evaluar(configuracion_encima(sc))
    assert all(r.cobertura == 1.0 and r.cubierta for r in ev.regiones)
    assert 0 < ev.fitness <= 1


def test_satelites_lejanos_no_cubren_nada():
    sc = escenario_inicial()
    cfg = np.tile([-80.0, 0.0, 2000.0], (8, 1))
    cfg[:, 1] = np.arange(8) * 45
    ev = Evaluador(sc).evaluar(cfg)
    assert ev.n_cubiertas == 0 and ev.fitness == 0


def test_pesos_de_prioridad_cambian_el_fitness():
    sc = escenario_inicial()
    cfg = configuracion_encima(sc)
    cfg[0] = (-60, 170, 900)  # deja sin cubrir Yucatán (alta)
    base = Evaluador(sc).evaluar(cfg).fitness
    menor = Evaluador(sc.con_prioridad("Península de Yucatán", "normal")).evaluar(cfg).fitness
    assert menor > base  # perder una región Normal cuesta menos que perder una Alta


def test_evaluacion_por_lote_e_individual_coinciden():
    sc = escenario_base()
    rng = np.random.default_rng(3)
    pob = np.stack([configuracion_encima(sc) + rng.normal(0, [1, 1, 10], (8, 3)) for _ in range(5)])
    ev = Evaluador(sc)
    lote = ev.evaluar_poblacion(pob)
    for i in range(5):
        individual = ev.evaluar(pob[i])
        assert individual.fitness == pytest.approx(lote["fitness"][i])
        assert individual.factible == bool(lote["factible"][i])


# -------------------------------------------------------------- restricciones
@pytest.mark.parametrize("nombre_zona", ["Atlántico Norte", "Pacífico Sur", "Polar Norte", "Objeto A", "Objeto B"])
def test_satelite_dentro_de_cada_zona_es_invalido(nombre_zona):
    sc = escenario_base()
    z = next(z for z in sc.zonas if z.nombre == nombre_zona)
    cfg = configuracion_encima(sc)
    cfg[7, :2] = destino(z.lat, z.lon, 45, z.radio_grados * 0.9)
    ev = Evaluador(sc).evaluar(cfg)
    assert not ev.factible
    assert [v.objetivo for v in ev.violaciones] == [nombre_zona]
    assert ev.violacion == pytest.approx(z.radio_grados * 0.1, rel=1e-6)


def test_satelite_justo_fuera_de_la_zona_es_valido():
    sc = escenario_base()
    z = sc.zonas[0]
    cfg = configuracion_encima(sc)
    cfg[7, :2] = destino(z.lat, z.lon, 90, z.radio_grados + 0.01)
    assert Evaluador(sc).evaluar(cfg).factible


def test_se_exige_la_separacion_minima():
    sc = escenario_base()
    cfg = configuracion_encima(sc)
    cfg[7] = (cfg[0, 0] + 1.0, cfg[0, 1], 900)  # a 1° del satélite 0 (< 2°)
    v = listar_violaciones(cfg, sc)
    assert len(v) == 1 and v[0].restriccion == "separacion"


def test_zona_nueva_invalida_la_configuracion_anterior():
    sc = escenario_base()
    cfg = configuracion_encima(sc)
    assert Evaluador(sc).evaluar(cfg).factible
    cambiado = sc.con_zona(Zona("Nueva", cfg[3, 0], cfg[3, 1], 3.0))
    ev = Evaluador(cambiado).evaluar(cfg)
    assert not ev.factible and ev.violaciones[0].satelite == 3


def test_cota_de_referencia_supera_una_solucion_conocida():
    sc = escenario_base()
    ev = Evaluador(sc).evaluar(configuracion_encima(sc, alt=520))
    assert ev.fitness <= cota_referencia(sc)["fitness"]
    assert set(aceptacion(ev, sc)) >= {"C1 válida (sin violaciones)", "ACEPTADA"}


def test_redundancia_exigida_se_adapta_a_los_satelites_sobrantes():
    # Tres regiones Alta pero solo dos satélites sobrantes: dos regiones Alta
    # con redundancia deben bastar para cumplir C3.
    sc = escenario_base().con_prioridad("Interior de Australia", "alta")
    cfg = configuracion_encima(sc, alt=520)
    cfg[6] = (sc.regiones[0].lat + 2.5, sc.regiones[0].lon, 1300)  # 2.º satélite sobre Yucatán
    cfg[7] = (sc.regiones[1].lat + 2.5, sc.regiones[1].lon, 1300)  # 2.º satélite sobre la Amazonía
    criterios = aceptacion(Evaluador(sc).evaluar(cfg), sc)
    c3 = next(k for k in criterios if k.startswith("C3"))
    assert "≥2" in c3 and criterios[c3]

Writing pruebas/test_modelo.py


In [13]:
%%writefile pruebas/test_algoritmo_genetico.py
import numpy as np
import pytest

from astroshield.algoritmo_genetico import (
    ParametrosAG,
    alinear,
    ejecutar_ag,
    poblacion_aleatoria,
    reflejar_altitud,
    reparar,
)
from astroshield.escenario import Zona, escenario_base
from astroshield.evaluacion import Evaluador, cota_referencia
from astroshield.geometria import distancia_angular
from astroshield.referencias import busqueda_aleatoria, colocacion_voraz

RAPIDO = ParametrosAG(tam_poblacion=40, generaciones=60)


@pytest.fixture(scope="module")
def ejecucion_base():
    return ejecutar_ag(escenario_base(), RAPIDO, semilla=1)


def test_ag_mejora_sobre_la_poblacion_inicial(ejecucion_base):
    assert ejecucion_base.eval_mejor.fitness > ejecucion_base.eval_inicial.fitness + 0.2
    assert len(ejecucion_base.historial) == RAPIDO.generaciones + 1


def test_mejor_historico_nunca_disminuye(ejecucion_base):
    curva = [h["mejor_rango_historico"] for h in ejecucion_base.historial]
    assert all(b >= a - 1e-12 for a, b in zip(curva, curva[1:]))


def test_solucion_final_respeta_todas_las_zonas(ejecucion_base):
    sc = ejecucion_base.escenario
    mejor = ejecucion_base.mejor
    assert ejecucion_base.eval_mejor.factible
    for z in sc.zonas:
        assert distancia_angular(mejor[:, 0], mejor[:, 1], z.lat, z.lon).min() >= z.radio_grados


def test_altitudes_dentro_de_limites(ejecucion_base):
    sc = ejecucion_base.escenario
    pob = ejecucion_base.poblacion_final
    assert pob[..., 2].min() >= sc.alt_min_km and pob[..., 2].max() <= sc.alt_max_km


def test_misma_semilla_es_reproducible():
    sc = escenario_base()
    a = ejecutar_ag(sc, ParametrosAG(tam_poblacion=20, generaciones=10), semilla=7)
    b = ejecutar_ag(sc, ParametrosAG(tam_poblacion=20, generaciones=10), semilla=7)
    assert np.array_equal(a.mejor, b.mejor)


def test_restricciones_imposibles_se_reportan_no_se_ocultan():
    # 8 puntos en una esfera no pueden estar todos a 80° (cota de Tammes ~74.9°).
    sc = escenario_base().con_parametros(separacion_min_grados=80.0)
    r = ejecutar_ag(sc, ParametrosAG(tam_poblacion=30, generaciones=20), semilla=0)
    assert not r.encontro_factible
    assert r.eval_mejor.violacion > 0 and r.eval_mejor.violaciones
    assert np.isnan(r.historial[-1]["fitness_mejor"])


def test_region_inalcanzable_queda_sin_cubrir_pero_la_solucion_es_valida():
    sc = escenario_base().con_zona(Zona("Bloqueo Amazonía", -3.5, -62.0, 15.0))
    r = ejecutar_ag(sc, RAPIDO, semilla=0)
    amazonia = next(x for x in r.eval_mejor.regiones if x.nombre == "Amazonía brasileña")
    assert r.encontro_factible and amazonia.cobertura == 0.0


def test_arranque_en_caliente_conserva_la_solucion_semilla(ejecucion_base):
    r = ejecutar_ag(ejecucion_base.escenario, ParametrosAG(tam_poblacion=40, generaciones=1), semilla=5,
                    poblacion_inicial=ejecucion_base.mejor[None])
    assert r.eval_inicial.fitness == pytest.approx(ejecucion_base.eval_mejor.fitness)


def test_ag_supera_busqueda_aleatoria_con_el_mismo_presupuesto(ejecucion_base):
    _, curva = busqueda_aleatoria(ejecucion_base.escenario, ejecucion_base.evaluaciones, semilla=1)
    assert ejecucion_base.eval_mejor.fitness > curva[-1][1]


def test_voraz_y_ag_bajo_la_cota_de_referencia(ejecucion_base):
    sc = ejecucion_base.escenario
    cota = cota_referencia(sc)["fitness"]
    _, g = colocacion_voraz(sc)
    assert g.factible and g.fitness <= cota and ejecucion_base.eval_mejor.fitness <= cota


# ---------------------------------------------------------------- operadores
def test_reparar_saca_al_satelite_de_la_zona():
    sc = escenario_base()
    individuo = np.array([[45.0, -35.0, 800.0], [44.0, -30.0, 800.0]])
    reparar(np.random.default_rng(0), individuo, sc)
    assert Evaluador(sc.con_parametros(n_satelites=2)).evaluar(individuo).factible


def test_alinear_empareja_los_satelites_mas_cercanos():
    a = np.array([[0, 0, 1000], [0, 90, 1000], [45, 180, 1000]], float)
    b = a[[2, 0, 1]] + [0.5, 0.5, 0]
    assert np.allclose(alinear(a, b)[:, :2], a[:, :2] + 0.5)


def test_reflejar_altitud_se_queda_dentro():
    x = reflejar_altitud(np.array([400.0, 2100.0, 4100.0, 1000.0]), 500, 2000)
    assert np.all((x >= 500) & (x <= 2000)) and x[3] == 1000


def test_modo_de_restriccion_invalido_se_rechaza():
    with pytest.raises(ValueError):
        ParametrosAG(manejo_restricciones="magia")


def test_forma_de_la_poblacion_aleatoria():
    pob = poblacion_aleatoria(np.random.default_rng(0), escenario_base(), 5)
    assert pob.shape == (5, 8, 3)

Writing pruebas/test_algoritmo_genetico.py


In [14]:
%%writefile pruebas/test_visualizacion.py
"""Comprobaciones de las gráficas estáticas con nombres en español."""
import matplotlib

matplotlib.use("Agg")
import numpy as np  # noqa: E402

from astroshield.escenario import escenario_base  # noqa: E402
from astroshield.visualizacion import graficar_mapa_mision  # noqa: E402


def test_mapa_de_mision_muestra_regiones_con_acentos():
    sc = escenario_base()
    configuracion = np.array([[r.lat, r.lon, 800] for r in sc.regiones] + [[0, 90, 800], [0, -90, 800]])
    fig = graficar_mapa_mision(sc, configuracion)
    textos = [t.get_text() for t in fig.axes[0].texts]
    assert any("Península de Yucatán" in t for t in textos)
    assert any("Japón" in t for t in textos)
    assert "6/6 regiones cubiertas" in fig.axes[0].get_title()

Writing pruebas/test_visualizacion.py


In [15]:
%%writefile pruebas/test_centro_control.py
"""El centro de control debe mostrar exactamente lo que produjo el AG."""
import json
import re

import numpy as np
import pytest

from astroshield import centro_control as C
from astroshield import experimentos as X
from astroshield.algoritmo_genetico import ParametrosAG, ejecutar_ag
from astroshield.escenario import escenario_base

RAPIDO = ParametrosAG(tam_poblacion=24, generaciones=12)


@pytest.fixture(scope="module")
def todo():
    return X.ejecutar_todo(RAPIDO, n_semillas=2, n_ablacion=1)


@pytest.fixture(scope="module")
def datos(todo):
    return C.datos_centro_control(todo, {"aprobadas": 1, "fallidas": 0, "resumen": "1 passed", "codigo_salida": 0})


def _datos_incrustados(html):
    return json.loads(re.search(r'<script id="datos" type="application/json">(.*?)</script>', html, re.S).group(1))


def test_instantaneas_en_la_generacion_0_y_en_cada_mejora():
    r = ejecutar_ag(escenario_base(), RAPIDO, semilla=3)
    curva = [h["mejor_rango_historico"] for h in r.historial]
    mejoras = {g for g in range(1, len(curva)) if curva[g] > curva[g - 1]}
    assert set(r.instantaneas) == {0, len(curva) - 1} | mejoras
    assert np.array_equal(r.instantaneas[len(curva) - 1], r.mejor)


def test_el_panel_dibuja_la_configuracion_real_del_ag(todo, datos):
    base = todo["base"]
    final = next(v for v in datos["vistas"] if v["id"] == "base")["cuadros"][-1]
    dibujada = np.array([s[:3] for s in final["s"]])
    assert np.abs(dibujada - base.mejor).max() < 1e-3
    assert final["f"] == round(base.eval_mejor.fitness, 4)
    assert [r[3] for r in final["r"]] == [int(x.cubierta) for x in base.eval_mejor.regiones]
    assert datos["verificacion"]["fitness_igual"] and datos["verificacion"]["cobertura_igual"]


def test_la_vista_principal_tiene_un_cuadro_por_mejora(todo, datos):
    base = todo["base"]
    vista = next(v for v in datos["vistas"] if v["id"] == "base")
    assert [c["g"] for c in vista["cuadros"]] == sorted(base.instantaneas)
    assert len(vista["hist"]["mejor"]) == len(base.historial)


def test_cada_prueba_tiene_condiciones_resultado_e_interpretacion(datos):
    textos = [datos["evolucion"]["texto"], datos["ejecuciones"]["texto"], datos["restricciones"]["texto"]["auditoria"],
              datos["restricciones"]["texto"]["e5"], datos["limite"]["e9"]["texto"], datos["propia"]["texto"],
              datos["calibracion"]["texto"]]
    for t in textos:
        assert t["condiciones"] and t["resultado"] and t["interpretacion"]
    assert all(c["texto"] for c in datos["limite"]["casos"])


def test_html_autocontenido_en_espanol_con_interruptor_en_3d(datos):
    html = C.html_centro_control(datos)
    assert '<html lang="es">' in html and "CENTRO DE CONTROL" in html
    assert 'role="switch" aria-checked="true"' in html  # 3D por defecto
    assert "d3/7.8.5/d3.min.js" in html and "globe.gl@2.33.1" in html
    assert html.count("</script>") == html.count("<script")  # los datos no cierran la etiqueta antes de tiempo
    incrustados = _datos_incrustados(html)
    assert incrustados["vistas"][0]["cuadros"][-1] == datos["vistas"][0]["cuadros"][-1]


def test_vistas_cubren_todas_las_pruebas(datos):
    ids = {v["id"] for v in datos["vistas"]}
    assert {"base", "e1", "e4", "e5-R1-sin", "e6-I1", "e6-I2", "e6-I3", "e9-b-anterior", "e8-sin"} <= ids
    assert all(v["escenario"] in datos["escenarios"] for v in datos["vistas"])


def test_solucion_anterior_bajo_zona_nueva_se_marca_como_invalida(datos):
    anterior = next(v for v in datos["vistas"] if v["id"] == "e9-b-anterior")["cuadros"][0]
    assert anterior["ok"] == 0 and any(s[4] for s in anterior["s"]) and anterior["x"]


def test_laboratorio_reemplaza_sus_vistas_sin_duplicarlas(todo, datos):
    base = todo["base"]
    lab = escenario_base().con_prioridad("Interior de Australia", "alta", nombre="Laboratorio")
    r = ejecutar_ag(lab, RAPIDO, semilla=1)
    una = C.agregar_laboratorio(datos, r, base.mejor, base.escenario)
    dos = C.agregar_laboratorio(una, r, base.mejor, base.escenario)
    ids = [v["id"] for v in dos["vistas"]]
    assert ids.count("lab") == 1 and ids.count("lab-anterior") == 1 and dos["vista_inicial"] == "lab"
    assert len(dos["vistas"]) == len(datos["vistas"]) + 2

Writing pruebas/test_centro_control.py


In [16]:
%%writefile pytest.ini
[pytest]
testpaths = pruebas
pythonpath = .

Writing pytest.ini


In [17]:
#@title Datos de la ejecución de referencia (comprimidos)
import base64, gzip
REFERENCIA_B64 = """
...
"""
with open("outputs/datos_referencia.json", "wb") as fh:
    fh.write(gzip.decompress(base64.b64decode("".join(REFERENCIA_B64.split()))))
print("Datos de referencia listos.")

Datos de referencia listos.


## 2. Configuración

In [18]:
#@title Configuración de la ejecución { display-mode: "form" }
MODO = "completo"  #@param ["completo", "rápido", "solo referencia"]
#@markdown **completo** ≈ 4 min (parámetros finales, 10 semillas) · **rápido** ≈ 1 min · **solo referencia** no ejecuta el AG.
import importlib, importlib.util, json, subprocess, sys, warnings
EN_COLAB = "google.colab" in sys.modules
REQUERIDAS = ["numpy", "scipy", "pandas", "matplotlib", "pytest"]
faltan = [m for m in REQUERIDAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
warnings.filterwarnings("ignore")
sys.path.insert(0, ".")
for _m in [m for m in sys.modules if m == "astroshield" or m.startswith("astroshield.")]:
    del sys.modules[_m]  # al volver a ejecutar, toma los cambios de las celdas %%writefile
import numpy as np
import astroshield
from astroshield import experimentos as X
from astroshield import centro_control as CC
from astroshield.algoritmo_genetico import ParametrosAG, ejecutar_ag
from astroshield.escenario import Zona, escenario_base
if MODO == "completo":
    PARAMETROS, N_SEMILLAS, N_ABLACION = ParametrosAG(), 10, 5
else:
    PARAMETROS, N_SEMILLAS, N_ABLACION = ParametrosAG(generaciones=120, tam_poblacion=80), 4, 2
print(f"AstroShield {astroshield.__version__} · modo: {MODO} · Python {sys.version.split()[0]} · Colab: {EN_COLAB}")

AstroShield 2.0.0 · modo: completo · Python 3.11.15 · Colab: False


## 3. Centro de control

Se abre con la **ejecución de referencia** y la sección 4 lo actualiza con tu ejecución. Usa el selector
*Configuración en el mapa* para ver cualquier prueba en el globo, el botón ▶ para reproducir la evolución y
*Comparar con inicial* para ver qué cambió. Todos los textos de interpretación se calculan con los datos mostrados.

In [19]:
with open("outputs/datos_referencia.json", encoding="utf-8") as fh:
    DATOS = json.load(fh)
DATOS["modo"] = "referencia"
PANEL = CC.mostrar_centro_control(DATOS)

## 4. Ejecutar el algoritmo y actualizar el centro de control

In [20]:
if MODO == "solo referencia":
    print("Modo «solo referencia»: el centro de control muestra la ejecución de referencia.")
else:
    TODO = X.ejecutar_todo(PARAMETROS, n_semillas=N_SEMILLAS, n_ablacion=N_ABLACION, progreso=True)
    print("Pruebas unitarias...", flush=True)
    PRUEBAS = CC.ejecutar_pruebas_unitarias(".")
    print(" ", PRUEBAS["resumen"])
    DATOS = CC.datos_centro_control(TODO, PRUEBAS, modo=MODO)
    CC.mostrar_centro_control(DATOS, PANEL)
    CC.guardar_centro_control(DATOS, "outputs/centro_control.html")
    with open("outputs/resultados.json", "w", encoding="utf-8") as fh:
        json.dump(X.resumen_resultados(TODO), fh, ensure_ascii=False, indent=2, default=str)
    b = TODO["base"].eval_mejor
    print(f"\nEjecución principal: fitness {b.fitness:.4f}, {b.n_cubiertas}/6 regiones, válida: {b.factible}.")
    print("↑ El centro de control de la sección 3 ya muestra esta ejecución.")

[1/9] E1 · etapa 1 sin restricciones  (0 s)


[2/9] E2 · ejecución principal con todas las restricciones  (3 s)


[3/9] E7 · variación entre 10 semillas + búsqueda aleatoria + voraz  (7 s)


  semilla 0: fitness 0.9174 (1/10)


  semilla 1: fitness 0.9175 (2/10)


...

  variante: población 40


Listo en 224 s.


Pruebas unitarias...


  44 passed in 5.11s



Ejecución principal: fitness 0.9174, 6/6 regiones, válida: True.
↑ El centro de control de la sección 3 ya muestra esta ejecución.


## 5. Laboratorio (opcional)

Cambia las condiciones y ejecuta la celda. La solución de la ejecución principal se **revalida** con las nuevas
condiciones y luego se reoptimiza; ambas aparecen en el centro de control, en el grupo *Laboratorio*.

In [21]:
#@title Escenario personalizado { run: "manual", display-mode: "form" }
prioridad_australia = "alta"  #@param ["alta", "media", "normal"]
techo_altitud_km = 2000  #@param {type:"slider", min:600, max:2000, step:100}
nueva_zona_lat = 36.0  #@param {type:"number"}
nueva_zona_lon = 138.0  #@param {type:"number"}
nueva_zona_radio = 0.0  #@param {type:"slider", min:0, max:20, step:0.5}
generaciones = 200  #@param {type:"slider", min:20, max:600, step:20}
semilla = 42  #@param {type:"integer"}
ejecutar_laboratorio = False  #@param {type:"boolean"}
#@markdown Marca la casilla y ejecuta esta celda para probar tu escenario.

anterior = escenario_base()
lab = anterior.con_prioridad("Interior de Australia", prioridad_australia, nombre="Laboratorio")
lab = lab.con_parametros(alt_max_km=float(techo_altitud_km))
if nueva_zona_radio > 0:
    lab = lab.con_zona(Zona("Zona del laboratorio", nueva_zona_lat, nueva_zona_lon, nueva_zona_radio), nombre="Laboratorio")
if not ejecutar_laboratorio:
    print("Laboratorio desactivado: marca «ejecutar_laboratorio» y vuelve a ejecutar esta celda.")
else:
    final_base = next(v for v in DATOS["vistas"] if v["id"] == "base")["cuadros"][-1]
    solucion_base = np.array([s[:3] for s in final_base["s"]])
    print("Solución base revalidada:", X.revalidar(solucion_base, anterior, lab))
    r_lab = ejecutar_ag(lab, ParametrosAG(generaciones=generaciones), semilla=semilla)
    print(f"Reoptimizada: fitness {r_lab.eval_mejor.fitness:.4f}, {r_lab.eval_mejor.n_cubiertas}/6 regiones, "
          f"válida: {r_lab.eval_mejor.factible}.")
    DATOS = CC.agregar_laboratorio(DATOS, r_lab, solucion_base, anterior)
    CC.mostrar_centro_control(DATOS, PANEL)
    print("↑ Revisa el grupo «Laboratorio» en el centro de control de la sección 3.")

Laboratorio desactivado: marca «ejecutar_laboratorio» y vuelve a ejecutar esta celda.


## 6. Descargar los resultados

Empaqueta el centro de control (HTML que se abre en cualquier navegador), `resultados.json` y el código en un `.zip`. En Colab se descarga automáticamente.

In [22]:
import zipfile
CC.guardar_centro_control(DATOS, "outputs/centro_control.html")
with zipfile.ZipFile("astroshield_resultados.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in ("outputs", "astroshield", "pruebas"):
        for raiz, _, archivos in os.walk(carpeta):
            for f in archivos:
                if not f.endswith(".pyc"):
                    z.write(os.path.join(raiz, f))
print("Creado astroshield_resultados.zip con", len(zipfile.ZipFile("astroshield_resultados.zip").namelist()), "archivos")
if EN_COLAB:
    from google.colab import files
    files.download("astroshield_resultados.zip")

Creado astroshield_resultados.zip con 17 archivos
